# AQUA — Future Aquaculture Water-Quality Prediction

## 1. Research Objective

Develop a predictive model that uses historical water-quality measurements to anticipate whether a pond's water quality will be within the recorded acceptable range at its next observation.

The objective is to investigate whether earlier water-quality conditions can provide useful early-warning information about subsequent water-quality status.

## 2. Prediction Definition

Each modelling example will represent two consecutive chronological observations from the same pond:

- **Input:** Measurements from the earlier observation.
- **Prediction target:** The recorded water-quality classification (`Is WQ in range?`) at the next observation for that pond.
- **Output:** `Yes` or `No`.

The initial prediction horizon is the next recorded observation, not a fixed number of days. The elapsed time between observations will be examined and documented.

## 3. Initial Predictors

The initial model will prioritise the earlier observation's:

- pH
- Dissolved oxygen (DO)
- Water temperature

Additional predictors will be considered only after checking their availability, reliability and whether they would be known when the prediction is made.

## 4. Research and Validation Principles

- Preserve the original dataset.
- Maintain chronological order within each pond.
- Prevent future information from entering the input features.
- Exclude target-leaking fields, particularly `Parameters out of range`.
- Account for repeated observations from the same pond.
- Avoid random row-level train-test splitting.
- Document measurement intervals, missing values and protocol limitations.

## 5. Expected Research Contribution

Evaluate whether historical pond measurements can support early identification of future out-of-range water-quality conditions and provide a foundation for an aquaculture decision-support system.

In [1]:
# CELL 1: LOAD AND VERIFY THE ORIGINAL DATASET

from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path("../data/raw/water_quality.csv")

df = pd.read_csv(data_path)

df["Parsed date"] = pd.to_datetime(
    df["Date of data collection"],
    format="%m/%d/%Y",
    errors="coerce"
)

verification = pd.DataFrame({
    "Check": [
        "Dataset rows",
        "Dataset columns",
        "Unique ponds",
        "Successfully parsed dates",
        "Unparsed dates",
        "First observation date",
        "Last observation date"
    ],
    "Result": [
        len(df),
        len(df.columns) - 1,
        df["pond_id"].nunique(),
        df["Parsed date"].notna().sum(),
        df["Parsed date"].isna().sum(),
        df["Parsed date"].min().date(),
        df["Parsed date"].max().date()
    ]
})

display(verification)

,Check,Result
0,Dataset rows,12431
1,Dataset columns,53
2,Unique ponds,317
3,Successfully parsed dates,12431
4,Unparsed dates,0
5,First observation date,2021-06-19
6,Last observation date,2026-08-06


In [2]:
# CELL 2: CONSECUTIVE OBSERVATION INTERVAL AUDIT

ordered_df = (
    df.dropna(subset=["pond_id", "Parsed date"])
    .sort_values(["pond_id", "Parsed date"])
    .copy()
)

ordered_df["Next observation date"] = (
    ordered_df.groupby("pond_id")["Parsed date"].shift(-1)
)

ordered_df["Days until next observation"] = (
    ordered_df["Next observation date"] - ordered_df["Parsed date"]
).dt.days

intervals = ordered_df["Days until next observation"].dropna()

interval_summary = pd.DataFrame({
    "Metric": [
        "Total consecutive observation pairs",
        "Unique ponds with consecutive observations",
        "Minimum interval (days)",
        "Median interval (days)",
        "Mean interval (days)",
        "75th percentile interval (days)",
        "90th percentile interval (days)",
        "Maximum interval (days)",
        "Pairs within 1–3 days",
        "Pairs within 30 days"
    ],
    "Value": [
        len(intervals),
        ordered_df.loc[
            ordered_df["Days until next observation"].notna(),
            "pond_id"
        ].nunique(),
        intervals.min(),
        intervals.median(),
        round(intervals.mean(), 2),
        intervals.quantile(0.75),
        intervals.quantile(0.90),
        intervals.max(),
        intervals.between(1, 3).sum(),
        intervals.between(1, 30).sum()
    ]
})

display(interval_summary)

,Metric,Value
0,Total consecutive observation pairs,12114.00
1,Unique ponds with consecutive observations,316.00
2,Minimum interval (days),0.00
3,Median interval (days),2.00
4,Mean interval (days),13.27
5,75th percentile interval (days),21.00
6,90th percentile interval (days),33.00
7,Maximum interval (days),1098.00
8,Pairs within 1–3 days,553.00
9,Pairs within 30 days,4552.00


In [3]:
# CELL 3: SAME-DAY CONSECUTIVE OBSERVATION AUDIT

same_day_pairs = ordered_df[
    ordered_df["Days until next observation"] == 0
].copy()

same_day_pairs["Current time"] = same_day_pairs[
    "Time of data collection"
]

same_day_pairs["Next time"] = (
    same_day_pairs.groupby("pond_id")["Time of data collection"]
    .shift(-1)
)

same_day_pairs["Time comparison"] = np.where(
    same_day_pairs["Current time"].isna()
    | same_day_pairs["Next time"].isna(),
    "Missing time",
    np.where(
        same_day_pairs["Current time"]
        == same_day_pairs["Next time"],
        "Same recorded time",
        "Different recorded times"
    )
)

same_day_summary = (
    same_day_pairs["Time comparison"]
    .value_counts(dropna=False)
    .rename_axis("Time comparison")
    .reset_index(name="Pair count")
)

display(same_day_summary)

,Time comparison,Pair count
0,Different recorded times,5486
1,Missing time,313
2,Same recorded time,153


In [4]:
# CELL 4: CORRECTED SAME-DAY OBSERVATION TIME AUDIT

time_audit = ordered_df.copy()

time_audit["Next recorded time"] = (
    time_audit.groupby("pond_id")["Time of data collection"]
    .shift(-1)
)

time_audit["Time comparison"] = np.where(
    time_audit["Days until next observation"] != 0,
    "Not a same-day pair",
    np.where(
        time_audit["Time of data collection"].isna()
        | time_audit["Next recorded time"].isna(),
        "Missing time",
        np.where(
            time_audit["Time of data collection"]
            == time_audit["Next recorded time"],
            "Same recorded time",
            "Different recorded times"
        )
    )
)

corrected_summary = (
    time_audit.loc[
        time_audit["Days until next observation"] == 0,
        "Time comparison"
    ]
    .value_counts()
    .rename_axis("Time comparison")
    .reset_index(name="Pair count")
)

display(corrected_summary)

,Time comparison,Pair count
0,Different recorded times,5950
1,Same recorded time,2


In [5]:
# CELL 5: INSPECT RECORDED TIME FORMATS

time_col = (
    ordered_df["Time of data collection"]
    .astype("string")
    .str.strip()
)

time_counts = (
    time_col.value_counts(dropna=False)
    .head(20)
    .rename_axis("Recorded time")
    .reset_index(name="Frequency")
)

display(time_counts)

,Recorded time,Frequency
0,07:10,216
1,07:15,196
2,16:10,175
3,07:30,159
4,06:45,159
5,07:20,147
6,16:30,145
7,07:05,137
8,16:15,128
9,16:20,126


In [6]:
# CELL 6: VERIFY RECORDED TIME FORMAT

time_col = (
    ordered_df["Time of data collection"]
    .astype("string")
    .str.strip()
)

parsed_time = pd.to_datetime(
    time_col,
    format="%H:%M",
    errors="coerce"
)

missing_count = time_col.isna().sum()
unparsed_count = (time_col.notna() & parsed_time.isna()).sum()
parsed_count = parsed_time.notna().sum()

time_format_audit = pd.DataFrame({
    "Metric": [
        "Total records",
        "Successfully parsed times",
        "Missing times",
        "Non-missing times that failed parsing"
    ],
    "Count": [
        len(time_col),
        parsed_count,
        missing_count,
        unparsed_count
    ]
})

display(time_format_audit)

,Metric,Count
0,Total records,12431
1,Successfully parsed times,12431
2,Missing times,0
3,Non-missing times that failed parsing,0


In [7]:
# CELL 7: REBUILD OBSERVATIONS IN CHRONOLOGICAL ORDER

chronological_df = ordered_df.copy()

# Combine the verified date and time into one timestamp.
chronological_df["Observation timestamp"] = pd.to_datetime(
    chronological_df["Parsed date"].dt.strftime("%Y-%m-%d")
    + " "
    + chronological_df["Time of data collection"].astype("string").str.strip(),
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

# Sort every pond's observations by their actual timestamps.
chronological_df = (
    chronological_df
    .sort_values(
        ["pond_id", "Observation timestamp"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

# Identify the next observation for the same pond.
chronological_df["Next observation timestamp"] = (
    chronological_df.groupby("pond_id")["Observation timestamp"]
    .shift(-1)
)

# Calculate the elapsed time between consecutive observations.
chronological_df["Hours until next observation"] = (
    chronological_df["Next observation timestamp"]
    - chronological_df["Observation timestamp"]
).dt.total_seconds() / 3600

chronological_df["Days until next observation"] = (
    chronological_df["Hours until next observation"] / 24
)

# Keep records that have a subsequent observation.
consecutive_pairs = chronological_df.dropna(
    subset=["Next observation timestamp"]
).copy()

# Summarise the corrected chronological intervals.
positive_intervals = consecutive_pairs.loc[
    consecutive_pairs["Hours until next observation"] > 0,
    "Days until next observation"
]

same_timestamp_count = (
    consecutive_pairs["Hours until next observation"] == 0
).sum()

chronology_summary = pd.DataFrame({
    "Metric": [
        "Consecutive observation pairs",
        "Pairs with identical timestamps (ambiguous order)",
        "Pairs with positive time gaps",
        "Pairs separated by 24–72 hours",
        "Pairs with positive gaps up to 30 days",
        "Median positive interval (days)",
        "Mean positive interval (days)",
        "90th percentile positive interval (days)",
        "Maximum positive interval (days)"
    ],
    "Value": [
        len(consecutive_pairs),
        int(same_timestamp_count),
        len(positive_intervals),
        int(
            consecutive_pairs["Hours until next observation"]
            .between(24, 72, inclusive="both").sum()
        ),
        int(
            consecutive_pairs["Hours until next observation"]
            .between(0, 30 * 24, inclusive="right").sum()
        ),
        round(positive_intervals.median(), 2),
        round(positive_intervals.mean(), 2),
        round(positive_intervals.quantile(0.90), 2),
        round(positive_intervals.max(), 2)
    ]
})

display(chronology_summary)

,Metric,Value
0,Consecutive observation pairs,12114.00
1,Pairs with identical timestamps (ambiguous order),2.00
2,Pairs with positive time gaps,12112.00
3,Pairs separated by 24–72 hours,481.00
4,Pairs with positive gaps up to 30 days,10493.00
5,Median positive interval (days),1.57
6,Mean positive interval (days),13.28
7,90th percentile positive interval (days),32.99
8,Maximum positive interval (days),1097.55


In [8]:
# CELL 8: IDENTIFY AMBIGUOUS TIMESTAMP TRANSITIONS

chronological_df["Timestamp duplicated within pond"] = (
    chronological_df.duplicated(
        subset=["pond_id", "Observation timestamp"],
        keep=False
    )
)

chronological_df["Next timestamp duplicated"] = (
    chronological_df.groupby("pond_id")[
        "Timestamp duplicated within pond"
    ]
    .shift(-1)
    .fillna(False)
    .astype(bool)
)

pair_audit = chronological_df.dropna(
    subset=["Next observation timestamp"]
).copy()

pair_audit["Ambiguous timestamp transition"] = (
    pair_audit["Timestamp duplicated within pond"]
    | pair_audit["Next timestamp duplicated"]
)

duplicate_rows = int(
    chronological_df["Timestamp duplicated within pond"].sum()
)

duplicate_groups = (
    chronological_df.loc[
        chronological_df["Timestamp duplicated within pond"]
    ]
    .groupby(["pond_id", "Observation timestamp"])
    .ngroups
)

ambiguous_pairs = int(
    pair_audit["Ambiguous timestamp transition"].sum()
)

timestamp_audit_summary = pd.DataFrame({
    "Metric": [
        "Total consecutive pairs",
        "Rows belonging to duplicated pond timestamps",
        "Duplicated pond-timestamp groups",
        "Transitions touching duplicated timestamps",
        "Unambiguous consecutive transitions"
    ],
    "Count": [
        len(pair_audit),
        duplicate_rows,
        duplicate_groups,
        ambiguous_pairs,
        len(pair_audit) - ambiguous_pairs
    ]
})

display(timestamp_audit_summary)

,Metric,Count
0,Total consecutive pairs,12114
1,Rows belonging to duplicated pond timestamps,4
2,Duplicated pond-timestamp groups,2
3,Transitions touching duplicated timestamps,4
4,Unambiguous consecutive transitions,12110


In [9]:
# CELL 10: CORRECTED FUTURE-PREDICTION DATASET CONSTRUCTION

# Work on a copy; preserve the original chronological dataset.
prediction_work = chronological_df.copy()

# Identify observations sharing the same pond and timestamp.
prediction_work["Timestamp duplicated within pond"] = (
    prediction_work.duplicated(
        subset=["pond_id", "Observation timestamp"],
        keep=False
    )
)

# Identify whether the next observation has a duplicated timestamp.
prediction_work["Next timestamp duplicated"] = (
    prediction_work.groupby("pond_id")[
        "Timestamp duplicated within pond"
    ]
    .shift(-1)
    .fillna(False)
    .astype(bool)
)

# Mark transitions whose chronological order is ambiguous.
prediction_work["Ambiguous timestamp transition"] = (
    prediction_work["Timestamp duplicated within pond"]
    | prediction_work["Next timestamp duplicated"]
)

# Attach the next observation's water-quality status.
prediction_work["Next water quality status"] = (
    prediction_work.groupby("pond_id")["Is WQ in range?"]
    .shift(-1)
)

# Keep rows that have a subsequent observation.
prediction_pairs = prediction_work.dropna(
    subset=["Next observation timestamp"]
).copy()

# Exclude transitions involving duplicated timestamps.
prediction_pairs = prediction_pairs.loc[
    ~prediction_pairs["Ambiguous timestamp transition"]
].copy()

# Check availability of the three initial predictors.
predictor_columns = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

missing_predictor_rows = (
    prediction_pairs[predictor_columns]
    .isna()
    .any(axis=1)
    .sum()
)

complete_predictor_rows = (
    prediction_pairs[predictor_columns]
    .notna()
    .all(axis=1)
    .sum()
)

# Count the next-observation outcomes.
next_status_counts = (
    prediction_pairs["Next water quality status"]
    .value_counts(dropna=False)
)

other_status_count = sum(
    count
    for status, count in next_status_counts.items()
    if pd.notna(status) and status not in ["Yes", "No"]
)

prediction_dataset_summary = pd.DataFrame({
    "Metric": [
        "Unambiguous prediction pairs",
        "Pairs with a known next water-quality status",
        "Pairs with a missing next water-quality status",
        "Next status = Yes",
        "Next status = No",
        "Other next-status values",
        "Rows with all three current predictors available",
        "Rows missing at least one current predictor"
    ],
    "Count": [
        len(prediction_pairs),
        prediction_pairs["Next water quality status"].notna().sum(),
        prediction_pairs["Next water quality status"].isna().sum(),
        next_status_counts.get("Yes", 0),
        next_status_counts.get("No", 0),
        other_status_count,
        complete_predictor_rows,
        missing_predictor_rows
    ]
})

display(prediction_dataset_summary)

,Metric,Count
0,Unambiguous prediction pairs,12110
1,Pairs with a known next water-quality status,12110
2,Pairs with a missing next water-quality status,0
3,Next status = Yes,9838
4,Next status = No,2272
5,Other next-status values,0
6,Rows with all three current predictors available,12082
7,Rows missing at least one current predictor,28


In [10]:
# CELL 11: AUDIT MISSING PREDICTORS AND FUTURE OUTCOMES

predictor_columns = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

# Count missing values separately for each predictor.
missing_by_feature = pd.DataFrame({
    "Section": "Missing values by predictor",
    "Metric": predictor_columns,
    "Count": [
        prediction_pairs[column].isna().sum()
        for column in predictor_columns
    ]
})

missing_by_feature["Percentage of all pairs"] = (
    missing_by_feature["Count"] / len(prediction_pairs) * 100
)

# Classify each prediction pair by predictor completeness.
prediction_pairs["Predictor completeness"] = np.where(
    prediction_pairs[predictor_columns].notna().all(axis=1),
    "Complete predictors",
    "At least one missing predictor"
)

# Compare predictor completeness with the next-observation outcome.
completeness_outcome = (
    prediction_pairs
    .groupby(
        ["Predictor completeness", "Next water quality status"],
        observed=True
    )
    .size()
    .reset_index(name="Count")
)

completeness_outcome["Section"] = (
    "Predictor completeness vs future outcome"
)

completeness_outcome["Metric"] = (
    completeness_outcome["Predictor completeness"]
    + " — future status "
    + completeness_outcome["Next water quality status"].astype(str)
)

completeness_outcome["Percentage of all pairs"] = (
    completeness_outcome["Count"] / len(prediction_pairs) * 100
)

# Combine both checks into one output table.
missing_summary = missing_by_feature[
    ["Section", "Metric", "Count", "Percentage of all pairs"]
]

outcome_summary = completeness_outcome[
    ["Section", "Metric", "Count", "Percentage of all pairs"]
]

cell_11_summary = pd.concat(
    [missing_summary, outcome_summary],
    ignore_index=True
)

cell_11_summary["Percentage of all pairs"] = (
    cell_11_summary["Percentage of all pairs"].round(2)
)

display(cell_11_summary)

,Section,Metric,Count,Percentage of all pairs
0,Missing values by predictor,pH,16,0.13
1,Missing values by predictor,DO (mg/L),17,0.14
2,Missing values by predictor,Temp (in °C),18,0.15
3,Predictor completeness vs future outcome,At least one missing predictor — future status No,7,0.06
4,Predictor completeness vs future outcome,At least one missing predictor — future status...,21,0.17
5,Predictor completeness vs future outcome,Complete predictors — future status No,2265,18.70
6,Predictor completeness vs future outcome,Complete predictors — future status Yes,9817,81.07


In [11]:
# CELL 12: CREATE THE INITIAL MODELLING DATASET

predictor_columns = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

model_columns = [
    "pond_id",
    "Observation timestamp",
    "Next observation timestamp",
    "Hours until next observation",
    "Days until next observation",
    "pH",
    "DO (mg/L)",
    "Temp (in °C)",
    "Next water quality status"
]

# Keep only complete predictors and valid binary outcomes.
baseline_model_data = prediction_pairs.loc[
    prediction_pairs[predictor_columns].notna().all(axis=1)
    & prediction_pairs["Next water quality status"].isin(["Yes", "No"]),
    model_columns
].copy()

# Summarise the initial modelling dataset.
total_rows = len(baseline_model_data)

yes_count = (
    baseline_model_data["Next water quality status"] == "Yes"
).sum()

no_count = (
    baseline_model_data["Next water quality status"] == "No"
).sum()

modeling_summary = pd.DataFrame({
    "Metric": [
        "Original unambiguous prediction pairs",
        "Complete prediction pairs retained",
        "Prediction pairs excluded",
        "Unique ponds represented",
        "Future status = Yes",
        "Future status = No",
        "Future status = Yes (%)",
        "Future status = No (%)",
        "Rows with missing predictor values",
        "Rows with missing future outcome",
        "Median interval between observations (days)"
    ],
    "Value": [
        len(prediction_pairs),
        total_rows,
        len(prediction_pairs) - total_rows,
        baseline_model_data["pond_id"].nunique(),
        yes_count,
        no_count,
        round(yes_count / total_rows * 100, 2),
        round(no_count / total_rows * 100, 2),
        int(baseline_model_data[predictor_columns].isna().any(axis=1).sum()),
        int(baseline_model_data["Next water quality status"].isna().sum()),
        round(
            baseline_model_data["Days until next observation"].median(),
            2
        )
    ]
})

display(modeling_summary)

,Metric,Value
0,Original unambiguous prediction pairs,12110.00
1,Complete prediction pairs retained,12082.00
2,Prediction pairs excluded,28.00
3,Unique ponds represented,316.00
4,Future status = Yes,9817.00
5,Future status = No,2265.00
6,Future status = Yes (%),81.25
7,Future status = No (%),18.75
8,Rows with missing predictor values,0.00
9,Rows with missing future outcome,0.00


In [12]:
# CELL 13: ASSESS CANDIDATE TEMPORAL VALIDATION SPLIT

# Use the 80th percentile of the next-observation timestamps as the cutoff.
candidate_cutoff = baseline_model_data[
    "Next observation timestamp"
].quantile(0.80)

# Training examples must have their target observation before the cutoff.
training_mask = (
    baseline_model_data["Next observation timestamp"]
    < candidate_cutoff
)

# Testing examples must begin at or after the cutoff.
testing_mask = (
    baseline_model_data["Observation timestamp"]
    >= candidate_cutoff
)

# Identify pairs that cross the cutoff.
boundary_mask = (
    (baseline_model_data["Observation timestamp"] < candidate_cutoff)
    & (baseline_model_data["Next observation timestamp"] >= candidate_cutoff)
)

temporal_training = baseline_model_data.loc[training_mask].copy()
temporal_testing = baseline_model_data.loc[testing_mask].copy()

def temporal_split_summary(name, data):
    yes_count = (
        data["Next water quality status"] == "Yes"
    ).sum()

    no_count = (
        data["Next water quality status"] == "No"
    ).sum()

    return {
        "Period": name,
        "Prediction pairs": len(data),
        "Unique ponds": data["pond_id"].nunique(),
        "Future status Yes": yes_count,
        "Future status No": no_count,
        "Future status No (%)": round(
            no_count / len(data) * 100, 2
        ) if len(data) else np.nan,
        "First observation": (
            data["Observation timestamp"].min()
        ),
        "Last next observation": (
            data["Next observation timestamp"].max()
        )
    }

split_summary = pd.DataFrame([
    {
        "Period": "Candidate cutoff",
        "Prediction pairs": 0,
        "Unique ponds": 0,
        "Future status Yes": 0,
        "Future status No": 0,
        "Future status No (%)": np.nan,
        "First observation": candidate_cutoff,
        "Last next observation": candidate_cutoff
    },
    temporal_split_summary("Training period", temporal_training),
    temporal_split_summary("Testing period", temporal_testing),
    {
        "Period": "Excluded boundary pairs",
        "Prediction pairs": int(boundary_mask.sum()),
        "Unique ponds": baseline_model_data.loc[
            boundary_mask, "pond_id"
        ].nunique(),
        "Future status Yes": np.nan,
        "Future status No": np.nan,
        "Future status No (%)": np.nan,
        "First observation": baseline_model_data.loc[
            boundary_mask, "Observation timestamp"
        ].min(),
        "Last next observation": baseline_model_data.loc[
            boundary_mask, "Next observation timestamp"
        ].max()
    }
])

display(split_summary)

,Period,Prediction pairs,Unique ponds,Future status Yes,Future status No,Future status No (%),First observation,Last next observation
0,Candidate cutoff,0,0,0.0,0.0,NaN,2025-11-14 07:44:00,2025-11-14 07:44:00
1,Training period,9665,296,7723.0,1942.0,20.09,2021-07-04 08:20:00,2025-11-14 07:20:00
2,Testing period,2298,138,1982.0,316.0,13.75,2025-11-14 07:50:00,2026-08-06 16:22:00
3,Excluded boundary pairs,119,119,NaN,NaN,NaN,2023-06-16 17:18:00,2026-06-18 06:32:00


In [13]:
# CELL 14: AUDIT POND OVERLAP BETWEEN TRAINING AND TESTING

training_ponds = set(temporal_training["pond_id"].unique())
testing_ponds = set(temporal_testing["pond_id"].unique())

shared_ponds = training_ponds & testing_ponds
training_only_ponds = training_ponds - testing_ponds
testing_only_ponds = testing_ponds - training_ponds

# Separate test observations from previously seen and unseen ponds.
test_shared = temporal_testing[
    temporal_testing["pond_id"].isin(shared_ponds)
].copy()

test_unseen = temporal_testing[
    ~temporal_testing["pond_id"].isin(shared_ponds)
].copy()

def test_group_summary(group_name, data):
    no_count = (
        data["Next water quality status"] == "No"
    ).sum()

    return {
        "Test group": group_name,
        "Prediction pairs": len(data),
        "Unique ponds": data["pond_id"].nunique(),
        "Future status No": int(no_count),
        "Future status No (%)": round(
            no_count / len(data) * 100, 2
        ) if len(data) else 0
    }

pond_overlap_summary = pd.DataFrame([
    {
        "Test group": "Pond overlap summary",
        "Prediction pairs": np.nan,
        "Unique ponds": len(shared_ponds),
        "Future status No": np.nan,
        "Future status No (%)": np.nan
    },
    {
        "Test group": "Training-only ponds",
        "Prediction pairs": np.nan,
        "Unique ponds": len(training_only_ponds),
        "Future status No": np.nan,
        "Future status No (%)": np.nan
    },
    {
        "Test group": "Testing-only ponds",
        "Prediction pairs": np.nan,
        "Unique ponds": len(testing_only_ponds),
        "Future status No": np.nan,
        "Future status No (%)": np.nan
    },
    test_group_summary("Test observations from shared ponds", test_shared),
    test_group_summary("Test observations from unseen ponds", test_unseen)
])

display(pond_overlap_summary)

,Test group,Prediction pairs,Unique ponds,Future status No,Future status No (%)
0,Pond overlap summary,NaN,118,NaN,NaN
1,Training-only ponds,NaN,178,NaN,NaN
2,Testing-only ponds,NaN,20,NaN,NaN
3,Test observations from shared ponds,2232.0,118,312.0,13.98
4,Test observations from unseen ponds,66.0,20,4.0,6.06


In [14]:
# CELL 15: PREPARE TRAINING AND TESTING INPUTS

# Initial predictor variables.
predictor_columns = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

# Encode the future outcome.
# 1 = Out of range (No)
# 0 = Within recorded range (Yes)
target_mapping = {
    "Yes": 0,
    "No": 1
}

# Primary training and testing datasets.
X_train = temporal_training[predictor_columns].copy()
X_test_primary = test_shared[predictor_columns].copy()

y_train = (
    temporal_training["Next water quality status"]
    .map(target_mapping)
)

y_test_primary = (
    test_shared["Next water quality status"]
    .map(target_mapping)
)

# Supplementary test dataset for previously unseen ponds.
X_test_unseen = test_unseen[predictor_columns].copy()

y_test_unseen = (
    test_unseen["Next water quality status"]
    .map(target_mapping)
)

# Summarise the prepared datasets.
prepared_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Primary testing — previously monitored ponds",
        "Supplementary testing — unseen ponds"
    ],
    "Observations": [
        len(X_train),
        len(X_test_primary),
        len(X_test_unseen)
    ],
    "Predictor columns": [
        X_train.shape[1],
        X_test_primary.shape[1],
        X_test_unseen.shape[1]
    ],
    "Out-of-range cases (1)": [
        int(y_train.sum()),
        int(y_test_primary.sum()),
        int(y_test_unseen.sum())
    ],
    "Within-range cases (0)": [
        int((y_train == 0).sum()),
        int((y_test_primary == 0).sum()),
        int((y_test_unseen == 0).sum())
    ],
    "Missing predictor values": [
        int(X_train.isna().any(axis=1).sum()),
        int(X_test_primary.isna().any(axis=1).sum()),
        int(X_test_unseen.isna().any(axis=1).sum())
    ],
    "Missing target values": [
        int(y_train.isna().sum()),
        int(y_test_primary.isna().sum()),
        int(y_test_unseen.isna().sum())
    ]
})

display(prepared_summary)

,Dataset,Observations,Predictor columns,Out-of-range cases (1),Within-range cases (0),Missing predictor values,Missing target values
0,Training,9665,3,1942,7723,0,0
1,Primary testing — previously monitored ponds,2232,3,312,1920,0,0
2,Supplementary testing — unseen ponds,66,3,4,62,0,0


In [15]:
# CELL 16: BASELINE AND LOGISTIC REGRESSION EVALUATION

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Model 1: majority-class baseline.
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)

# Model 2: logistic regression.
# Scaling is fitted only on training data through the pipeline.
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_model.fit(X_train, y_train)

# Evaluate both models on the primary temporal test dataset.
models = {
    "Majority-class baseline": dummy_model,
    "Logistic Regression": logistic_model
}

results = []

for model_name, model in models.items():

    predictions = model.predict(X_test_primary)
    probabilities = model.predict_proba(X_test_primary)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test_primary, predictions),
        "Precision (out-of-range)": precision_score(
            y_test_primary, predictions, zero_division=0
        ),
        "Recall (out-of-range)": recall_score(
            y_test_primary, predictions, zero_division=0
        ),
        "F1 (out-of-range)": f1_score(
            y_test_primary, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_primary, predictions
        ),
        "Average precision": average_precision_score(
            y_test_primary, probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_test_primary, probabilities
        )
    })

benchmark_results = pd.DataFrame(results)

metric_columns = [
    "Accuracy",
    "Precision (out-of-range)",
    "Recall (out-of-range)",
    "F1 (out-of-range)",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

benchmark_results[metric_columns] = (
    benchmark_results[metric_columns].round(4)
)

display(benchmark_results)

,Model,Accuracy,Precision (out-of-range),Recall (out-of-range),F1 (out-of-range),Balanced accuracy,Average precision,ROC-AUC
0,Majority-class baseline,0.8602,0.0000,0.0000,0.0000,0.5000,0.1398,0.5000
1,Logistic Regression,0.8629,0.6875,0.0353,0.0671,0.5163,0.3797,0.6501


In [16]:
# CELL 17: TEMPORAL VALIDATION AND CLASSIFICATION THRESHOLD ANALYSIS

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

# Establish a chronological validation cutoff within the training period.
validation_cutoff = temporal_training[
    "Next observation timestamp"
].quantile(0.80)

# Keep inner-training pairs whose future outcomes occur before the cutoff.
inner_train_mask = (
    temporal_training["Next observation timestamp"]
    < validation_cutoff
)

# Validation pairs must begin at or after the cutoff.
validation_mask = (
    temporal_training["Observation timestamp"]
    >= validation_cutoff
)

# Exclude pairs crossing the validation boundary.
inner_training = temporal_training.loc[inner_train_mask].copy()
validation_data = temporal_training.loc[validation_mask].copy()

# Extract predictors and encode future outcomes.
X_inner_train = inner_training[predictor_columns]
y_inner_train = inner_training["Next water quality status"].map(
    target_mapping
)

X_validation = validation_data[predictor_columns]
y_validation = validation_data["Next water quality status"].map(
    target_mapping
)

# Train a fresh model using only the earlier inner-training period.
validation_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

validation_model.fit(X_inner_train, y_inner_train)

# Generate probabilities for the later validation period.
validation_probabilities = validation_model.predict_proba(
    X_validation
)[:, 1]

# Compare candidate classification thresholds.
threshold_results = []

for threshold in np.arange(0.10, 1.00, 0.10):

    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions,
        labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_validation, predictions
        )
    })

threshold_results = pd.DataFrame(threshold_results)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy"
]

threshold_results[metric_columns] = (
    threshold_results[metric_columns].round(4)
)

display(threshold_results)

,Threshold,True positives,False positives,False negatives,True negatives,Precision,Recall,F1-score,Balanced accuracy
0,0.1,143,1673,0,13,0.0787,1.0000,0.1460,0.5039
1,0.2,117,1019,26,667,0.1030,0.8182,0.1830,0.6069
2,0.3,37,193,106,1493,0.1609,0.2587,0.1984,0.5721
3,0.4,21,18,122,1668,0.5385,0.1469,0.2308,0.5681
4,0.5,8,5,135,1681,0.6154,0.0559,0.1026,0.5265
5,0.6,2,1,141,1685,0.6667,0.0140,0.0274,0.5067
6,0.7,0,0,143,1686,0.0000,0.0000,0.0000,0.5000
7,0.8,0,0,143,1686,0.0000,0.0000,0.0000,0.5000
8,0.9,0,0,143,1686,0.0000,0.0000,0.0000,0.5000


In [17]:
# CELL 18: FINAL TEMPORAL TEST EVALUATION

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

# Obtain predicted probabilities for the untouched primary test set.
test_probabilities = logistic_model.predict_proba(
    X_test_primary
)[:, 1]

# Compare the default threshold with the validation-selected threshold.
candidate_thresholds = [0.50, 0.20]

final_test_results = []

for threshold in candidate_thresholds:

    predictions = (
        test_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_primary,
        predictions,
        labels=[0, 1]
    ).ravel()

    final_test_results.append({
        "Threshold": threshold,
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(
            predictions.mean() * 100, 2
        ),
        "Accuracy": accuracy_score(
            y_test_primary, predictions
        ),
        "Precision (out-of-range)": precision_score(
            y_test_primary, predictions, zero_division=0
        ),
        "Recall (out-of-range)": recall_score(
            y_test_primary, predictions, zero_division=0
        ),
        "F1-score (out-of-range)": f1_score(
            y_test_primary, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_primary, predictions
        )
    })

final_test_summary = pd.DataFrame(final_test_results)

metric_columns = [
    "Alert rate (%)",
    "Accuracy",
    "Precision (out-of-range)",
    "Recall (out-of-range)",
    "F1-score (out-of-range)",
    "Balanced accuracy"
]

final_test_summary[metric_columns] = (
    final_test_summary[metric_columns].round(4)
)

display(final_test_summary)

,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Accuracy,Precision (out-of-range),Recall (out-of-range),F1-score (out-of-range),Balanced accuracy
0,0.5,11,5,301,1915,0.72,0.8629,0.6875,0.0353,0.0671,0.5163
1,0.2,191,863,121,1057,47.22,0.5591,0.1812,0.6122,0.2796,0.5814


In [18]:
# CELL 18: FINAL TEMPORAL TEST EVALUATION

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

# Obtain predicted probabilities for the untouched primary test set.
test_probabilities = logistic_model.predict_proba(
    X_test_primary
)[:, 1]

# Compare the default threshold with the validation-selected threshold.
candidate_thresholds = [0.50, 0.20]

final_test_results = []

for threshold in candidate_thresholds:

    predictions = (
        test_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_primary,
        predictions,
        labels=[0, 1]
    ).ravel()

    final_test_results.append({
        "Threshold": threshold,
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(
            predictions.mean() * 100, 2
        ),
        "Accuracy": accuracy_score(
            y_test_primary, predictions
        ),
        "Precision (out-of-range)": precision_score(
            y_test_primary, predictions, zero_division=0
        ),
        "Recall (out-of-range)": recall_score(
            y_test_primary, predictions, zero_division=0
        ),
        "F1-score (out-of-range)": f1_score(
            y_test_primary, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_primary, predictions
        )
    })

final_test_summary = pd.DataFrame(final_test_results)

metric_columns = [
    "Alert rate (%)",
    "Accuracy",
    "Precision (out-of-range)",
    "Recall (out-of-range)",
    "F1-score (out-of-range)",
    "Balanced accuracy"
]

final_test_summary[metric_columns] = (
    final_test_summary[metric_columns].round(4)
)

display(final_test_summary)

,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Accuracy,Precision (out-of-range),Recall (out-of-range),F1-score (out-of-range),Balanced accuracy
0,0.5,11,5,301,1915,0.72,0.8629,0.6875,0.0353,0.0671,0.5163
1,0.2,191,863,121,1057,47.22,0.5591,0.1812,0.6122,0.2796,0.5814


In [19]:
# CELL 19: PERFORMANCE BY OBSERVATION TIME

# Reuse the untouched primary test predictions.
selected_threshold = 0.20

test_diagnostic = test_shared[
    ["pond_id", "Next water quality status"]
].copy()

# Attach the original recorded measurement times.
test_diagnostic["Recorded time"] = (
    prediction_pairs.loc[
        test_shared.index,
        "Time of data collection"
    ].astype("string").str.strip()
)

# Attach predicted probabilities and classifications.
test_diagnostic["Predicted probability"] = test_probabilities

test_diagnostic["Predicted out-of-range"] = (
    test_diagnostic["Predicted probability"]
    >= selected_threshold
).astype(int)

test_diagnostic["Actual out-of-range"] = (
    test_diagnostic["Next water quality status"]
    .map(target_mapping)
)

# Classify measurements by time of day.
parsed_test_times = pd.to_datetime(
    test_diagnostic["Recorded time"],
    format="%H:%M",
    errors="coerce"
)

test_diagnostic["Observation period"] = np.where(
    parsed_test_times.dt.hour < 12,
    "Morning (before 12:00)",
    "Later day (12:00 onward)"
)

# Calculate performance separately for each period.
time_results = []

for period, group in test_diagnostic.groupby("Observation period"):

    tn, fp, fn, tp = confusion_matrix(
        group["Actual out-of-range"],
        group["Predicted out-of-range"],
        labels=[0, 1]
    ).ravel()

    time_results.append({
        "Observation period": period,
        "Prediction pairs": len(group),
        "Actual out-of-range cases": int(group["Actual out-of-range"].sum()),
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Precision": precision_score(
            group["Actual out-of-range"],
            group["Predicted out-of-range"],
            zero_division=0
        ),
        "Recall": recall_score(
            group["Actual out-of-range"],
            group["Predicted out-of-range"],
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            group["Actual out-of-range"],
            group["Predicted out-of-range"]
        )
    })

time_performance_summary = pd.DataFrame(time_results)

metric_columns = [
    "Precision",
    "Recall",
    "Balanced accuracy"
]

time_performance_summary[metric_columns] = (
    time_performance_summary[metric_columns].round(4)
)

display(time_performance_summary)

,Observation period,Prediction pairs,Actual out-of-range cases,True positives,False positives,False negatives,True negatives,Precision,Recall,Balanced accuracy
0,Later day (12:00 onward),1014,155,50,169,105,690,0.2283,0.3226,0.5629
1,Morning (before 12:00),1218,157,141,694,16,367,0.1689,0.8981,0.6220


In [20]:
# CELL 20: EXAMINE MEASUREMENTS BY TIME OF DAY AND FUTURE OUTCOME

# Retrieve the original records corresponding to the primary test set.
time_analysis = prediction_pairs.loc[test_shared.index].copy()

# Classify the earlier observation's recorded time.
recorded_times = pd.to_datetime(
    time_analysis["Time of data collection"].astype("string").str.strip(),
    format="%H:%M",
    errors="coerce"
)

time_analysis["Observation period"] = np.where(
    recorded_times.dt.hour < 12,
    "Morning (before 12:00)",
    "Later day (12:00 onward)"
)

# Group measurements by observation period and actual future outcome.
measurement_summary = (
    time_analysis
    .groupby(
        ["Observation period", "Next water quality status"],
        observed=True
    )
    .agg(
        Prediction_pairs=("pond_id", "size"),
        Median_DO=("DO (mg/L)", "median"),
        Mean_DO=("DO (mg/L)", "mean"),
        Median_pH=("pH", "median"),
        Median_temperature=("Temp (in °C)", "median")
    )
    .reset_index()
)

measurement_summary[
    ["Median_DO", "Mean_DO", "Median_pH", "Median_temperature"]
] = measurement_summary[
    ["Median_DO", "Mean_DO", "Median_pH", "Median_temperature"]
].round(3)

display(measurement_summary)

,Observation period,Next water quality status,Prediction_pairs,Median_DO,Mean_DO,Median_pH,Median_temperature
0,Later day (12:00 onward),No,155,11.3,11.540,8.45,30.7
1,Later day (12:00 onward),Yes,859,11.1,11.015,8.41,30.3
2,Morning (before 12:00),No,157,3.0,2.755,8.35,28.9
3,Morning (before 12:00),Yes,1061,3.3,3.377,8.18,28.4


In [21]:
# CELL 20: EXAMINE MEASUREMENTS BY TIME OF DAY AND FUTURE OUTCOME

# Retrieve the original records corresponding to the primary test set.
time_analysis = prediction_pairs.loc[test_shared.index].copy()

# Classify the earlier observation's recorded time.
recorded_times = pd.to_datetime(
    time_analysis["Time of data collection"].astype("string").str.strip(),
    format="%H:%M",
    errors="coerce"
)

time_analysis["Observation period"] = np.where(
    recorded_times.dt.hour < 12,
    "Morning (before 12:00)",
    "Later day (12:00 onward)"
)

# Group measurements by observation period and actual future outcome.
measurement_summary = (
    time_analysis
    .groupby(
        ["Observation period", "Next water quality status"],
        observed=True
    )
    .agg(
        Prediction_pairs=("pond_id", "size"),
        Median_DO=("DO (mg/L)", "median"),
        Mean_DO=("DO (mg/L)", "mean"),
        Median_pH=("pH", "median"),
        Median_temperature=("Temp (in °C)", "median")
    )
    .reset_index()
)

measurement_summary[
    ["Median_DO", "Mean_DO", "Median_pH", "Median_temperature"]
] = measurement_summary[
    ["Median_DO", "Mean_DO", "Median_pH", "Median_temperature"]
].round(3)

display(measurement_summary)

,Observation period,Next water quality status,Prediction_pairs,Median_DO,Mean_DO,Median_pH,Median_temperature
0,Later day (12:00 onward),No,155,11.3,11.540,8.45,30.7
1,Later day (12:00 onward),Yes,859,11.1,11.015,8.41,30.3
2,Morning (before 12:00),No,157,3.0,2.755,8.35,28.9
3,Morning (before 12:00),Yes,1061,3.3,3.377,8.18,28.4


In [22]:
# CELL 22: CORRECTED TIME-AWARE LOGISTIC REGRESSION

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

# Create additional features using the combined observation timestamp.
def create_time_features(data):

    features = data[predictor_columns].copy()

    observation_time = pd.to_datetime(
        data["Observation timestamp"],
        errors="coerce"
    )

    if observation_time.isna().any():
        raise ValueError("Some observation timestamps could not be parsed.")

    # Morning = 1; later day = 0.
    features["morning_measurement"] = (
        observation_time.dt.hour < 12
    ).astype(int)

    # Allow the relationship between DO and outcome to differ by time.
    features["do_x_morning"] = (
        features["DO (mg/L)"]
        * features["morning_measurement"]
    )

    return features


# Prepare the chronological training and validation datasets.
X_inner_time = create_time_features(inner_training)
X_validation_time = create_time_features(validation_data)

# Train the time-aware model using only the earlier training period.
time_aware_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

time_aware_model.fit(X_inner_time, y_inner_train)

# Generate validation probabilities from both models.
original_validation_probabilities = validation_model.predict_proba(
    X_validation
)[:, 1]

time_aware_validation_probabilities = time_aware_model.predict_proba(
    X_validation_time
)[:, 1]

# Compare both models using identical thresholds.
comparison_results = []

for model_name, probabilities in [
    ("Original Logistic Regression", original_validation_probabilities),
    ("Time-aware Logistic Regression", time_aware_validation_probabilities)
]:

    for threshold in np.arange(0.10, 1.00, 0.10):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        comparison_results.append({
            "Model": model_name,
            "Threshold": round(threshold, 2),
            "Alert rate (%)": round(predictions.mean() * 100, 2),
            "Precision": precision_score(
                y_validation, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_validation, predictions, zero_division=0
            ),
            "F1-score": f1_score(
                y_validation, predictions, zero_division=0
            ),
            "Balanced accuracy": balanced_accuracy_score(
                y_validation, predictions
            )
        })

time_model_comparison = pd.DataFrame(comparison_results)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy"
]

time_model_comparison[metric_columns] = (
    time_model_comparison[metric_columns].round(4)
)

display(time_model_comparison)

,Model,Threshold,Alert rate (%),Precision,Recall,F1-score,Balanced accuracy
0,Original Logistic Regression,0.1,99.29,0.0787,1.0000,0.1460,0.5039
1,Original Logistic Regression,0.2,62.11,0.1030,0.8182,0.1830,0.6069
2,Original Logistic Regression,0.3,12.58,0.1609,0.2587,0.1984,0.5721
3,Original Logistic Regression,0.4,2.13,0.5385,0.1469,0.2308,0.5681
4,Original Logistic Regression,0.5,0.71,0.6154,0.0559,0.1026,0.5265
5,Original Logistic Regression,0.6,0.16,0.6667,0.0140,0.0274,0.5067
6,Original Logistic Regression,0.7,0.00,0.0000,0.0000,0.0000,0.5000
7,Original Logistic Regression,0.8,0.00,0.0000,0.0000,0.0000,0.5000
8,Original Logistic Regression,0.9,0.00,0.0000,0.0000,0.0000,0.5000
9,Time-aware Logistic Regression,0.1,97.81,0.0799,1.0000,0.1480,0.5119


In [23]:
# CELL 23: FINAL TEST OF THE TIME-AWARE LOGISTIC REGRESSION

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score
)

# Prepare time-aware training and primary testing features.
X_train_time_final = create_time_features(temporal_training)
X_test_time_final = create_time_features(test_shared)

y_train_final = temporal_training[
    "Next water quality status"
].map(target_mapping)

y_test_final = test_shared[
    "Next water quality status"
].map(target_mapping)

# Train the time-aware model on the complete training period.
time_aware_final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

time_aware_final_model.fit(
    X_train_time_final,
    y_train_final
)

# Generate probabilities for the untouched primary test period.
original_probabilities = logistic_model.predict_proba(
    X_test_primary
)[:, 1]

time_aware_probabilities = time_aware_final_model.predict_proba(
    X_test_time_final
)[:, 1]

# Compare both models at the candidate thresholds.
final_comparison = []

for model_name, probabilities in [
    ("Original Logistic Regression", original_probabilities),
    ("Time-aware Logistic Regression", time_aware_probabilities)
]:

    for threshold in [0.20, 0.30, 0.50]:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_test_final,
            predictions,
            labels=[0, 1]
        ).ravel()

        final_comparison.append({
            "Model": model_name,
            "Threshold": threshold,
            "True positives": tp,
            "False positives": fp,
            "False negatives": fn,
            "True negatives": tn,
            "Alert rate (%)": round(
                predictions.mean() * 100, 2
            ),
            "Precision": precision_score(
                y_test_final, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test_final, predictions, zero_division=0
            ),
            "F1-score": f1_score(
                y_test_final, predictions, zero_division=0
            ),
            "Balanced accuracy": balanced_accuracy_score(
                y_test_final, predictions
            )
        })

final_comparison_df = pd.DataFrame(final_comparison)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy"
]

final_comparison_df[metric_columns] = (
    final_comparison_df[metric_columns].round(4)
)

display(final_comparison_df)

,Model,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Precision,Recall,F1-score,Balanced accuracy
0,Original Logistic Regression,0.2,191,863,121,1057,47.22,0.1812,0.6122,0.2796,0.5814
1,Original Logistic Regression,0.3,106,127,206,1793,10.44,0.4549,0.3397,0.3890,0.6368
2,Original Logistic Regression,0.5,11,5,301,1915,0.72,0.6875,0.0353,0.0671,0.5163
3,Time-aware Logistic Regression,0.2,187,731,125,1189,41.13,0.2037,0.5994,0.3041,0.6093
4,Time-aware Logistic Regression,0.3,101,91,211,1829,8.60,0.5260,0.3237,0.4008,0.6382
5,Time-aware Logistic Regression,0.5,37,12,275,1908,2.20,0.7551,0.1186,0.2050,0.5562


In [24]:
# CELL 24: COMPLETE VALIDATION-SELECTED THRESHOLD EVALUATION

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score
)

# All thresholds below were examined or selected using validation data.
selected_thresholds = [0.20, 0.30, 0.40, 0.50]

threshold_test_results = []

for threshold in selected_thresholds:

    predictions = (
        time_aware_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_final,
        predictions,
        labels=[0, 1]
    ).ravel()

    threshold_test_results.append({
        "Threshold": threshold,
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Accuracy": accuracy_score(y_test_final, predictions),
        "Precision": precision_score(
            y_test_final, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test_final, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_test_final, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_final, predictions
        )
    })

threshold_test_summary = pd.DataFrame(threshold_test_results)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy"
]

threshold_test_summary[metric_columns] = (
    threshold_test_summary[metric_columns].round(4)
)

display(threshold_test_summary)

,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Accuracy,Precision,Recall,F1-score,Balanced accuracy
0,0.2,187,731,125,1189,41.13,0.6165,0.2037,0.5994,0.3041,0.6093
1,0.3,101,91,211,1829,8.60,0.8647,0.5260,0.3237,0.4008,0.6382
2,0.4,64,32,248,1888,4.30,0.8746,0.6667,0.2051,0.3137,0.5942
3,0.5,37,12,275,1908,2.20,0.8714,0.7551,0.1186,0.2050,0.5562


In [25]:
# CELL 25: SUPPLEMENTARY EVALUATION ON UNSEEN PONDS

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score
)

# Prepare time-aware features for previously unseen ponds.
X_unseen_time = create_time_features(test_unseen)

# Preserve the actual future outcomes.
y_unseen = test_unseen[
    "Next water quality status"
].map(target_mapping)

# Use the already-trained time-aware model.
unseen_probabilities = time_aware_final_model.predict_proba(
    X_unseen_time
)[:, 1]

# Apply the threshold selected using validation data.
primary_threshold = 0.20

unseen_predictions = (
    unseen_probabilities >= primary_threshold
).astype(int)

# Calculate confusion matrix counts.
tn, fp, fn, tp = confusion_matrix(
    y_unseen,
    unseen_predictions,
    labels=[0, 1]
).ravel()

unseen_evaluation = pd.DataFrame({
    "Metric": [
        "Unseen-pond prediction pairs",
        "Unique unseen ponds",
        "Actual out-of-range cases",
        "True positives",
        "False positives",
        "False negatives",
        "True negatives",
        "Alert rate (%)",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced accuracy"
    ],
    "Value": [
        len(test_unseen),
        test_unseen["pond_id"].nunique(),
        int(y_unseen.sum()),
        tp,
        fp,
        fn,
        tn,
        round(unseen_predictions.mean() * 100, 2),
        accuracy_score(y_unseen, unseen_predictions),
        precision_score(y_unseen, unseen_predictions, zero_division=0),
        recall_score(y_unseen, unseen_predictions, zero_division=0),
        f1_score(y_unseen, unseen_predictions, zero_division=0),
        balanced_accuracy_score(y_unseen, unseen_predictions)
    ]
})

unseen_evaluation["Value"] = unseen_evaluation["Value"].round(4)

display(unseen_evaluation)

,Metric,Value
0,Unseen-pond prediction pairs,66.0000
1,Unique unseen ponds,20.0000
2,Actual out-of-range cases,4.0000
3,True positives,4.0000
4,False positives,45.0000
5,False negatives,0.0000
6,True negatives,17.0000
7,Alert rate (%),74.2400
8,Accuracy,0.3182
9,Precision,0.0816


In [26]:
# CELL 26: UNSEEN-POND PERFORMANCE AUDIT

unseen_pond_audit = test_unseen[
    ["pond_id"]
].copy()

unseen_pond_audit["Actual out-of-range"] = (
    np.asarray(y_unseen)
)

unseen_pond_audit["Predicted out-of-range"] = (
    unseen_predictions
)

unseen_pond_audit["True positive"] = (
    (unseen_pond_audit["Actual out-of-range"] == 1)
    & (unseen_pond_audit["Predicted out-of-range"] == 1)
).astype(int)

unseen_pond_audit["False positive"] = (
    (unseen_pond_audit["Actual out-of-range"] == 0)
    & (unseen_pond_audit["Predicted out-of-range"] == 1)
).astype(int)

unseen_pond_audit["False negative"] = (
    (unseen_pond_audit["Actual out-of-range"] == 1)
    & (unseen_pond_audit["Predicted out-of-range"] == 0)
).astype(int)

pond_level_summary = (
    unseen_pond_audit
    .groupby("pond_id")
    .agg(
        Prediction_pairs=("pond_id", "size"),
        Actual_out_of_range=("Actual out-of-range", "sum"),
        Predicted_alerts=("Predicted out-of-range", "sum"),
        True_positives=("True positive", "sum"),
        False_positives=("False positive", "sum"),
        False_negatives=("False negative", "sum")
    )
    .reset_index()
    .sort_values(
        ["Actual_out_of_range", "True_positives"],
        ascending=False
    )
)

display(pond_level_summary)

,pond_id,Prediction_pairs,Actual_out_of_range,Predicted_alerts,True_positives,False_positives,False_negatives
8,ara2_438777b9,5,2,4,2,2,0
17,ara2_d8a0ad32,5,2,4,2,2,0
0,ara2_0206e1e9,5,0,3,0,3,0
1,ara2_048a77e9,1,0,1,0,1,0
2,ara2_25e534f2,1,0,1,0,1,0
3,ara2_296bcb81,1,0,1,0,1,0
4,ara2_2bd14b5a,6,0,6,0,6,0
5,ara2_31503b0c,2,0,2,0,2,0
6,ara2_3c745150,6,0,6,0,6,0
7,ara2_41e97843,1,0,1,0,1,0


In [27]:
# CELL 27: TIME-AWARE MODEL PERFORMANCE BY OBSERVATION PERIOD

# Use the validation-selected threshold.
selected_threshold = 0.20

# Prepare the primary test observations.
period_audit = test_shared[
    ["pond_id", "Observation timestamp", "Next water quality status"]
].copy()

# Classify the measurement period.
recorded_timestamps = pd.to_datetime(
    period_audit["Observation timestamp"],
    errors="coerce"
)

period_audit["Observation period"] = np.where(
    recorded_timestamps.dt.hour < 12,
    "Morning (before 12:00)",
    "Later day (12:00 onward)"
)

# Attach predictions from both models.
period_audit["Original prediction"] = (
    original_probabilities >= selected_threshold
).astype(int)

period_audit["Time-aware prediction"] = (
    time_aware_probabilities >= selected_threshold
).astype(int)

period_audit["Actual outcome"] = np.asarray(y_test_final)

# Calculate metrics separately for each period and model.
period_results = []

for period, group in period_audit.groupby("Observation period"):

    for model_name, prediction_column in [
        ("Original Logistic Regression", "Original prediction"),
        ("Time-aware Logistic Regression", "Time-aware prediction")
    ]:

        actual = group["Actual outcome"]
        predicted = group[prediction_column]

        tn, fp, fn, tp = confusion_matrix(
            actual,
            predicted,
            labels=[0, 1]
        ).ravel()

        period_results.append({
            "Observation period": period,
            "Model": model_name,
            "Prediction pairs": len(group),
            "Actual out-of-range cases": int(actual.sum()),
            "True positives": tp,
            "False positives": fp,
            "False negatives": fn,
            "Precision": precision_score(
                actual, predicted, zero_division=0
            ),
            "Recall": recall_score(
                actual, predicted, zero_division=0
            ),
            "Balanced accuracy": balanced_accuracy_score(
                actual, predicted
            )
        })

period_model_comparison = pd.DataFrame(period_results)

metric_columns = ["Precision", "Recall", "Balanced accuracy"]

period_model_comparison[metric_columns] = (
    period_model_comparison[metric_columns].round(4)
)

display(period_model_comparison)

,Observation period,Model,Prediction pairs,Actual out-of-range cases,True positives,False positives,False negatives,Precision,Recall,Balanced accuracy
0,Later day (12:00 onward),Original Logistic Regression,1014,155,50,169,105,0.2283,0.3226,0.5629
1,Later day (12:00 onward),Time-aware Logistic Regression,1014,155,55,188,100,0.2263,0.3548,0.5680
2,Morning (before 12:00),Original Logistic Regression,1218,157,141,694,16,0.1689,0.8981,0.6220
3,Morning (before 12:00),Time-aware Logistic Regression,1218,157,132,543,25,0.1956,0.8408,0.6645


In [28]:
# CELL 28: MODEL PERFORMANCE BY REGION

region_audit = pd.DataFrame(index=test_shared.index)

# Retrieve the region associated with each test observation.
region_audit["Region"] = prediction_pairs.loc[
    test_shared.index, "region"
]

region_audit["Actual outcome"] = np.asarray(y_test_final)

region_audit["Original prediction"] = (
    original_probabilities >= 0.20
).astype(int)

region_audit["Time-aware prediction"] = (
    time_aware_probabilities >= 0.20
).astype(int)

region_results = []

for region, group in region_audit.groupby("Region"):

    actual = group["Actual outcome"]

    for model_name, prediction_column in [
        ("Original Logistic Regression", "Original prediction"),
        ("Time-aware Logistic Regression", "Time-aware prediction")
    ]:

        predicted = group[prediction_column]

        tn, fp, fn, tp = confusion_matrix(
            actual,
            predicted,
            labels=[0, 1]
        ).ravel()

        region_results.append({
            "Region": region,
            "Model": model_name,
            "Prediction pairs": len(group),
            "Actual out-of-range cases": int(actual.sum()),
            "True positives": tp,
            "False positives": fp,
            "False negatives": fn,
            "True negatives": tn,
            "Precision": precision_score(
                actual, predicted, zero_division=0
            ),
            "Recall": recall_score(
                actual, predicted, zero_division=0
            ),
            "Balanced accuracy": balanced_accuracy_score(
                actual, predicted
            )
        })

region_comparison = pd.DataFrame(region_results)

metric_columns = [
    "Precision",
    "Recall",
    "Balanced accuracy"
]

region_comparison[metric_columns] = (
    region_comparison[metric_columns].round(4)
)

display(region_comparison)

,Region,Model,Prediction pairs,Actual out-of-range cases,True positives,False positives,False negatives,True negatives,Precision,Recall,Balanced accuracy
0,Eluru,Original Logistic Regression,2232,312,191,863,121,1057,0.1812,0.6122,0.5814
1,Eluru,Time-aware Logistic Regression,2232,312,187,731,125,1189,0.2037,0.5994,0.6093


In [29]:
# CELL 29: AUDIT REGIONAL COVERAGE ACROSS MODELLING GROUPS

evaluation_groups = {
    "Training": temporal_training,
    "Primary testing — shared ponds": test_shared,
    "Supplementary testing — unseen ponds": test_unseen
}

regional_results = []

for group_name, group_data in evaluation_groups.items():

    # Retrieve the corresponding original records.
    original_records = prediction_pairs.loc[group_data.index]

    for region, region_data in original_records.groupby("region"):

        regional_results.append({
            "Dataset": group_name,
            "Region": region,
            "Prediction pairs": len(region_data),
            "Unique ponds": region_data["pond_id"].nunique(),
            "Future status Yes": (
                region_data["Next water quality status"] == "Yes"
            ).sum(),
            "Future status No": (
                region_data["Next water quality status"] == "No"
            ).sum(),
            "First observation": region_data["Observation timestamp"].min(),
            "Last next observation": region_data["Next observation timestamp"].max()
        })

regional_coverage_summary = pd.DataFrame(regional_results)

display(regional_coverage_summary)

,Dataset,Region,Prediction pairs,Unique ponds,Future status Yes,Future status No,First observation,Last next observation
0,Training,Eluru,6559,233,5261,1298,2021-07-12 06:15:00,2025-11-14 07:20:00
1,Training,Nellore,3106,63,2462,644,2021-07-04 08:20:00,2024-10-31 17:18:00
2,Primary testing — shared ponds,Eluru,2232,118,1920,312,2025-11-14 07:50:00,2026-08-06 16:22:00
3,Supplementary testing — unseen ponds,Eluru,66,20,62,4,2025-11-26 07:52:00,2026-07-31 16:52:00


In [30]:
# CELL 30: ASSESS NELLORE-SPECIFIC TEMPORAL VALIDATION

# Keep complete, unambiguous Nellore prediction pairs.
nellore_pairs = prediction_pairs.loc[
    (prediction_pairs["region"] == "Nellore")
    & prediction_pairs[predictor_columns].notna().all(axis=1)
    & prediction_pairs["Next water quality status"].isin(["Yes", "No"])
].copy()

# Define an 80% historical training / 20% later testing cutoff.
nellore_cutoff = nellore_pairs[
    "Next observation timestamp"
].quantile(0.80)

nellore_train_mask = (
    nellore_pairs["Next observation timestamp"] < nellore_cutoff
)

nellore_test_mask = (
    nellore_pairs["Observation timestamp"] >= nellore_cutoff
)

nellore_boundary_mask = (
    (nellore_pairs["Observation timestamp"] < nellore_cutoff)
    & (nellore_pairs["Next observation timestamp"] >= nellore_cutoff)
)

nellore_train = nellore_pairs.loc[nellore_train_mask].copy()
nellore_test = nellore_pairs.loc[nellore_test_mask].copy()

nellore_training_ponds = set(nellore_train["pond_id"].unique())
nellore_testing_ponds = set(nellore_test["pond_id"].unique())

nellore_split_summary = []

for group_name, group_data in [
    ("Nellore historical training", nellore_train),
    ("Nellore historical testing", nellore_test)
]:

    no_count = (
        group_data["Next water quality status"] == "No"
    ).sum()

    nellore_split_summary.append({
        "Period": group_name,
        "Prediction pairs": len(group_data),
        "Unique ponds": group_data["pond_id"].nunique(),
        "Future status Yes": (
            group_data["Next water quality status"] == "Yes"
        ).sum(),
        "Future status No": no_count,
        "Future status No (%)": round(
            no_count / len(group_data) * 100, 2
        ) if len(group_data) else np.nan,
        "First observation": group_data["Observation timestamp"].min(),
        "Last next observation": group_data["Next observation timestamp"].max()
    })

nellore_split_summary.append({
    "Period": "Excluded boundary pairs",
    "Prediction pairs": int(nellore_boundary_mask.sum()),
    "Unique ponds": nellore_pairs.loc[
        nellore_boundary_mask, "pond_id"
    ].nunique(),
    "Future status Yes": np.nan,
    "Future status No": np.nan,
    "Future status No (%)": np.nan,
    "First observation": nellore_pairs.loc[
        nellore_boundary_mask, "Observation timestamp"
    ].min(),
    "Last next observation": nellore_pairs.loc[
        nellore_boundary_mask, "Next observation timestamp"
    ].max()
})

nellore_split_summary.append({
    "Period": "Pond overlap between Nellore periods",
    "Prediction pairs": np.nan,
    "Unique ponds": len(nellore_training_ponds & nellore_testing_ponds),
    "Future status Yes": np.nan,
    "Future status No": np.nan,
    "Future status No (%)": np.nan,
    "First observation": nellore_cutoff,
    "Last next observation": nellore_cutoff
})

nellore_validation_summary = pd.DataFrame(nellore_split_summary)

display(nellore_validation_summary)

,Period,Prediction pairs,Unique ponds,Future status Yes,Future status No,Future status No (%),First observation,Last next observation
0,Nellore historical training,2484.0,63,1889.0,595.0,23.95,2021-07-04 08:20:00,2024-01-11 16:05:00
1,Nellore historical testing,573.0,49,529.0,44.0,7.68,2024-01-22 06:44:00,2024-10-31 17:18:00
2,Excluded boundary pairs,49.0,49,NaN,NaN,NaN,2022-08-06 16:30:00,2024-07-09 06:15:00
3,Pond overlap between Nellore periods,NaN,49,NaN,NaN,NaN,2024-01-22 06:44:00,2024-01-22 06:44:00


In [31]:
# CELL 31: CREATE NELLORE INTERNAL TEMPORAL VALIDATION SPLIT

# Use only the historical Nellore training data.
# The later Nellore testing period remains untouched.

nellore_validation_cutoff = nellore_train[
    "Next observation timestamp"
].quantile(0.80)

# Earlier observations are used for internal model training.
nellore_inner_train_mask = (
    nellore_train["Next observation timestamp"]
    < nellore_validation_cutoff
)

# Later observations form the internal validation set.
nellore_validation_mask = (
    nellore_train["Observation timestamp"]
    >= nellore_validation_cutoff
)

# Exclude transitions crossing the internal validation boundary.
nellore_validation_boundary_mask = (
    (nellore_train["Observation timestamp"] < nellore_validation_cutoff)
    & (nellore_train["Next observation timestamp"] >= nellore_validation_cutoff)
)

nellore_inner_training = nellore_train.loc[
    nellore_inner_train_mask
].copy()

nellore_validation = nellore_train.loc[
    nellore_validation_mask
].copy()

nellore_validation_boundary = nellore_train.loc[
    nellore_validation_boundary_mask
].copy()


# Summarise the three groups.
nellore_internal_groups = [
    ("Nellore internal training", nellore_inner_training),
    ("Nellore internal validation", nellore_validation),
    ("Excluded validation boundary pairs", nellore_validation_boundary)
]

nellore_internal_results = []

for group_name, group_data in nellore_internal_groups:

    no_count = (
        group_data["Next water quality status"] == "No"
    ).sum()

    nellore_internal_results.append({
        "Period": group_name,
        "Prediction pairs": len(group_data),
        "Unique ponds": group_data["pond_id"].nunique(),
        "Future status Yes": (
            group_data["Next water quality status"] == "Yes"
        ).sum(),
        "Future status No": no_count,
        "Future status No (%)": round(
            no_count / len(group_data) * 100, 2
        ) if len(group_data) else np.nan,
        "First observation": group_data["Observation timestamp"].min(),
        "Last next observation": group_data["Next observation timestamp"].max()
    })

nellore_internal_summary = pd.DataFrame(nellore_internal_results)

display(nellore_internal_summary)

,Period,Prediction pairs,Unique ponds,Future status Yes,Future status No,Future status No (%),First observation,Last next observation
0,Nellore internal training,1987,44,1466,521,26.22,2021-07-04 08:20:00,2023-06-22 17:35:00
1,Nellore internal validation,468,48,400,68,14.53,2023-07-03 06:56:00,2024-01-11 16:05:00
2,Excluded validation boundary pairs,29,29,23,6,20.69,2023-03-13 16:43:00,2023-09-26 07:04:00


In [32]:
# CELL 32: TRAIN AND VALIDATE THE NELLORE TIME-AWARE MODEL

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Prepare Nellore internal training and validation features.
X_nellore_inner = create_time_features(nellore_inner_training)
X_nellore_validation = create_time_features(nellore_validation)

# Encode future water-quality outcomes.
y_nellore_inner = (
    nellore_inner_training["Next water quality status"]
    .map(target_mapping)
)

y_nellore_validation = (
    nellore_validation["Next water quality status"]
    .map(target_mapping)
)

# Train a separate Nellore model.
nellore_time_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

nellore_time_model.fit(
    X_nellore_inner,
    y_nellore_inner
)

# Predict probabilities on the internal validation period.
nellore_validation_probabilities = (
    nellore_time_model.predict_proba(X_nellore_validation)[:, 1]
)

# Evaluate candidate thresholds.
nellore_threshold_results = []

for threshold in np.arange(0.10, 1.00, 0.10):

    predictions = (
        nellore_validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_nellore_validation,
        predictions,
        labels=[0, 1]
    ).ravel()

    nellore_threshold_results.append({
        "Threshold": round(threshold, 2),
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Precision": precision_score(
            y_nellore_validation,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_nellore_validation,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_nellore_validation,
            predictions,
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_nellore_validation,
            predictions
        )
    })

nellore_threshold_summary = pd.DataFrame(
    nellore_threshold_results
)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy"
]

nellore_threshold_summary[metric_columns] = (
    nellore_threshold_summary[metric_columns].round(4)
)

display(nellore_threshold_summary)

,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Precision,Recall,F1-score,Balanced accuracy
0,0.1,68,375,0,25,94.66,0.1535,1.0000,0.2661,0.5312
1,0.2,57,224,11,176,60.04,0.2028,0.8382,0.3266,0.6391
2,0.3,52,95,16,305,31.41,0.3537,0.7647,0.4837,0.7636
3,0.4,40,17,28,383,12.18,0.7018,0.5882,0.6400,0.7729
4,0.5,34,4,34,396,8.12,0.8947,0.5000,0.6415,0.7450
5,0.6,20,2,48,398,4.70,0.9091,0.2941,0.4444,0.6446
6,0.7,12,1,56,399,2.78,0.9231,0.1765,0.2963,0.5870
7,0.8,5,0,63,400,1.07,1.0000,0.0735,0.1370,0.5368
8,0.9,0,0,68,400,0.00,0.0000,0.0000,0.0000,0.5000


In [33]:
# CELL 33: FINAL NELLORE HISTORICAL TEST EVALUATION

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Lock the threshold selected using Nellore validation data.
nellore_selected_threshold = 0.40

# Prepare the complete Nellore historical training and testing features.
X_nellore_train_final = create_time_features(nellore_train)
X_nellore_test_final = create_time_features(nellore_test)

y_nellore_train_final = (
    nellore_train["Next water quality status"]
    .map(target_mapping)
)

y_nellore_test_final = (
    nellore_test["Next water quality status"]
    .map(target_mapping)
)

# Train the final Nellore-specific model.
nellore_final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

nellore_final_model.fit(
    X_nellore_train_final,
    y_nellore_train_final
)

# Predict probabilities on the untouched historical test period.
nellore_test_probabilities = (
    nellore_final_model.predict_proba(X_nellore_test_final)[:, 1]
)

# Apply the threshold selected during internal validation.
nellore_test_predictions = (
    nellore_test_probabilities >= nellore_selected_threshold
).astype(int)

# Calculate test performance.
tn, fp, fn, tp = confusion_matrix(
    y_nellore_test_final,
    nellore_test_predictions,
    labels=[0, 1]
).ravel()

nellore_final_test_summary = pd.DataFrame({
    "Metric": [
        "Historical test observations",
        "Unique test ponds",
        "Actual out-of-range cases",
        "True positives",
        "False positives",
        "False negatives",
        "True negatives",
        "Alert rate (%)",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced accuracy",
        "Average precision",
        "ROC-AUC"
    ],
    "Value": [
        len(nellore_test),
        nellore_test["pond_id"].nunique(),
        int(y_nellore_test_final.sum()),
        tp,
        fp,
        fn,
        tn,
        round(nellore_test_predictions.mean() * 100, 2),
        accuracy_score(y_nellore_test_final, nellore_test_predictions),
        precision_score(y_nellore_test_final, nellore_test_predictions, zero_division=0),
        recall_score(y_nellore_test_final, nellore_test_predictions, zero_division=0),
        f1_score(y_nellore_test_final, nellore_test_predictions, zero_division=0),
        balanced_accuracy_score(y_nellore_test_final, nellore_test_predictions),
        average_precision_score(y_nellore_test_final, nellore_test_probabilities),
        roc_auc_score(y_nellore_test_final, nellore_test_probabilities)
    ]
})

nellore_final_test_summary["Value"] = (
    nellore_final_test_summary["Value"].round(4)
)

display(nellore_final_test_summary)

,Metric,Value
0,Historical test observations,573.0000
1,Unique test ponds,49.0000
2,Actual out-of-range cases,44.0000
3,True positives,20.0000
4,False positives,13.0000
5,False negatives,24.0000
6,True negatives,516.0000
7,Alert rate (%),5.7600
8,Accuracy,0.9354
9,Precision,0.6061


In [34]:
# CELL 34: POND-LEVEL BOOTSTRAP CONFIDENCE INTERVALS

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

# Preserve the original test observations and predictions.
bootstrap_data = nellore_test.reset_index(drop=True)

actual_values = np.asarray(y_nellore_test_final)
predicted_values = np.asarray(nellore_test_predictions)

pond_ids = bootstrap_data["pond_id"].to_numpy()
unique_pond_ids = pd.unique(pond_ids)

# Store the row positions belonging to each pond.
pond_positions = {
    pond: np.flatnonzero(pond_ids == pond)
    for pond in unique_pond_ids
}

rng = np.random.default_rng(42)

bootstrap_results = []

# Resample whole ponds, preserving all observations within each selected pond.
n_bootstrap = 2000

for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_pond_ids,
        size=len(unique_pond_ids),
        replace=True
    )

    sampled_positions = np.concatenate([
        pond_positions[pond]
        for pond in sampled_ponds
    ])

    sampled_actual = actual_values[sampled_positions]
    sampled_predicted = predicted_values[sampled_positions]

    # Balanced accuracy requires both actual classes.
    if len(np.unique(sampled_actual)) < 2:
        continue

    bootstrap_results.append({
        "Precision": precision_score(
            sampled_actual,
            sampled_predicted,
            zero_division=0
        ),
        "Recall": recall_score(
            sampled_actual,
            sampled_predicted,
            zero_division=0
        ),
        "F1-score": f1_score(
            sampled_actual,
            sampled_predicted,
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            sampled_actual,
            sampled_predicted
        )
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

# Calculate percentile-based 95% confidence intervals.
point_estimates = {
    "Precision": precision_score(
        actual_values, predicted_values, zero_division=0
    ),
    "Recall": recall_score(
        actual_values, predicted_values, zero_division=0
    ),
    "F1-score": f1_score(
        actual_values, predicted_values, zero_division=0
    ),
    "Balanced accuracy": balanced_accuracy_score(
        actual_values, predicted_values
    )
}

bootstrap_summary = pd.DataFrame({
    "Metric": list(point_estimates.keys()),
    "Observed estimate": list(point_estimates.values()),
    "95% CI lower": [
        bootstrap_df[column].quantile(0.025)
        for column in point_estimates
    ],
    "95% CI upper": [
        bootstrap_df[column].quantile(0.975)
        for column in point_estimates
    ]
})

bootstrap_summary[
    ["Observed estimate", "95% CI lower", "95% CI upper"]
] = bootstrap_summary[
    ["Observed estimate", "95% CI lower", "95% CI upper"]
].round(4)

display(bootstrap_summary)

,Metric,Observed estimate,95% CI lower,95% CI upper
0,Precision,0.6061,0.4332,0.7391
1,Recall,0.4545,0.2766,0.6327
2,F1-score,0.5195,0.3438,0.6667
3,Balanced accuracy,0.7150,0.6267,0.8028


In [35]:
# CELL 35: ELURU POND-LEVEL BOOTSTRAP CONFIDENCE INTERVALS

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

# Align actual outcomes, probabilities and pond IDs.
eluru_bootstrap_data = pd.DataFrame({
    "pond_id": test_shared["pond_id"].to_numpy(),
    "actual": np.asarray(y_test_final),
    "probability": np.asarray(time_aware_probabilities)
})

# Preserve the validation-selected threshold.
eluru_selected_threshold = 0.20

eluru_bootstrap_data["predicted"] = (
    eluru_bootstrap_data["probability"]
    >= eluru_selected_threshold
).astype(int)

unique_eluru_ponds = (
    eluru_bootstrap_data["pond_id"].unique()
)

# Identify all observations belonging to each pond.
pond_positions = {
    pond: np.flatnonzero(
        eluru_bootstrap_data["pond_id"].to_numpy() == pond
    )
    for pond in unique_eluru_ponds
}

rng = np.random.default_rng(42)

bootstrap_results = []
n_bootstrap = 2000

for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_eluru_ponds,
        size=len(unique_eluru_ponds),
        replace=True
    )

    sampled_positions = np.concatenate([
        pond_positions[pond]
        for pond in sampled_ponds
    ])

    sampled_actual = (
        eluru_bootstrap_data["actual"].to_numpy()[sampled_positions]
    )

    sampled_predictions = (
        eluru_bootstrap_data["predicted"].to_numpy()[sampled_positions]
    )

    # Both actual classes must be present.
    if len(np.unique(sampled_actual)) < 2:
        continue

    bootstrap_results.append({
        "Precision": precision_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            sampled_actual,
            sampled_predictions
        )
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

actual_values = eluru_bootstrap_data["actual"].to_numpy()
predicted_values = eluru_bootstrap_data["predicted"].to_numpy()

point_estimates = {
    "Precision": precision_score(
        actual_values, predicted_values, zero_division=0
    ),
    "Recall": recall_score(
        actual_values, predicted_values, zero_division=0
    ),
    "F1-score": f1_score(
        actual_values, predicted_values, zero_division=0
    ),
    "Balanced accuracy": balanced_accuracy_score(
        actual_values, predicted_values
    )
}

eluru_bootstrap_summary = pd.DataFrame({
    "Metric": list(point_estimates.keys()),
    "Observed estimate": list(point_estimates.values()),
    "95% CI lower": [
        bootstrap_df[column].quantile(0.025)
        for column in point_estimates
    ],
    "95% CI upper": [
        bootstrap_df[column].quantile(0.975)
        for column in point_estimates
    ],
    "Valid bootstrap replicates": len(bootstrap_df)
})

eluru_bootstrap_summary[
    ["Observed estimate", "95% CI lower", "95% CI upper"]
] = eluru_bootstrap_summary[
    ["Observed estimate", "95% CI lower", "95% CI upper"]
].round(4)

display(eluru_bootstrap_summary)

,Metric,Observed estimate,95% CI lower,95% CI upper,Valid bootstrap replicates
0,Precision,0.2037,0.1582,0.2512,2000
1,Recall,0.5994,0.5331,0.6552,2000
2,F1-score,0.3041,0.2463,0.3594,2000
3,Balanced accuracy,0.6093,0.5767,0.6379,2000


In [36]:
# CELL 36: CONSOLIDATED REGIONAL MODEL PERFORMANCE

regional_model_results = pd.DataFrame([
    {
        "Region": "Eluru",
        "Evaluation": "Primary temporal test",
        "Test observations": 2232,
        "Test ponds": 118,
        "Out-of-range cases": 312,
        "Selected threshold": 0.20,
        "Precision": 0.2037,
        "Precision CI lower": 0.1582,
        "Precision CI upper": 0.2512,
        "Recall": 0.5994,
        "Recall CI lower": 0.5331,
        "Recall CI upper": 0.6552,
        "F1-score": 0.3041,
        "F1 CI lower": 0.2463,
        "F1 CI upper": 0.3594,
        "Balanced accuracy": 0.6093,
        "Balanced accuracy CI lower": 0.5767,
        "Balanced accuracy CI upper": 0.6379
    },
    {
        "Region": "Nellore",
        "Evaluation": "Historical temporal test",
        "Test observations": 573,
        "Test ponds": 49,
        "Out-of-range cases": 44,
        "Selected threshold": 0.40,
        "Precision": 0.6061,
        "Precision CI lower": 0.4332,
        "Precision CI upper": 0.7391,
        "Recall": 0.4545,
        "Recall CI lower": 0.2766,
        "Recall CI upper": 0.6327,
        "F1-score": 0.5195,
        "F1 CI lower": 0.3438,
        "F1 CI upper": 0.6667,
        "Balanced accuracy": 0.7150,
        "Balanced accuracy CI lower": 0.6267,
        "Balanced accuracy CI upper": 0.8028
    }
])

display(regional_model_results)

,Region,Evaluation,Test observations,Test ponds,Out-of-range cases,Selected threshold,Precision,Precision CI lower,Precision CI upper,Recall,Recall CI lower,Recall CI upper,F1-score,F1 CI lower,F1 CI upper,Balanced accuracy,Balanced accuracy CI lower,Balanced accuracy CI upper
0,Eluru,Primary temporal test,2232,118,312,0.2,0.2037,0.1582,0.2512,0.5994,0.5331,0.6552,0.3041,0.2463,0.3594,0.6093,0.5767,0.6379
1,Nellore,Historical temporal test,573,49,44,0.4,0.6061,0.4332,0.7391,0.4545,0.2766,0.6327,0.5195,0.3438,0.6667,0.7150,0.6267,0.8028


# AQUA — Predictive Modelling Conclusions and Limitations

## 1. Research objective

The predictive modelling experiment investigated whether historical aquaculture water-quality measurements could help anticipate the recorded water-quality status at the next observation from the same pond.

The prediction target was the next observation's `Is WQ in range?` value.

The initial predictors were pH, dissolved oxygen and temperature. A second model incorporated measurement time and an interaction between dissolved oxygen and morning measurement.

## 2. Prediction dataset

After chronological ordering, ambiguous timestamp transitions and incomplete predictor records were excluded.

The resulting modelling dataset contained 12,082 prediction pairs from 316 ponds.

The prediction target was imbalanced:
- Within-range outcomes: 81.25%.
- Out-of-range outcomes: 18.75%.

The median interval between consecutive observations was approximately 1.56 days, although observation intervals varied considerably. Therefore, the prediction horizon represents the next recorded observation rather than a fixed future time.

## 3. Temporal validation strategy

The primary evaluation used chronological training and testing periods.

The primary Eluru test set contained 2,232 observations from 118 previously monitored ponds.

A separate historical temporal experiment was conducted for Nellore, using 2,484 training pairs and 573 later testing pairs.

A supplementary evaluation on previously unseen Eluru ponds contained only 66 prediction pairs from 20 ponds.

## 4. Model comparison

Logistic Regression was used as the initial modelling method.

Measurement time was introduced as an additional predictor, together with an interaction between morning measurement and dissolved oxygen.

The time-aware model showed improved performance compared with the original model in the primary Eluru test evaluation and provided useful results in the separate Nellore historical experiment.

The final regional results were:

| Metric | Eluru | Nellore |
|---|---:|---:|
| Precision | 20.37% | 60.61% |
| Recall | 59.94% | 45.45% |
| F1-score | 0.3041 | 0.5195 |
| Balanced accuracy | 60.93% | 71.50% |

The classification thresholds were selected using the respective internal validation periods, rather than the final test results.

## 5. Uncertainty estimation

Pond-level bootstrap resampling with 2,000 replicates was used to estimate 95% confidence intervals.

The resulting intervals were:

| Metric | Eluru | Nellore |
|---|---:|---:|
| Precision | 15.82%–25.12% | 43.32%–73.91% |
| Recall | 53.31%–65.52% | 27.66%–63.27% |
| F1-score | 24.63%–35.94% | 34.38%–66.67% |
| Balanced accuracy | 57.67%–63.79% | 62.67%–80.28% |

These intervals reflect variability from resampling the observed ponds. They do not capture all sources of uncertainty, including model retraining, threshold selection and future distribution changes.

## 6. Important limitations

1. The outcome classes are imbalanced, making accuracy alone insufficient for evaluating model performance.

2. Observation intervals are irregular. The model predicts the next recorded observation, not a fixed number of hours or days into the future.

3. Dissolved oxygen has a time-dependent acceptable range. Measurement time improved the initial model, but further investigation is required.

4. Water-quality measurement protocols and classification criteria changed over the study period. The consistency of the target definition across periods requires further investigation.

5. The primary Eluru model evaluation produced substantial false alarms, limiting its immediate suitability as a farmer-facing warning system.

6. The Nellore experiment uses a different historical period and outcome distribution. Its results should not be interpreted as a direct comparison with Eluru.

7. The unseen-pond evaluation contained only four out-of-range cases. Its results are supplementary and cannot establish reliable generalisation to new ponds.

## 7. Overall conclusion

The initial experiments demonstrate that historical water-quality measurements contain predictive information about the next recorded water-quality classification.

However, predictive performance varies across regions and observation periods. False alarms, missed out-of-range cases, target-definition changes and limited unseen-pond evidence remain important challenges.

AQUA should therefore be considered an exploratory predictive research project rather than a validated operational early-warning system.

Further work should investigate target consistency, regional and temporal distribution changes, model calibration and generalisation before practical deployment is considered.

In [37]:
# CELL 37: TARGET DISTRIBUTION AROUND AUGUST 2025 TRANSITION

# Use all unambiguous prediction pairs.
# The target date is the date of the NEXT observation.
target_period_audit = prediction_pairs[
    [
        "region",
        "Next observation timestamp",
        "Next water quality status"
    ]
].copy()

target_period_audit["Target date"] = pd.to_datetime(
    target_period_audit["Next observation timestamp"],
    errors="coerce"
)

# Divide observations into three historical periods.
target_period_audit["Target period"] = np.select(
    [
        target_period_audit["Target date"] < pd.Timestamp("2025-08-01"),

        (
            (target_period_audit["Target date"] >= pd.Timestamp("2025-08-01"))
            & (target_period_audit["Target date"] < pd.Timestamp("2025-09-01"))
        )
    ],
    [
        "Before August 2025",
        "August 2025 transition month"
    ],
    default="September 2025 onward"
)

# Establish chronological period ordering.
period_order = [
    "Before August 2025",
    "August 2025 transition month",
    "September 2025 onward"
]

target_period_audit["Target period"] = pd.Categorical(
    target_period_audit["Target period"],
    categories=period_order,
    ordered=True
)

# Count future outcomes separately by region and period.
period_counts = (
    target_period_audit
    .groupby(
        ["region", "Target period", "Next water quality status"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Yes", "No"], fill_value=0)
)

period_counts["Prediction pairs"] = (
    period_counts["Yes"] + period_counts["No"]
)

period_counts["Future status No (%)"] = (
    period_counts["No"] / period_counts["Prediction pairs"] * 100
).round(2)

period_summary = period_counts.reset_index()

period_summary = period_summary[
    [
        "region",
        "Target period",
        "Prediction pairs",
        "Yes",
        "No",
        "Future status No (%)"
    ]
]

display(period_summary)

Next water quality status,region,Target period,Prediction pairs,Yes,No,Future status No (%)
0,Eluru,Before August 2025,5765,4567,1198,20.78
1,Eluru,August 2025 transition month,178,173,5,2.81
2,Eluru,September 2025 onward,3051,2631,420,13.77
3,Nellore,Before August 2025,3116,2467,649,20.83


In [38]:
# CELL 38: MONTHLY FUTURE WATER-QUALITY STATUS DISTRIBUTION

monthly_audit = prediction_pairs.copy()

monthly_audit["Target month"] = (
    monthly_audit["Next observation timestamp"]
    .dt.to_period("M")
)

# Examine monthly distributions for Eluru during 2025.
eluru_monthly = monthly_audit.loc[
    (monthly_audit["region"] == "Eluru")
    & (
        monthly_audit["Next observation timestamp"].dt.year == 2025
    )
].copy()

monthly_summary = (
    eluru_monthly
    .groupby(
        ["Target month", "Next water quality status"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Yes", "No"], fill_value=0)
)

monthly_summary["Prediction pairs"] = (
    monthly_summary["Yes"] + monthly_summary["No"]
)

monthly_summary["Out-of-range rate (%)"] = (
    monthly_summary["No"]
    / monthly_summary["Prediction pairs"]
    * 100
).round(2)

monthly_summary = monthly_summary.reset_index()

display(
    monthly_summary[
        [
            "Target month",
            "Prediction pairs",
            "Yes",
            "No",
            "Out-of-range rate (%)"
        ]
    ]
)

Next water quality status,Target month,Prediction pairs,Yes,No,Out-of-range rate (%)
0,2025-01,218,213,5,2.29
1,2025-02,246,218,28,11.38
2,2025-03,249,218,31,12.45
3,2025-04,304,298,6,1.97
4,2025-05,205,198,7,3.41
5,2025-06,214,200,14,6.54
6,2025-07,222,217,5,2.25
7,2025-08,178,173,5,2.81
8,2025-09,314,271,43,13.69
9,2025-10,185,166,19,10.27


In [39]:
# CELL 39: MULTIYEAR MONTHLY TARGET DISTRIBUTION — ELURU

eluru_monthly_all = prediction_pairs.loc[
    prediction_pairs["region"] == "Eluru"
].copy()

eluru_monthly_all["Target month"] = (
    eluru_monthly_all["Next observation timestamp"]
    .dt.to_period("M")
)

# Retain January 2024 onward.
eluru_monthly_all = eluru_monthly_all.loc[
    eluru_monthly_all["Target month"]
    >= pd.Period("2024-01", freq="M")
]

monthly_multiyear = (
    eluru_monthly_all
    .groupby("Target month", observed=True)["Next water quality status"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["Yes", "No"], fill_value=0)
)

monthly_multiyear["Prediction pairs"] = (
    monthly_multiyear["Yes"] + monthly_multiyear["No"]
)

monthly_multiyear["Out-of-range rate (%)"] = (
    monthly_multiyear["No"]
    / monthly_multiyear["Prediction pairs"]
    * 100
).round(2)

monthly_multiyear = monthly_multiyear.reset_index()

display(
    monthly_multiyear[
        [
            "Target month",
            "Prediction pairs",
            "Yes",
            "No",
            "Out-of-range rate (%)"
        ]
    ]
)

Next water quality status,Target month,Prediction pairs,Yes,No,Out-of-range rate (%)
0,2024-01,125,103,22,17.60
1,2024-02,138,108,30,21.74
2,2024-03,108,106,2,1.85
3,2024-04,139,131,8,5.76
4,2024-05,157,150,7,4.46
5,2024-06,70,69,1,1.43
6,2024-07,51,47,4,7.84
7,2024-08,116,67,49,42.24
8,2024-09,164,135,29,17.68
9,2024-10,194,187,7,3.61


In [40]:
# CELL 40: AUDIT REASONS FOR FUTURE OUT-OF-RANGE CLASSIFICATIONS

reason_audit = prediction_pairs.copy()

# Retrieve the reasons recorded at the NEXT observation.
next_reasons = (
    chronological_df.groupby("pond_id")["Parameters out of range"]
    .shift(-1)
)

reason_audit["Next parameters out of range"] = (
    next_reasons.reindex(reason_audit.index)
)

# Classify observations according to the date of the future outcome.
target_dates = pd.to_datetime(
    reason_audit["Next observation timestamp"]
)

reason_audit["Target period"] = np.select(
    [
        target_dates < pd.Timestamp("2025-08-01"),
        (
            (target_dates >= pd.Timestamp("2025-08-01"))
            & (target_dates < pd.Timestamp("2025-09-01"))
        )
    ],
    [
        "Before August 2025",
        "August 2025 transition month"
    ],
    default="September 2025 onward"
)

# Retrospective analysis of actual future out-of-range observations.
out_of_range_reasons = reason_audit.loc[
    reason_audit["Next water quality status"] == "No"
].copy()

out_of_range_reasons["Next parameters out of range"] = (
    out_of_range_reasons["Next parameters out of range"]
    .fillna("Missing reason field")
)

# Count recorded reason combinations by region and historical period.
reason_counts = (
    out_of_range_reasons
    .groupby(["region", "Target period"])["Next parameters out of range"]
    .value_counts(dropna=False)
    .rename("Count")
    .reset_index()
)

# Calculate each reason's share within its region and period.
reason_counts["Percentage within period (%)"] = (
    reason_counts["Count"]
    / reason_counts.groupby(["region", "Target period"])["Count"]
      .transform("sum")
    * 100
).round(2)

# Retain the five most common reason combinations in each group.
reason_counts = (
    reason_counts
    .sort_values(
        ["region", "Target period", "Count"],
        ascending=[True, True, False]
    )
    .groupby(
        ["region", "Target period"],
        group_keys=False
    )
    .head(5)
    .reset_index(drop=True)
)

display(reason_counts)

,region,Target period,Next parameters out of range,Count,Percentage within period (%)
0,Eluru,August 2025 transition month,Dissolved Oxygen,3,60.00
1,Eluru,August 2025 transition month,"Dissolved Oxygen, Ammonia Tan Nh3N",2,40.00
2,Eluru,Before August 2025,Dissolved Oxygen,496,41.40
3,Eluru,Before August 2025,Turbidity,233,19.45
4,Eluru,Before August 2025,Ph,152,12.69
5,Eluru,Before August 2025,"Dissolved Oxygen, Ph",139,11.60
6,Eluru,Before August 2025,Ammonia Tan Nh3N,40,3.34
7,Eluru,September 2025 onward,Dissolved Oxygen,197,46.90
8,Eluru,September 2025 onward,Ph,96,22.86
9,Eluru,September 2025 onward,"Dissolved Oxygen, Ph",65,15.48


In [41]:
# CELL 41: INDIVIDUAL OUT-OF-RANGE PARAMETER ANALYSIS

# Start with actual future out-of-range observations.
parameter_audit = out_of_range_reasons[
    [
        "region",
        "Target period",
        "Next parameters out of range"
    ]
].copy()

# Separate individual parameters from combined reason descriptions.
parameter_audit["Individual parameter"] = (
    parameter_audit["Next parameters out of range"]
    .astype("string")
    .str.split(",")
)

parameter_mentions = parameter_audit.explode(
    "Individual parameter"
)

parameter_mentions["Individual parameter"] = (
    parameter_mentions["Individual parameter"]
    .str.strip()
)

# Count individual parameter mentions.
parameter_summary = (
    parameter_mentions
    .groupby(
        ["region", "Target period", "Individual parameter"],
        observed=True
    )
    .size()
    .reset_index(name="Parameter mentions")
)

# Calculate the number of out-of-range observations in each period.
period_totals = (
    out_of_range_reasons
    .groupby(["region", "Target period"], observed=True)
    .size()
    .rename("Out-of-range observations")
    .reset_index()
)

parameter_summary = parameter_summary.merge(
    period_totals,
    on=["region", "Target period"],
    how="left"
)

parameter_summary["Percentage of out-of-range observations (%)"] = (
    parameter_summary["Parameter mentions"]
    / parameter_summary["Out-of-range observations"]
    * 100
).round(2)

parameter_summary = parameter_summary.sort_values(
    ["region", "Target period", "Parameter mentions"],
    ascending=[True, True, False]
).reset_index(drop=True)

display(parameter_summary)

,region,Target period,Individual parameter,Parameter mentions,Out-of-range observations,Percentage of out-of-range observations (%)
0,Eluru,August 2025 transition month,Dissolved Oxygen,5,5,100.00
1,Eluru,August 2025 transition month,Ammonia Tan Nh3N,2,5,40.00
2,Eluru,Before August 2025,Dissolved Oxygen,737,1198,61.52
3,Eluru,Before August 2025,Ph,358,1198,29.88
4,Eluru,Before August 2025,Turbidity,273,1198,22.79
5,Eluru,Before August 2025,Ammonia Tan Nh3N,143,1198,11.94
6,Eluru,September 2025 onward,Dissolved Oxygen,298,420,70.95
7,Eluru,September 2025 onward,Ph,189,420,45.00
8,Eluru,September 2025 onward,Ammonia Nh3,62,420,14.76
9,Nellore,Before August 2025,Dissolved Oxygen,428,649,65.95


In [42]:
# CELL 42: AUDIT COMPLETENESS OF RECORDED OUT-OF-RANGE REASONS

reason_coverage = out_of_range_reasons.copy()

# Identify records where no reason was recorded.
reason_coverage["Reason missing"] = (
    reason_coverage["Next parameters out of range"]
    .eq("Missing reason field")
)

# Summarise reason-field completeness by region and period.
reason_coverage_summary = (
    reason_coverage
    .groupby(
        ["region", "Target period"],
        observed=True
    )
    .agg(
        Out_of_range_observations=(
            "Next parameters out of range", "size"
        ),
        Missing_reason_records=(
            "Reason missing", "sum"
        ),
        Distinct_recorded_reason_combinations=(
            "Next parameters out of range",
            lambda values: values[
                values != "Missing reason field"
            ].nunique()
        )
    )
    .reset_index()
)

reason_coverage_summary["Missing reason (%)"] = (
    reason_coverage_summary["Missing_reason_records"]
    / reason_coverage_summary["Out_of_range_observations"]
    * 100
).round(2)

display(reason_coverage_summary)

,region,Target period,Out_of_range_observations,Missing_reason_records,Distinct_recorded_reason_combinations,Missing reason (%)
0,Eluru,August 2025 transition month,5,0,2,0.0
1,Eluru,Before August 2025,1198,0,13,0.0
2,Eluru,September 2025 onward,420,0,7,0.0
3,Nellore,Before August 2025,649,0,9,0.0


## Target Distribution and Classification Consistency — Findings

The recorded future water-quality status was examined across regions and historical periods surrounding the August 2025 ammonia measurement transition.

### 1. Changes in target distribution

Eluru showed considerable monthly variation in recorded out-of-range classifications.

The out-of-range rate was:
- 20.78% before August 2025.
- 2.81% during August 2025.
- 13.77% from September 2025 onward.

Historical monthly fluctuations were also observed before the measurement transition. Therefore, the August 2025 change cannot independently explain the observed variation.

### 2. Recorded reasons for out-of-range classifications

Dissolved oxygen was the most frequently recorded reason across the examined region-period groups.

pH was also frequently identified. Ammonia terminology differed across the measurement transition:
- Earlier records predominantly referenced Ammonia TAN (NH3-N).
- Later Eluru records referenced Ammonia NH3.

These measurements must remain distinct because they represent different ammonia measurement definitions.

### 3. Reason-field completeness

All 2,272 audited out-of-range observations had a recorded value in the `Parameters out of range` field.

The absence of missing reason entries does not establish that classification criteria or reporting practices remained consistent.

### 4. Modelling implications

The prediction target represents the recorded `Is WQ in range?` classification rather than an independently reconstructed, scientifically uniform water-quality standard.

The field `Parameters out of range` remains excluded from predictive modelling because it directly describes the classification outcome.

Future modelling should account for possible temporal changes in classification criteria and ammonia measurement protocols.

### 5. Conclusion

The dataset provides useful historical information for predictive modelling, but temporal consistency of the recorded target remains an important limitation.

Model performance should therefore be interpreted as prediction of the recorded future classification within the evaluated historical periods, not as proof of predicting a universally consistent biological water-quality condition.

In [43]:
# CELL 46: RANDOM FOREST VS TIME-AWARE LOGISTIC REGRESSION

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Train Random Forest using the same internal training data.
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_inner_time,
    y_inner_train
)

# Generate validation probabilities.
rf_validation_probabilities = (
    random_forest_model.predict_proba(X_validation_time)[:, 1]
)

# Compare with the existing time-aware Logistic Regression.
model_probabilities = {
    "Time-aware Logistic Regression": time_aware_validation_probabilities,
    "Random Forest": rf_validation_probabilities
}

# Evaluate identical thresholds for both models.
rf_comparison_results = []

for model_name, probabilities in model_probabilities.items():

    for threshold in np.arange(0.10, 1.00, 0.10):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        rf_comparison_results.append({
            "Model": model_name,
            "Threshold": round(threshold, 2),
            "Alert rate (%)": round(predictions.mean() * 100, 2),
            "Precision": precision_score(
                y_validation, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_validation, predictions, zero_division=0
            ),
            "F1-score": f1_score(
                y_validation, predictions, zero_division=0
            ),
            "Balanced accuracy": balanced_accuracy_score(
                y_validation, predictions
            ),
            "Average precision": average_precision_score(
                y_validation, probabilities
            ),
            "ROC-AUC": roc_auc_score(
                y_validation, probabilities
            )
        })

rf_comparison_summary = pd.DataFrame(rf_comparison_results)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

rf_comparison_summary[metric_columns] = (
    rf_comparison_summary[metric_columns].round(4)
)

display(rf_comparison_summary)

,Model,Threshold,Alert rate (%),Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,Time-aware Logistic Regression,0.1,97.81,0.0799,1.0000,0.1480,0.5119,0.2282,0.7037
1,Time-aware Logistic Regression,0.2,60.03,0.1102,0.8462,0.1950,0.6333,0.2282,0.7037
2,Time-aware Logistic Regression,0.3,10.39,0.2053,0.2727,0.2342,0.5916,0.2282,0.7037
3,Time-aware Logistic Regression,0.4,4.10,0.3733,0.1958,0.2569,0.5840,0.2282,0.7037
4,Time-aware Logistic Regression,0.5,2.57,0.3404,0.1119,0.1684,0.5468,0.2282,0.7037
5,Time-aware Logistic Regression,0.6,1.59,0.3793,0.0769,0.1279,0.5331,0.2282,0.7037
6,Time-aware Logistic Regression,0.7,0.55,0.7000,0.0490,0.0915,0.5236,0.2282,0.7037
7,Time-aware Logistic Regression,0.8,0.05,1.0000,0.0070,0.0139,0.5035,0.2282,0.7037
8,Time-aware Logistic Regression,0.9,0.00,0.0000,0.0000,0.0000,0.5000,0.2282,0.7037
9,Random Forest,0.1,89.28,0.0851,0.9720,0.1565,0.5430,0.2559,0.6936


In [44]:
# CELL 47: FINAL RANDOM FOREST VS LOGISTIC REGRESSION

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Train Random Forest on the complete Eluru training period.
random_forest_final_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

random_forest_final_model.fit(
    X_train_time_final,
    y_train_final
)

# Predict probabilities on the untouched primary test set.
rf_test_probabilities = (
    random_forest_final_model
    .predict_proba(X_test_time_final)[:, 1]
)

# Compare validation-selected operating points.
final_rf_comparisons = [
    ("Time-aware Logistic Regression", 0.40, time_aware_probabilities),
    ("Random Forest", 0.60, rf_test_probabilities),
    ("Random Forest — higher recall", 0.30, rf_test_probabilities)
]

final_rf_results = []

for model_name, threshold, probabilities in final_rf_comparisons:

    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_final,
        predictions,
        labels=[0, 1]
    ).ravel()

    final_rf_results.append({
        "Model": model_name,
        "Threshold": threshold,
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Accuracy": accuracy_score(y_test_final, predictions),
        "Precision": precision_score(
            y_test_final, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test_final, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_test_final, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_final, predictions
        ),
        "Average precision": average_precision_score(
            y_test_final, probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_test_final, probabilities
        )
    })

final_rf_comparison = pd.DataFrame(final_rf_results)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

final_rf_comparison[metric_columns] = (
    final_rf_comparison[metric_columns].round(4)
)

display(final_rf_comparison)

,Model,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Accuracy,Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,Time-aware Logistic Regression,0.4,64,32,248,1888,4.30,0.8746,0.6667,0.2051,0.3137,0.5942,0.3722,0.6701
1,Random Forest,0.6,108,89,204,1831,8.83,0.8687,0.5482,0.3462,0.4244,0.6499,0.4504,0.7420
2,Random Forest — higher recall,0.3,211,665,101,1255,39.25,0.6568,0.2409,0.6763,0.3552,0.6650,0.4504,0.7420


In [45]:
# CELL 48: RANDOM FOREST PERMUTATION FEATURE IMPORTANCE

from sklearn.inspection import permutation_importance

# Evaluate feature importance using the validation dataset.
permutation_result = permutation_importance(
    random_forest_model,
    X_validation_time,
    y_validation,
    scoring="average_precision",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

feature_importance_summary = pd.DataFrame({
    "Feature": X_validation_time.columns,
    "Mean importance": permutation_result.importances_mean,
    "Importance standard deviation": permutation_result.importances_std
})

feature_importance_summary = (
    feature_importance_summary
    .sort_values("Mean importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance_summary[
    ["Mean importance", "Importance standard deviation"]
] = feature_importance_summary[
    ["Mean importance", "Importance standard deviation"]
].round(5)

display(feature_importance_summary)

,Feature,Mean importance,Importance standard deviation
0,pH,0.13800,0.00744
1,DO (mg/L),0.06326,0.01623
2,morning_measurement,0.00524,0.00889
3,Temp (in °C),0.00144,0.01135
4,do_x_morning,-0.00701,0.01418


In [46]:
# CELL 49: RANDOM FOREST FEATURE-ABLATION EXPERIMENT

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    balanced_accuracy_score
)

# Define the three feature configurations.
feature_configurations = {
    "pH + DO": [
        "pH",
        "DO (mg/L)"
    ],
    "pH + DO + Temperature": [
        "pH",
        "DO (mg/L)",
        "Temp (in °C)"
    ],
    "All five features": list(X_inner_time.columns)
}

ablation_results = []

# Train and evaluate each configuration on identical data.
for configuration_name, selected_features in feature_configurations.items():

    X_train_variant = X_inner_time[selected_features]
    X_validation_variant = X_validation_time[selected_features]

    model = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_variant, y_inner_train)

    # Validation probabilities.
    probabilities = model.predict_proba(
        X_validation_variant
    )[:, 1]

    # Use 0.60 as a common diagnostic threshold.
    # This is for comparison, not final threshold selection.
    predictions = (probabilities >= 0.60).astype(int)

    ablation_results.append({
        "Feature configuration": configuration_name,
        "Number of features": len(selected_features),
        "Average precision": average_precision_score(
            y_validation, probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_validation, probabilities
        ),
        "F1-score at 0.60": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Balanced accuracy at 0.60": balanced_accuracy_score(
            y_validation, predictions
        )
    })

ablation_summary = pd.DataFrame(ablation_results)

metric_columns = [
    "Average precision",
    "ROC-AUC",
    "F1-score at 0.60",
    "Balanced accuracy at 0.60"
]

ablation_summary[metric_columns] = (
    ablation_summary[metric_columns].round(4)
)

display(ablation_summary)

,Feature configuration,Number of features,Average precision,ROC-AUC,F1-score at 0.60,Balanced accuracy at 0.60
0,pH + DO,2,0.2378,0.6645,0.2606,0.6031
1,pH + DO + Temperature,3,0.2607,0.6922,0.2846,0.6074
2,All five features,5,0.2559,0.6936,0.2900,0.6106


In [47]:
# CELL 50: THREE-FEATURE RANDOM FOREST THRESHOLD EVALUATION

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Use only the three original water-quality predictors.
X_inner_three = inner_training[predictor_columns].copy()
X_validation_three = validation_data[predictor_columns].copy()

# Train the selected three-feature Random Forest.
rf_three_feature_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_three_feature_model.fit(
    X_inner_three,
    y_inner_train
)

# Generate validation probabilities.
rf_three_validation_probabilities = (
    rf_three_feature_model.predict_proba(X_validation_three)[:, 1]
)

# Evaluate candidate thresholds.
three_feature_results = []

for threshold in np.arange(0.10, 1.00, 0.10):

    predictions = (
        rf_three_validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions,
        labels=[0, 1]
    ).ravel()

    three_feature_results.append({
        "Threshold": round(threshold, 2),
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_validation, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_validation, predictions
        ),
        "Average precision": average_precision_score(
            y_validation, rf_three_validation_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_validation, rf_three_validation_probabilities
        )
    })

three_feature_summary = pd.DataFrame(three_feature_results)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

three_feature_summary[metric_columns] = (
    three_feature_summary[metric_columns].round(4)
)

display(three_feature_summary)

,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,0.1,139,1542,4,144,91.91,0.0827,0.9720,0.1524,0.5287,0.2607,0.6922
1,0.2,126,1158,17,528,70.20,0.0981,0.8811,0.1766,0.5971,0.2607,0.6922
2,0.3,95,685,48,1001,42.65,0.1218,0.6643,0.2059,0.6290,0.2607,0.6922
3,0.4,66,333,77,1353,21.82,0.1654,0.4615,0.2435,0.6320,0.2607,0.6922
4,0.5,46,155,97,1531,10.99,0.2289,0.3217,0.2674,0.6149,0.2607,0.6922
5,0.6,38,86,105,1600,6.78,0.3065,0.2657,0.2846,0.6074,0.2607,0.6922
6,0.7,33,59,110,1627,5.03,0.3587,0.2308,0.2809,0.5979,0.2607,0.6922
7,0.8,25,32,118,1654,3.12,0.4386,0.1748,0.2500,0.5779,0.2607,0.6922
8,0.9,17,8,126,1678,1.37,0.6800,0.1189,0.2024,0.5571,0.2607,0.6922


In [48]:
# CELL 51: FINAL TEST OF THREE-FEATURE RANDOM FOREST

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Train using the complete Eluru training period.
rf_three_feature_final_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_three_feature_final_model.fit(
    temporal_training[predictor_columns],
    y_train_final
)

# Generate probabilities on the untouched primary test set.
rf_three_feature_test_probabilities = (
    rf_three_feature_final_model.predict_proba(
        test_shared[predictor_columns]
    )[:, 1]
)

# Compare the two operating points selected using validation data.
final_three_feature_results = []

for threshold in [0.40, 0.60]:

    predictions = (
        rf_three_feature_test_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_final,
        predictions,
        labels=[0, 1]
    ).ravel()

    final_three_feature_results.append({
        "Model": "Three-feature Random Forest",
        "Threshold": threshold,
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Accuracy": accuracy_score(y_test_final, predictions),
        "Precision": precision_score(
            y_test_final, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test_final, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_test_final, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test_final, predictions
        ),
        "Average precision": average_precision_score(
            y_test_final, rf_three_feature_test_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_test_final, rf_three_feature_test_probabilities
        )
    })

final_three_feature_summary = pd.DataFrame(
    final_three_feature_results
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

final_three_feature_summary[metric_columns] = (
    final_three_feature_summary[metric_columns].round(4)
)

display(final_three_feature_summary)

,Model,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Accuracy,Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,Three-feature Random Forest,0.4,163,351,149,1569,23.03,0.7760,0.3171,0.5224,0.3947,0.6698,0.456,0.7456
1,Three-feature Random Forest,0.6,109,82,203,1838,8.56,0.8723,0.5707,0.3494,0.4334,0.6533,0.456,0.7456


In [49]:
# CELL 52: THREE-FEATURE RANDOM FOREST ON UNSEEN PONDS

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Prepare predictors and actual outcomes.
X_unseen_three = test_unseen[predictor_columns].copy()

y_unseen_three = (
    test_unseen["Next water quality status"]
    .map(target_mapping)
)

# Generate probabilities using the final three-feature model.
unseen_rf_probabilities = (
    rf_three_feature_final_model
    .predict_proba(X_unseen_three)[:, 1]
)

# Apply the threshold selected using validation data.
unseen_rf_threshold = 0.60

unseen_rf_predictions = (
    unseen_rf_probabilities >= unseen_rf_threshold
).astype(int)

# Calculate confusion matrix counts.
tn, fp, fn, tp = confusion_matrix(
    y_unseen_three,
    unseen_rf_predictions,
    labels=[0, 1]
).ravel()

# Calculate supplementary evaluation metrics.
unseen_rf_summary = pd.DataFrame({
    "Metric": [
        "Unseen-pond observations",
        "Unique unseen ponds",
        "Actual out-of-range cases",
        "True positives",
        "False positives",
        "False negatives",
        "True negatives",
        "Alert rate (%)",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced accuracy",
        "Average precision",
        "ROC-AUC"
    ],
    "Value": [
        len(test_unseen),
        test_unseen["pond_id"].nunique(),
        int(y_unseen_three.sum()),
        tp,
        fp,
        fn,
        tn,
        round(unseen_rf_predictions.mean() * 100, 2),
        accuracy_score(y_unseen_three, unseen_rf_predictions),
        precision_score(y_unseen_three, unseen_rf_predictions, zero_division=0),
        recall_score(y_unseen_three, unseen_rf_predictions, zero_division=0),
        f1_score(y_unseen_three, unseen_rf_predictions, zero_division=0),
        balanced_accuracy_score(y_unseen_three, unseen_rf_predictions),
        average_precision_score(y_unseen_three, unseen_rf_probabilities),
        roc_auc_score(y_unseen_three, unseen_rf_probabilities)
    ]
})

unseen_rf_summary["Value"] = unseen_rf_summary["Value"].round(4)

display(unseen_rf_summary)

,Metric,Value
0,Unseen-pond observations,66.0000
1,Unique unseen ponds,20.0000
2,Actual out-of-range cases,4.0000
3,True positives,2.0000
4,False positives,6.0000
5,False negatives,2.0000
6,True negatives,56.0000
7,Alert rate (%),12.1200
8,Accuracy,0.8788
9,Precision,0.2500


In [50]:
# CELL 53: THREE-FEATURE RANDOM FOREST UNSEEN-POND AUDIT

unseen_rf_audit = test_unseen[["pond_id"]].copy()

unseen_rf_audit["Actual out-of-range"] = np.asarray(
    y_unseen_three
)

unseen_rf_audit["Predicted out-of-range"] = (
    unseen_rf_predictions
)

unseen_rf_audit["True positive"] = (
    (unseen_rf_audit["Actual out-of-range"] == 1)
    & (unseen_rf_audit["Predicted out-of-range"] == 1)
).astype(int)

unseen_rf_audit["False positive"] = (
    (unseen_rf_audit["Actual out-of-range"] == 0)
    & (unseen_rf_audit["Predicted out-of-range"] == 1)
).astype(int)

unseen_rf_audit["False negative"] = (
    (unseen_rf_audit["Actual out-of-range"] == 1)
    & (unseen_rf_audit["Predicted out-of-range"] == 0)
).astype(int)

unseen_rf_pond_summary = (
    unseen_rf_audit
    .groupby("pond_id")
    .agg(
        Prediction_pairs=("pond_id", "size"),
        Actual_out_of_range=("Actual out-of-range", "sum"),
        Predicted_alerts=("Predicted out-of-range", "sum"),
        True_positives=("True positive", "sum"),
        False_positives=("False positive", "sum"),
        False_negatives=("False negative", "sum")
    )
    .reset_index()
    .sort_values(
        ["Actual_out_of_range", "True_positives"],
        ascending=False
    )
)

display(unseen_rf_pond_summary)

,pond_id,Prediction_pairs,Actual_out_of_range,Predicted_alerts,True_positives,False_positives,False_negatives
8,ara2_438777b9,5,2,3,1,2,1
17,ara2_d8a0ad32,5,2,3,1,2,1
0,ara2_0206e1e9,5,0,0,0,0,0
1,ara2_048a77e9,1,0,0,0,0,0
2,ara2_25e534f2,1,0,1,0,1,0
3,ara2_296bcb81,1,0,0,0,0,0
4,ara2_2bd14b5a,6,0,1,0,1,0
5,ara2_31503b0c,2,0,0,0,0,0
6,ara2_3c745150,6,0,0,0,0,0
7,ara2_41e97843,1,0,0,0,0,0


In [51]:
# CELL 54: THREE-FEATURE RANDOM FOREST BOOTSTRAP CONFIDENCE INTERVALS

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Use the validation-selected primary threshold.
rf_selected_threshold = 0.60

# Preserve the untouched Eluru test observations.
rf_bootstrap_data = pd.DataFrame({
    "pond_id": test_shared["pond_id"].to_numpy(),
    "actual": np.asarray(y_test_final),
    "probability": np.asarray(rf_three_feature_test_probabilities)
})

rf_bootstrap_data["predicted"] = (
    rf_bootstrap_data["probability"]
    >= rf_selected_threshold
).astype(int)

# Identify the unique ponds.
unique_ponds = rf_bootstrap_data["pond_id"].unique()

# Store all observation positions belonging to each pond.
pond_positions = {
    pond: np.flatnonzero(
        rf_bootstrap_data["pond_id"].to_numpy() == pond
    )
    for pond in unique_ponds
}

rng = np.random.default_rng(42)

bootstrap_results = []
n_bootstrap = 2000

# Resample whole ponds, retaining all observations within each selected pond.
for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_ponds,
        size=len(unique_ponds),
        replace=True
    )

    sampled_positions = np.concatenate([
        pond_positions[pond]
        for pond in sampled_ponds
    ])

    sampled_actual = (
        rf_bootstrap_data["actual"].to_numpy()[sampled_positions]
    )

    sampled_predictions = (
        rf_bootstrap_data["predicted"].to_numpy()[sampled_positions]
    )

    sampled_probabilities = (
        rf_bootstrap_data["probability"].to_numpy()[sampled_positions]
    )

    # Both classes must be present for balanced accuracy and ROC-AUC.
    if len(np.unique(sampled_actual)) < 2:
        continue

    bootstrap_results.append({
        "Precision": precision_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            sampled_actual,
            sampled_predictions
        ),
        "Average precision": average_precision_score(
            sampled_actual,
            sampled_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            sampled_actual,
            sampled_probabilities
        )
    })

rf_bootstrap_df = pd.DataFrame(bootstrap_results)

# Calculate observed test-set estimates.
actual_values = rf_bootstrap_data["actual"].to_numpy()
predicted_values = rf_bootstrap_data["predicted"].to_numpy()
probability_values = rf_bootstrap_data["probability"].to_numpy()

observed_metrics = {
    "Precision": precision_score(
        actual_values, predicted_values, zero_division=0
    ),
    "Recall": recall_score(
        actual_values, predicted_values, zero_division=0
    ),
    "F1-score": f1_score(
        actual_values, predicted_values, zero_division=0
    ),
    "Balanced accuracy": balanced_accuracy_score(
        actual_values, predicted_values
    ),
    "Average precision": average_precision_score(
        actual_values, probability_values
    ),
    "ROC-AUC": roc_auc_score(
        actual_values, probability_values
    )
}

# Calculate percentile-based 95% confidence intervals.
rf_bootstrap_summary = pd.DataFrame({
    "Metric": list(observed_metrics.keys()),
    "Observed estimate": list(observed_metrics.values()),
    "95% CI lower": [
        rf_bootstrap_df[column].quantile(0.025)
        for column in observed_metrics
    ],
    "95% CI upper": [
        rf_bootstrap_df[column].quantile(0.975)
        for column in observed_metrics
    ],
    "Valid bootstrap replicates": len(rf_bootstrap_df)
})

rf_bootstrap_summary[
    ["Observed estimate", "95% CI lower", "95% CI upper"]
] = rf_bootstrap_summary[
    ["Observed estimate", "95% CI lower", "95% CI upper"]
].round(4)

display(rf_bootstrap_summary)

,Metric,Observed estimate,95% CI lower,95% CI upper,Valid bootstrap replicates
0,Precision,0.5707,0.4428,0.6791,2000
1,Recall,0.3494,0.2641,0.4286,2000
2,F1-score,0.4334,0.3333,0.5225,2000
3,Balanced accuracy,0.6533,0.6088,0.6948,2000
4,Average precision,0.4560,0.3452,0.5588,2000
5,ROC-AUC,0.7456,0.6904,0.7921,2000


In [52]:
# CELL 55: PAIRED POND-LEVEL MODEL COMPARISON

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Use identical test observations for both models.
comparison_data = pd.DataFrame({
    "pond_id": test_shared["pond_id"].to_numpy(),
    "actual": np.asarray(y_test_final),
    "logistic_probability": np.asarray(time_aware_probabilities),
    "random_forest_probability": np.asarray(
        rf_three_feature_test_probabilities
    )
})

# Freeze the thresholds selected during validation.
comparison_data["logistic_prediction"] = (
    comparison_data["logistic_probability"] >= 0.20
).astype(int)

comparison_data["random_forest_prediction"] = (
    comparison_data["random_forest_probability"] >= 0.60
).astype(int)

# Metric calculation function.
def calculate_model_metrics(actual, predicted, probabilities):

    return {
        "Precision": precision_score(
            actual, predicted, zero_division=0
        ),
        "Recall": recall_score(
            actual, predicted, zero_division=0
        ),
        "F1-score": f1_score(
            actual, predicted, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            actual, predicted
        ),
        "Average precision": average_precision_score(
            actual, probabilities
        ),
        "ROC-AUC": roc_auc_score(
            actual, probabilities
        )
    }


# Calculate observed metric differences.
actual = comparison_data["actual"].to_numpy()

logistic_metrics = calculate_model_metrics(
    actual,
    comparison_data["logistic_prediction"].to_numpy(),
    comparison_data["logistic_probability"].to_numpy()
)

rf_metrics = calculate_model_metrics(
    actual,
    comparison_data["random_forest_prediction"].to_numpy(),
    comparison_data["random_forest_probability"].to_numpy()
)

# Prepare pond-level bootstrap sampling.
unique_ponds = comparison_data["pond_id"].unique()

pond_positions = {
    pond: np.flatnonzero(
        comparison_data["pond_id"].to_numpy() == pond
    )
    for pond in unique_ponds
}

rng = np.random.default_rng(42)

bootstrap_differences = []
n_bootstrap = 2000

for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_ponds,
        size=len(unique_ponds),
        replace=True
    )

    sampled_positions = np.concatenate([
        pond_positions[pond]
        for pond in sampled_ponds
    ])

    sampled_actual = actual[sampled_positions]

    if len(np.unique(sampled_actual)) < 2:
        continue

    # Logistic Regression metrics.
    logistic_bootstrap = calculate_model_metrics(
        sampled_actual,
        comparison_data["logistic_prediction"].to_numpy()[
            sampled_positions
        ],
        comparison_data["logistic_probability"].to_numpy()[
            sampled_positions
        ]
    )

    # Random Forest metrics.
    rf_bootstrap = calculate_model_metrics(
        sampled_actual,
        comparison_data["random_forest_prediction"].to_numpy()[
            sampled_positions
        ],
        comparison_data["random_forest_probability"].to_numpy()[
            sampled_positions
        ]
    )

    # Calculate paired differences.
    bootstrap_differences.append({
        metric: rf_bootstrap[metric] - logistic_bootstrap[metric]
        for metric in logistic_bootstrap
    })

bootstrap_difference_df = pd.DataFrame(bootstrap_differences)

# Consolidate observed differences and 95% confidence intervals.
model_difference_summary = pd.DataFrame({
    "Metric": list(logistic_metrics.keys()),
    "Logistic Regression": list(logistic_metrics.values()),
    "Random Forest": list(rf_metrics.values()),
    "Difference (RF - LR)": [
        rf_metrics[metric] - logistic_metrics[metric]
        for metric in logistic_metrics
    ],
    "95% CI lower": [
        bootstrap_difference_df[metric].quantile(0.025)
        for metric in logistic_metrics
    ],
    "95% CI upper": [
        bootstrap_difference_df[metric].quantile(0.975)
        for metric in logistic_metrics
    ],
    "Valid bootstrap replicates": len(bootstrap_difference_df)
})

numeric_columns = [
    "Logistic Regression",
    "Random Forest",
    "Difference (RF - LR)",
    "95% CI lower",
    "95% CI upper"
]

model_difference_summary[numeric_columns] = (
    model_difference_summary[numeric_columns].round(4)
)

display(model_difference_summary)

,Metric,Logistic Regression,Random Forest,Difference (RF - LR),95% CI lower,95% CI upper,Valid bootstrap replicates
0,Precision,0.2037,0.5707,0.3670,0.2685,0.4464,2000
1,Recall,0.5994,0.3494,-0.2500,-0.3152,-0.1875,2000
2,F1-score,0.3041,0.4334,0.1293,0.0584,0.1927,2000
3,Balanced accuracy,0.6093,0.6533,0.0440,0.0104,0.0765,2000
4,Average precision,0.3722,0.4560,0.0838,0.0402,0.1236,2000
5,ROC-AUC,0.6701,0.7456,0.0756,0.0371,0.1115,2000


In [53]:
# CELL 56: NELLORE THREE-FEATURE RANDOM FOREST VALIDATION

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Use the same three predictors selected for the leading model.
X_nellore_rf_train = nellore_inner_training[predictor_columns].copy()
X_nellore_rf_validation = nellore_validation[predictor_columns].copy()

# Train Random Forest using Nellore's internal training period.
nellore_rf_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

nellore_rf_model.fit(
    X_nellore_rf_train,
    y_nellore_inner
)

# Generate probabilities for the internal validation period.
nellore_rf_validation_probabilities = (
    nellore_rf_model.predict_proba(
        X_nellore_rf_validation
    )[:, 1]
)

# Compare classification thresholds.
nellore_rf_results = []

for threshold in np.arange(0.10, 1.00, 0.10):

    predictions = (
        nellore_rf_validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_nellore_validation,
        predictions,
        labels=[0, 1]
    ).ravel()

    nellore_rf_results.append({
        "Threshold": round(threshold, 2),
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Precision": precision_score(
            y_nellore_validation, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_nellore_validation, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_nellore_validation, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_nellore_validation, predictions
        ),
        "Average precision": average_precision_score(
            y_nellore_validation,
            nellore_rf_validation_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_nellore_validation,
            nellore_rf_validation_probabilities
        )
    })

nellore_rf_summary = pd.DataFrame(nellore_rf_results)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

nellore_rf_summary[metric_columns] = (
    nellore_rf_summary[metric_columns].round(4)
)

display(nellore_rf_summary)

,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,0.1,66,355,2,45,89.96,0.1568,0.9706,0.2699,0.5415,0.7188,0.8628
1,0.2,64,248,4,152,66.67,0.2051,0.9412,0.3368,0.6606,0.7188,0.8628
2,0.3,56,136,12,264,41.03,0.2917,0.8235,0.4308,0.7418,0.7188,0.8628
3,0.4,54,70,14,330,26.50,0.4355,0.7941,0.5625,0.8096,0.7188,0.8628
4,0.5,50,42,18,358,19.66,0.5435,0.7353,0.6250,0.8151,0.7188,0.8628
5,0.6,46,22,22,378,14.53,0.6765,0.6765,0.6765,0.8107,0.7188,0.8628
6,0.7,32,6,36,394,8.12,0.8421,0.4706,0.6038,0.7278,0.7188,0.8628
7,0.8,27,1,41,399,5.98,0.9643,0.3971,0.5625,0.6973,0.7188,0.8628
8,0.9,10,0,58,400,2.14,1.0000,0.1471,0.2564,0.5735,0.7188,0.8628


In [54]:
# CELL 57: FINAL NELLORE THREE-FEATURE RANDOM FOREST EVALUATION

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Train using the complete Nellore historical training period.
nellore_rf_final_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

nellore_rf_final_model.fit(
    nellore_train[predictor_columns],
    nellore_train["Next water quality status"].map(target_mapping)
)

# Predict probabilities on the untouched Nellore historical test set.
nellore_rf_test_probabilities = (
    nellore_rf_final_model
    .predict_proba(nellore_test[predictor_columns])[:, 1]
)

# Compare thresholds selected using Nellore validation data.
nellore_rf_test_results = []

for threshold in [0.50, 0.60]:

    predictions = (
        nellore_rf_test_probabilities >= threshold
    ).astype(int)

    actual = (
        nellore_test["Next water quality status"]
        .map(target_mapping)
    )

    tn, fp, fn, tp = confusion_matrix(
        actual,
        predictions,
        labels=[0, 1]
    ).ravel()

    nellore_rf_test_results.append({
        "Model": "Three-feature Random Forest",
        "Threshold": threshold,
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(
            actual, predictions, zero_division=0
        ),
        "Recall": recall_score(
            actual, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            actual, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            actual, predictions
        ),
        "Average precision": average_precision_score(
            actual, nellore_rf_test_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            actual, nellore_rf_test_probabilities
        )
    })

nellore_rf_final_summary = pd.DataFrame(
    nellore_rf_test_results
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

nellore_rf_final_summary[metric_columns] = (
    nellore_rf_final_summary[metric_columns].round(4)
)

display(nellore_rf_final_summary)

,Model,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Accuracy,Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,Three-feature Random Forest,0.5,21,21,23,508,7.33,0.9232,0.5000,0.4773,0.4884,0.7188,0.4818,0.7731
1,Three-feature Random Forest,0.6,21,12,23,517,5.76,0.9389,0.6364,0.4773,0.5455,0.7273,0.4818,0.7731


In [55]:
# CELL 58: NELLORE RANDOM FOREST POND-LEVEL BOOTSTRAP

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Apply the threshold selected during Nellore validation.
nellore_rf_selected_threshold = 0.60

nellore_rf_bootstrap_data = pd.DataFrame({
    "pond_id": nellore_test["pond_id"].to_numpy(),
    "actual": np.asarray(y_nellore_test_final),
    "probability": np.asarray(nellore_rf_test_probabilities)
})

nellore_rf_bootstrap_data["predicted"] = (
    nellore_rf_bootstrap_data["probability"]
    >= nellore_rf_selected_threshold
).astype(int)

# Identify individual ponds and their observation positions.
unique_ponds = nellore_rf_bootstrap_data["pond_id"].unique()

pond_positions = {
    pond: np.flatnonzero(
        nellore_rf_bootstrap_data["pond_id"].to_numpy() == pond
    )
    for pond in unique_ponds
}

rng = np.random.default_rng(42)

bootstrap_results = []
n_bootstrap = 2000

# Resample entire ponds rather than individual observations.
for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_ponds,
        size=len(unique_ponds),
        replace=True
    )

    sampled_positions = np.concatenate([
        pond_positions[pond]
        for pond in sampled_ponds
    ])

    sampled_actual = (
        nellore_rf_bootstrap_data["actual"]
        .to_numpy()[sampled_positions]
    )

    sampled_predictions = (
        nellore_rf_bootstrap_data["predicted"]
        .to_numpy()[sampled_positions]
    )

    sampled_probabilities = (
        nellore_rf_bootstrap_data["probability"]
        .to_numpy()[sampled_positions]
    )

    if len(np.unique(sampled_actual)) < 2:
        continue

    bootstrap_results.append({
        "Precision": precision_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            sampled_actual,
            sampled_predictions,
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            sampled_actual,
            sampled_predictions
        ),
        "Average precision": average_precision_score(
            sampled_actual,
            sampled_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            sampled_actual,
            sampled_probabilities
        )
    })

nellore_rf_bootstrap_df = pd.DataFrame(bootstrap_results)

# Calculate observed estimates.
observed_metrics = {
    "Precision": precision_score(
        y_nellore_test_final,
        nellore_rf_bootstrap_data["predicted"],
        zero_division=0
    ),
    "Recall": recall_score(
        y_nellore_test_final,
        nellore_rf_bootstrap_data["predicted"],
        zero_division=0
    ),
    "F1-score": f1_score(
        y_nellore_test_final,
        nellore_rf_bootstrap_data["predicted"],
        zero_division=0
    ),
    "Balanced accuracy": balanced_accuracy_score(
        y_nellore_test_final,
        nellore_rf_bootstrap_data["predicted"]
    ),
    "Average precision": average_precision_score(
        y_nellore_test_final,
        nellore_rf_test_probabilities
    ),
    "ROC-AUC": roc_auc_score(
        y_nellore_test_final,
        nellore_rf_test_probabilities
    )
}

nellore_rf_bootstrap_summary = pd.DataFrame({
    "Metric": list(observed_metrics.keys()),
    "Observed estimate": list(observed_metrics.values()),
    "95% CI lower": [
        nellore_rf_bootstrap_df[column].quantile(0.025)
        for column in observed_metrics
    ],
    "95% CI upper": [
        nellore_rf_bootstrap_df[column].quantile(0.975)
        for column in observed_metrics
    ],
    "Valid bootstrap replicates": len(nellore_rf_bootstrap_df)
})

numeric_columns = [
    "Observed estimate",
    "95% CI lower",
    "95% CI upper"
]

nellore_rf_bootstrap_summary[numeric_columns] = (
    nellore_rf_bootstrap_summary[numeric_columns].round(4)
)

display(nellore_rf_bootstrap_summary)

,Metric,Observed estimate,95% CI lower,95% CI upper,Valid bootstrap replicates
0,Precision,0.6364,0.4884,0.7805,2000
1,Recall,0.4773,0.3846,0.5641,2000
2,F1-score,0.5455,0.4444,0.6275,2000
3,Balanced accuracy,0.7273,0.6827,0.7708,2000
4,Average precision,0.4818,0.3492,0.5943,2000
5,ROC-AUC,0.7731,0.6772,0.8494,2000


In [56]:
# CELL 59: FAIR THREE-FEATURE MODEL COMPARISON — NELLORE

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Use the same three predictors as Random Forest.
X_nellore_lr_train = nellore_inner_training[predictor_columns].copy()
X_nellore_lr_validation = nellore_validation[predictor_columns].copy()

# Train Logistic Regression on the same historical training period.
nellore_lr_three_feature_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

nellore_lr_three_feature_model.fit(
    X_nellore_lr_train,
    y_nellore_inner
)

# Generate validation probabilities.
nellore_lr_three_probabilities = (
    nellore_lr_three_feature_model
    .predict_proba(X_nellore_lr_validation)[:, 1]
)

# Compare both algorithms on identical validation observations.
comparison_models = {
    "Three-feature Logistic Regression": nellore_lr_three_probabilities,
    "Three-feature Random Forest": nellore_rf_validation_probabilities
}

fair_comparison_results = []

for model_name, probabilities in comparison_models.items():

    for threshold in np.arange(0.10, 1.00, 0.10):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_nellore_validation,
            predictions,
            labels=[0, 1]
        ).ravel()

        fair_comparison_results.append({
            "Model": model_name,
            "Threshold": round(threshold, 2),
            "True positives": tp,
            "False positives": fp,
            "False negatives": fn,
            "True negatives": tn,
            "Alert rate (%)": round(predictions.mean() * 100, 2),
            "Precision": precision_score(
                y_nellore_validation,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_nellore_validation,
                predictions,
                zero_division=0
            ),
            "F1-score": f1_score(
                y_nellore_validation,
                predictions,
                zero_division=0
            ),
            "Balanced accuracy": balanced_accuracy_score(
                y_nellore_validation,
                predictions
            ),
            "Average precision": average_precision_score(
                y_nellore_validation,
                probabilities
            ),
            "ROC-AUC": roc_auc_score(
                y_nellore_validation,
                probabilities
            )
        })

fair_model_comparison = pd.DataFrame(fair_comparison_results)

metric_columns = [
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

fair_model_comparison[metric_columns] = (
    fair_model_comparison[metric_columns].round(4)
)

display(fair_model_comparison)

,Model,Threshold,True positives,False positives,False negatives,True negatives,Alert rate (%),Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,Three-feature Logistic Regression,0.1,68,370,0,30,93.59,0.1553,1.0000,0.2688,0.5375,0.5652,0.7603
1,Three-feature Logistic Regression,0.2,58,248,10,152,65.38,0.1895,0.8529,0.3102,0.6165,0.5652,0.7603
2,Three-feature Logistic Regression,0.3,44,123,24,277,35.68,0.2635,0.6471,0.3745,0.6698,0.5652,0.7603
3,Three-feature Logistic Regression,0.4,35,27,33,373,13.25,0.5645,0.5147,0.5385,0.7236,0.5652,0.7603
4,Three-feature Logistic Regression,0.5,20,3,48,397,4.91,0.8696,0.2941,0.4396,0.6433,0.5652,0.7603
5,Three-feature Logistic Regression,0.6,10,1,58,399,2.35,0.9091,0.1471,0.2532,0.5723,0.5652,0.7603
6,Three-feature Logistic Regression,0.7,5,1,63,399,1.28,0.8333,0.0735,0.1351,0.5355,0.5652,0.7603
7,Three-feature Logistic Regression,0.8,1,0,67,400,0.21,1.0000,0.0147,0.0290,0.5074,0.5652,0.7603
8,Three-feature Logistic Regression,0.9,0,0,68,400,0.00,0.0000,0.0000,0.0000,0.5000,0.5652,0.7603
9,Three-feature Random Forest,0.1,66,355,2,45,89.96,0.1568,0.9706,0.2699,0.5415,0.7188,0.8628


In [57]:
# CELL 60: FINAL MATCHED THREE-FEATURE COMPARISON — NELLORE

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Validation-selected thresholds based on maximum balanced accuracy.
selected_thresholds = {
    "Three-feature Logistic Regression": 0.4,
    "Three-feature Random Forest": 0.5
}

# Prepare the same three predictors for final training and testing.
X_nellore_final_train = nellore_train[predictor_columns].copy()
X_nellore_final_test = nellore_test[predictor_columns].copy()

y_nellore_final_train = (
    nellore_train["Next water quality status"]
    .map({"Yes": 0, "No": 1})
    .astype(int)
)

y_nellore_final_test = (
    nellore_test["Next water quality status"]
    .map({"Yes": 0, "No": 1})
    .astype(int)
)

# Train three-feature Logistic Regression.
nellore_final_lr_three = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

nellore_final_lr_three.fit(
    X_nellore_final_train,
    y_nellore_final_train
)

# Train three-feature Random Forest using the same configuration
# used in the earlier Nellore Random Forest experiment.
nellore_final_rf_three = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

nellore_final_rf_three.fit(
    X_nellore_final_train,
    y_nellore_final_train
)

# Generate probabilities on the identical untouched test observations.
final_test_probabilities = {
    "Three-feature Logistic Regression":
        nellore_final_lr_three.predict_proba(
            X_nellore_final_test
        )[:, 1],

    "Three-feature Random Forest":
        nellore_final_rf_three.predict_proba(
            X_nellore_final_test
        )[:, 1]
}

# Evaluate both models using their validation-selected thresholds.
final_comparison_rows = []

for model_name, probabilities in final_test_probabilities.items():

    threshold = selected_thresholds[model_name]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_nellore_final_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    final_comparison_rows.append({
        "Model": model_name,
        "Selected threshold": threshold,
        "Test observations": len(y_nellore_final_test),
        "True positives": tp,
        "False positives": fp,
        "False negatives": fn,
        "True negatives": tn,
        "Alert rate (%)": round(predictions.mean() * 100, 2),
        "Accuracy": accuracy_score(
            y_nellore_final_test,
            predictions
        ),
        "Precision": precision_score(
            y_nellore_final_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_nellore_final_test,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_nellore_final_test,
            predictions,
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_nellore_final_test,
            predictions
        ),
        "Average precision": average_precision_score(
            y_nellore_final_test,
            probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_nellore_final_test,
            probabilities
        )
    })

nellore_final_matched_comparison = pd.DataFrame(
    final_comparison_rows
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

nellore_final_matched_comparison[metric_columns] = (
    nellore_final_matched_comparison[metric_columns].round(4)
)

display(nellore_final_matched_comparison)

,Model,Selected threshold,Test observations,True positives,False positives,False negatives,True negatives,Alert rate (%),Accuracy,Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC
0,Three-feature Logistic Regression,0.4,573,17,10,27,519,4.71,0.9354,0.6296,0.3864,0.4789,0.6837,0.4413,0.6313
1,Three-feature Random Forest,0.5,573,21,21,23,508,7.33,0.9232,0.5000,0.4773,0.4884,0.7188,0.4818,0.7731


In [58]:
# CELL 61: PAIRED POND-LEVEL BOOTSTRAP — NELLORE MODEL COMPARISON

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

rng = np.random.default_rng(42)
n_bootstrap = 2000

# Prepare test outcomes, pond IDs and model probabilities.
y_test_array = y_nellore_final_test.to_numpy()
pond_ids = nellore_test["pond_id"].to_numpy()

lr_probabilities = final_test_probabilities[
    "Three-feature Logistic Regression"
]

rf_probabilities = final_test_probabilities[
    "Three-feature Random Forest"
]

lr_threshold = selected_thresholds[
    "Three-feature Logistic Regression"
]

rf_threshold = selected_thresholds[
    "Three-feature Random Forest"
]

lr_predictions = (lr_probabilities >= lr_threshold).astype(int)
rf_predictions = (rf_probabilities >= rf_threshold).astype(int)

unique_ponds = np.unique(pond_ids)

bootstrap_differences = {
    "Precision": [],
    "Recall": [],
    "F1-score": [],
    "Balanced accuracy": [],
    "Average precision": [],
    "ROC-AUC": []
}

# Resample ponds, not individual observations.
for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_ponds,
        size=len(unique_ponds),
        replace=True
    )

    sampled_positions = np.concatenate([
        np.flatnonzero(pond_ids == pond_id)
        for pond_id in sampled_ponds
    ])

    y_sample = y_test_array[sampled_positions]

    # Both classes must be present for a meaningful comparison.
    if len(np.unique(y_sample)) < 2:
        continue

    lr_pred_sample = lr_predictions[sampled_positions]
    rf_pred_sample = rf_predictions[sampled_positions]

    lr_prob_sample = lr_probabilities[sampled_positions]
    rf_prob_sample = rf_probabilities[sampled_positions]

    lr_metrics = {
        "Precision": precision_score(
            y_sample, lr_pred_sample, zero_division=0
        ),
        "Recall": recall_score(
            y_sample, lr_pred_sample, zero_division=0
        ),
        "F1-score": f1_score(
            y_sample, lr_pred_sample, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_sample, lr_pred_sample
        ),
        "Average precision": average_precision_score(
            y_sample, lr_prob_sample
        ),
        "ROC-AUC": roc_auc_score(
            y_sample, lr_prob_sample
        )
    }

    rf_metrics = {
        "Precision": precision_score(
            y_sample, rf_pred_sample, zero_division=0
        ),
        "Recall": recall_score(
            y_sample, rf_pred_sample, zero_division=0
        ),
        "F1-score": f1_score(
            y_sample, rf_pred_sample, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_sample, rf_pred_sample
        ),
        "Average precision": average_precision_score(
            y_sample, rf_prob_sample
        ),
        "ROC-AUC": roc_auc_score(
            y_sample, rf_prob_sample
        )
    }

    for metric in bootstrap_differences:
        bootstrap_differences[metric].append(
            rf_metrics[metric] - lr_metrics[metric]
        )

# Summarise paired differences: Random Forest minus Logistic Regression.
bootstrap_rows = []

for metric, differences in bootstrap_differences.items():

    bootstrap_rows.append({
        "Metric": metric,
        "Mean difference (RF - LR)": np.mean(differences),
        "95% CI lower": np.percentile(differences, 2.5),
        "95% CI upper": np.percentile(differences, 97.5)
    })

paired_bootstrap_results = pd.DataFrame(bootstrap_rows)

numeric_columns = [
    "Mean difference (RF - LR)",
    "95% CI lower",
    "95% CI upper"
]

paired_bootstrap_results[numeric_columns] = (
    paired_bootstrap_results[numeric_columns].round(4)
)

print(
    f"Valid bootstrap samples: "
    f"{len(next(iter(bootstrap_differences.values())))} "
    f"out of {n_bootstrap}"
)

display(paired_bootstrap_results)

Valid bootstrap samples: 2000 out of 2000


,Metric,Mean difference (RF - LR),95% CI lower,95% CI upper
0,Precision,-0.1207,-0.2857,0.0479
1,Recall,0.0988,-0.0962,0.2750
2,F1-score,0.0206,-0.1335,0.1904
3,Balanced accuracy,0.0390,-0.0559,0.1251
4,Average precision,0.0464,-0.0607,0.1558
5,ROC-AUC,0.1423,0.0621,0.2394


In [59]:
# CELL 62: CONSOLIDATED REGIONAL MODEL COMPARISON

regional_model_results = [
    {
        "Region": "Eluru",
        "Model": "Logistic Regression",
        "Predictors": 3,
        "Threshold": 0.2,
        "Test observations": 2232,
        "Test ponds": 118,
        "Out-of-range observations": 312,
        "True positives": 191,
        "False positives": 863,
        "False negatives": 121,
        "True negatives": 1057,
        "Accuracy": 0.5591,
        "Precision": 0.1812,
        "Recall": 0.6122,
        "F1-score": 0.2796,
        "Balanced accuracy": 0.5814,
        "Average precision": 0.3797,
        "ROC-AUC": 0.6501
    },
    {
        "Region": "Eluru",
        "Model": "Random Forest",
        "Predictors": 3,
        "Threshold": 0.4,
        "Test observations": 2232,
        "Test ponds": 118,
        "Out-of-range observations": 312,
        "True positives": 163,
        "False positives": 351,
        "False negatives": 149,
        "True negatives": 1569,
        "Accuracy": 0.7762,
        "Precision": 0.3171,
        "Recall": 0.5224,
        "F1-score": 0.3947,
        "Balanced accuracy": 0.6698,
        "Average precision": 0.4560,
        "ROC-AUC": 0.7456
    },
    {
        "Region": "Nellore",
        "Model": "Logistic Regression",
        "Predictors": 3,
        "Threshold": 0.4,
        "Test observations": 573,
        "Test ponds": 49,
        "Out-of-range observations": 44,
        "True positives": 17,
        "False positives": 10,
        "False negatives": 27,
        "True negatives": 519,
        "Accuracy": 0.9354,
        "Precision": 0.6296,
        "Recall": 0.3864,
        "F1-score": 0.4789,
        "Balanced accuracy": 0.6837,
        "Average precision": 0.4413,
        "ROC-AUC": 0.6313
    },
    {
        "Region": "Nellore",
        "Model": "Random Forest",
        "Predictors": 3,
        "Threshold": 0.5,
        "Test observations": 573,
        "Test ponds": 49,
        "Out-of-range observations": 44,
        "True positives": 21,
        "False positives": 21,
        "False negatives": 23,
        "True negatives": 508,
        "Accuracy": 0.9232,
        "Precision": 0.5000,
        "Recall": 0.4773,
        "F1-score": 0.4884,
        "Balanced accuracy": 0.7188,
        "Average precision": 0.4818,
        "ROC-AUC": 0.7731
    }
]

regional_model_comparison = pd.DataFrame(
    regional_model_results
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

regional_model_comparison[metric_columns] = (
    regional_model_comparison[metric_columns].round(4)
)

regional_model_comparison["Alert rate (%)"] = (
    (
        regional_model_comparison["True positives"]
        + regional_model_comparison["False positives"]
    )
    / regional_model_comparison["Test observations"]
    * 100
).round(2)

display(regional_model_comparison)

,Region,Model,Predictors,Threshold,Test observations,Test ponds,Out-of-range observations,True positives,False positives,False negatives,True negatives,Accuracy,Precision,Recall,F1-score,Balanced accuracy,Average precision,ROC-AUC,Alert rate (%)
0,Eluru,Logistic Regression,3,0.2,2232,118,312,191,863,121,1057,0.5591,0.1812,0.6122,0.2796,0.5814,0.3797,0.6501,47.22
1,Eluru,Random Forest,3,0.4,2232,118,312,163,351,149,1569,0.7762,0.3171,0.5224,0.3947,0.6698,0.4560,0.7456,23.03
2,Nellore,Logistic Regression,3,0.4,573,49,44,17,10,27,519,0.9354,0.6296,0.3864,0.4789,0.6837,0.4413,0.6313,4.71
3,Nellore,Random Forest,3,0.5,573,49,44,21,21,23,508,0.9232,0.5000,0.4773,0.4884,0.7188,0.4818,0.7731,7.33


In [60]:
# CELL 63: SAVE CONSOLIDATED REGIONAL MODEL RESULTS

from pathlib import Path

output_directory = Path("../outputs/tables")
output_directory.mkdir(parents=True, exist_ok=True)

output_file = output_directory / "regional_model_comparison.csv"

regional_model_comparison.to_csv(
    output_file,
    index=False
)

print("Regional model comparison saved successfully.")
print(f"File: {output_file.resolve()}")

Regional model comparison saved successfully.
File: C:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\tables\regional_model_comparison.csv


## Predictive Modelling — Results and Conclusions

### 1. Modelling objective

The objective was to investigate whether the water-quality status at the next recorded observation for a pond could be predicted using an earlier observation from the same pond.

The target was encoded as:

- 0 — Water quality in range (Yes)
- 1 — Water quality out of range (No)

The initial models used three predictors:

- pH
- Dissolved oxygen (DO)
- Water temperature

The evaluation used chronological partitions rather than random train-test splitting to reduce the risk of future-data leakage.

### 2. Regional test results

Both Logistic Regression and Random Forest were evaluated using three predictors.

| Region | Model | Accuracy | Precision | Recall | F1-score | Balanced accuracy | Average precision | ROC-AUC |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Eluru | Logistic Regression | 0.5591 | 0.1812 | 0.6122 | 0.2796 | 0.5814 | 0.3797 | 0.6501 |
| Eluru | Random Forest | 0.7762 | 0.3171 | 0.5224 | 0.3947 | 0.6698 | 0.4560 | 0.7456 |
| Nellore | Logistic Regression | 0.9354 | 0.6296 | 0.3864 | 0.4789 | 0.6837 | 0.4413 | 0.6313 |
| Nellore | Random Forest | 0.9232 | 0.5000 | 0.4773 | 0.4884 | 0.7188 | 0.4818 | 0.7731 |

Thresholds were selected using the corresponding regional validation data, not the final test data.

### 3. Regional findings

**Eluru**

Random Forest achieved higher accuracy, precision, F1-score, balanced accuracy, average precision and ROC-AUC than Logistic Regression at the selected thresholds.

Logistic Regression achieved higher recall, detecting a larger proportion of future out-of-range observations, but generated substantially more alerts.

**Nellore**

Random Forest achieved higher recall, F1-score, balanced accuracy, average precision and ROC-AUC.

Logistic Regression achieved higher accuracy and precision, with fewer false alerts.

The differences illustrate the trade-off between detecting more potentially problematic observations and limiting false alerts.

### 4. Uncertainty assessment

A paired pond-level bootstrap comparison was conducted for the matched three-feature models in Nellore.

The ROC-AUC difference favoured Random Forest:

- Mean difference (Random Forest minus Logistic Regression): 0.1423
- 95% confidence interval: [0.0621, 0.2394]

The confidence intervals for precision, recall, F1-score, balanced accuracy and average precision included zero.

Therefore, the Nellore results provide evidence of better ranking performance for Random Forest, but do not establish superiority across all classification metrics.

### 5. Research interpretation

Random Forest is a promising candidate for further investigation because it achieved stronger ROC-AUC and average precision in both regional test evaluations.

However, the preferred model depends on the intended operational objective. Higher recall may be important when missing a potentially dangerous water-quality condition is costly, whereas higher precision may be important when frequent false alerts create unnecessary work for farmers.

The results do not establish that either model is ready for real-world deployment.

### 6. Limitations

The following limitations must be considered:

1. The target represents the recorded water-quality classification at the next observation, not a fixed future time horizon.
2. Observation intervals are irregular and vary between ponds.
3. Measurements are repeated within ponds, so observations are not statistically independent.
4. The Eluru and Nellore evaluations cover different chronological periods and sample sizes.
5. The dataset contains changes in ammonia measurement protocols and potentially changing water-quality classification practices.
6. The models initially use only pH, DO and temperature. Other environmental and operational variables may contain additional information.
7. The test results are retrospective and do not demonstrate that the models improve aquaculture outcomes.

### 7. Current conclusion

The experiments demonstrate that pH, dissolved oxygen and temperature contain predictive information about subsequent recorded water-quality status.

Random Forest shows promising discrimination performance, but the observed results vary by region and evaluation metric.

Further validation, uncertainty analysis and scientific interpretation are required before recommending a model for practical aquaculture use.

In [61]:
# CELL 64: PAIRED POND-LEVEL BOOTSTRAP — ELURU MODEL COMPARISON

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Use the same three predictors and identical Eluru data partitions.
X_eluru_train = X_train.copy()
X_eluru_test = X_test_primary.copy()

y_eluru_train = np.asarray(y_train, dtype=int)
y_eluru_test = np.asarray(y_test_primary, dtype=int)

eluru_test_ponds = test_shared["pond_id"].to_numpy()

# Confirm that the test observations and pond identifiers align.
assert len(X_eluru_test) == len(y_eluru_test) == len(eluru_test_ponds)
assert X_eluru_train.shape[1] == 3

# Previously evaluated operating thresholds.
eluru_thresholds = {
    "Three-feature Logistic Regression": 0.2,
    "Three-feature Random Forest": 0.6
}

# Train three-feature Logistic Regression.
eluru_lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

eluru_lr_model.fit(
    X_eluru_train,
    y_eluru_train
)

# Train three-feature Random Forest.
eluru_rf_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

eluru_rf_model.fit(
    X_eluru_train,
    y_eluru_train
)

# Obtain probabilities on the identical Eluru test observations.
eluru_test_probabilities = {
    "Three-feature Logistic Regression":
        eluru_lr_model.predict_proba(X_eluru_test)[:, 1],

    "Three-feature Random Forest":
        eluru_rf_model.predict_proba(X_eluru_test)[:, 1]
}

# Apply the previously selected thresholds.
eluru_test_predictions = {
    model_name: (
        probabilities >= eluru_thresholds[model_name]
    ).astype(int)
    for model_name, probabilities in eluru_test_probabilities.items()
}

# Calculate original test metrics.
def calculate_eluru_metrics(y_true, predictions, probabilities):
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_true, predictions, zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "Average precision": average_precision_score(
            y_true, probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_true, probabilities
        )
    }

lr_original_metrics = calculate_eluru_metrics(
    y_eluru_test,
    eluru_test_predictions["Three-feature Logistic Regression"],
    eluru_test_probabilities["Three-feature Logistic Regression"]
)

rf_original_metrics = calculate_eluru_metrics(
    y_eluru_test,
    eluru_test_predictions["Three-feature Random Forest"],
    eluru_test_probabilities["Three-feature Random Forest"]
)

# Paired pond-level bootstrap.
rng = np.random.default_rng(42)
n_bootstrap = 2000

unique_eluru_ponds = np.unique(eluru_test_ponds)

bootstrap_differences = {
    metric: []
    for metric in lr_original_metrics
}

for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_eluru_ponds,
        size=len(unique_eluru_ponds),
        replace=True
    )

    sampled_positions = np.concatenate([
        np.flatnonzero(eluru_test_ponds == pond_id)
        for pond_id in sampled_ponds
    ])

    y_sample = y_eluru_test[sampled_positions]

    if len(np.unique(y_sample)) < 2:
        continue

    lr_pred = eluru_test_predictions[
        "Three-feature Logistic Regression"
    ][sampled_positions]

    rf_pred = eluru_test_predictions[
        "Three-feature Random Forest"
    ][sampled_positions]

    lr_prob = eluru_test_probabilities[
        "Three-feature Logistic Regression"
    ][sampled_positions]

    rf_prob = eluru_test_probabilities[
        "Three-feature Random Forest"
    ][sampled_positions]

    lr_metrics = calculate_eluru_metrics(
        y_sample, lr_pred, lr_prob
    )

    rf_metrics = calculate_eluru_metrics(
        y_sample, rf_pred, rf_prob
    )

    for metric in bootstrap_differences:
        bootstrap_differences[metric].append(
            rf_metrics[metric] - lr_metrics[metric]
        )

# Summarise paired differences: Random Forest minus Logistic Regression.
bootstrap_rows = []

for metric, differences in bootstrap_differences.items():

    bootstrap_rows.append({
        "Metric": metric,
        "Mean difference (RF - LR)": np.mean(differences),
        "95% CI lower": np.percentile(differences, 2.5),
        "95% CI upper": np.percentile(differences, 97.5)
    })

eluru_paired_bootstrap_results = pd.DataFrame(bootstrap_rows)

numeric_columns = [
    "Mean difference (RF - LR)",
    "95% CI lower",
    "95% CI upper"
]

eluru_paired_bootstrap_results[numeric_columns] = (
    eluru_paired_bootstrap_results[numeric_columns].round(4)
)

print(
    f"Eluru test observations: {len(y_eluru_test)} | "
    f"Test ponds: {len(unique_eluru_ponds)}"
)

print(
    f"Valid bootstrap samples: "
    f"{len(next(iter(bootstrap_differences.values())))} "
    f"out of {n_bootstrap}"
)

display(eluru_paired_bootstrap_results)

Eluru test observations: 2232 | Test ponds: 118
Valid bootstrap samples: 2000 out of 2000


,Metric,Mean difference (RF - LR),95% CI lower,95% CI upper
0,Accuracy,0.3134,0.2848,0.3404
1,Precision,0.3850,0.2922,0.4675
2,Recall,-0.2645,-0.3240,-0.1994
3,F1-score,0.1507,0.0875,0.2123
4,Balanced accuracy,0.0713,0.0382,0.1064
5,Average precision,0.0741,0.0269,0.1219
6,ROC-AUC,0.0950,0.0520,0.1395


In [62]:
# CELL 65: CORRECTED PAIRED POND-LEVEL BOOTSTRAP — ELURU

# Use the validation-selected thresholds reported in Cell 62.
corrected_thresholds = {
    "Three-feature Logistic Regression": 0.2,
    "Three-feature Random Forest": 0.4
}

y_true = y_eluru_test
pond_ids = eluru_test_ponds

lr_prob = eluru_test_probabilities[
    "Three-feature Logistic Regression"
]

rf_prob = eluru_test_probabilities[
    "Three-feature Random Forest"
]

lr_pred = (lr_prob >= corrected_thresholds[
    "Three-feature Logistic Regression"
]).astype(int)

rf_pred = (rf_prob >= corrected_thresholds[
    "Three-feature Random Forest"
]).astype(int)

unique_ponds = np.unique(pond_ids)

bootstrap_differences_corrected = {
    "Accuracy": [],
    "Precision": [],
    "Recall": [],
    "F1-score": [],
    "Balanced accuracy": [],
    "Average precision": [],
    "ROC-AUC": []
}

rng = np.random.default_rng(42)
n_bootstrap = 2000

for _ in range(n_bootstrap):

    sampled_ponds = rng.choice(
        unique_ponds,
        size=len(unique_ponds),
        replace=True
    )

    positions = np.concatenate([
        np.flatnonzero(pond_ids == pond)
        for pond in sampled_ponds
    ])

    y_sample = y_true[positions]

    if len(np.unique(y_sample)) < 2:
        continue

    lr_predictions = lr_pred[positions]
    rf_predictions = rf_pred[positions]

    lr_probabilities = lr_prob[positions]
    rf_probabilities = rf_prob[positions]

    lr_metrics = calculate_eluru_metrics(
        y_sample,
        lr_predictions,
        lr_probabilities
    )

    rf_metrics = calculate_eluru_metrics(
        y_sample,
        rf_predictions,
        rf_probabilities
    )

    for metric in bootstrap_differences_corrected:
        bootstrap_differences_corrected[metric].append(
            rf_metrics[metric] - lr_metrics[metric]
        )

corrected_rows = []

for metric, differences in bootstrap_differences_corrected.items():

    corrected_rows.append({
        "Metric": metric,
        "Mean difference (RF - LR)": np.mean(differences),
        "95% CI lower": np.percentile(differences, 2.5),
        "95% CI upper": np.percentile(differences, 97.5)
    })

corrected_eluru_bootstrap = pd.DataFrame(corrected_rows)

numeric_columns = [
    "Mean difference (RF - LR)",
    "95% CI lower",
    "95% CI upper"
]

corrected_eluru_bootstrap[numeric_columns] = (
    corrected_eluru_bootstrap[numeric_columns].round(4)
)

print("Eluru corrected threshold comparison")
print("Logistic Regression threshold: 0.2")
print("Random Forest threshold: 0.4")
print(f"Test ponds: {len(unique_ponds)}")
print(
    "Valid bootstrap samples:",
    len(next(iter(bootstrap_differences_corrected.values()))),
    "out of",
    n_bootstrap
)

display(corrected_eluru_bootstrap)

Eluru corrected threshold comparison
Logistic Regression threshold: 0.2
Random Forest threshold: 0.4
Test ponds: 118
Valid bootstrap samples: 2000 out of 2000


,Metric,Mean difference (RF - LR),95% CI lower,95% CI upper
0,Accuracy,0.2166,0.1880,0.2458
1,Precision,0.1337,0.0942,0.1734
2,Recall,-0.0917,-0.1648,-0.0166
3,F1-score,0.1127,0.0713,0.1540
4,Balanced accuracy,0.0874,0.0489,0.1255
5,Average precision,0.0741,0.0269,0.1219
6,ROC-AUC,0.0950,0.0520,0.1395


In [63]:
# CELL 65: CONSOLIDATED REGIONAL BOOTSTRAP COMPARISON

# Combine the corrected Eluru and Nellore bootstrap results.
eluru_bootstrap_summary = corrected_eluru_bootstrap.copy()
eluru_bootstrap_summary["Region"] = "Eluru"

nellore_bootstrap_summary = paired_bootstrap_results.copy()
nellore_bootstrap_summary["Region"] = "Nellore"

regional_bootstrap_summary = pd.concat(
    [
        eluru_bootstrap_summary,
        nellore_bootstrap_summary
    ],
    ignore_index=True
)

# Identify whether each confidence interval excludes zero.
regional_bootstrap_summary["CI excludes zero"] = (
    (regional_bootstrap_summary["95% CI lower"] > 0)
    | (regional_bootstrap_summary["95% CI upper"] < 0)
)

# Describe the direction of the difference.
regional_bootstrap_summary["Direction"] = np.where(
    regional_bootstrap_summary["Mean difference (RF - LR)"] > 0,
    "Favours Random Forest",
    "Favours Logistic Regression"
)

# Keep a clear reporting order.
regional_bootstrap_summary = regional_bootstrap_summary[
    [
        "Region",
        "Metric",
        "Mean difference (RF - LR)",
        "95% CI lower",
        "95% CI upper",
        "CI excludes zero",
        "Direction"
    ]
]

display(regional_bootstrap_summary)

# Save the consolidated results.
output_file = Path(
    "../outputs/tables/regional_bootstrap_comparison.csv"
)

output_file.parent.mkdir(parents=True, exist_ok=True)

regional_bootstrap_summary.to_csv(
    output_file,
    index=False
)

print(f"Saved: {output_file.resolve()}")

,Region,Metric,Mean difference (RF - LR),95% CI lower,95% CI upper,CI excludes zero,Direction
0,Eluru,Accuracy,0.2166,0.1880,0.2458,True,Favours Random Forest
1,Eluru,Precision,0.1337,0.0942,0.1734,True,Favours Random Forest
2,Eluru,Recall,-0.0917,-0.1648,-0.0166,True,Favours Logistic Regression
3,Eluru,F1-score,0.1127,0.0713,0.1540,True,Favours Random Forest
4,Eluru,Balanced accuracy,0.0874,0.0489,0.1255,True,Favours Random Forest
5,Eluru,Average precision,0.0741,0.0269,0.1219,True,Favours Random Forest
6,Eluru,ROC-AUC,0.0950,0.0520,0.1395,True,Favours Random Forest
7,Nellore,Precision,-0.1207,-0.2857,0.0479,False,Favours Logistic Regression
8,Nellore,Recall,0.0988,-0.0962,0.2750,False,Favours Random Forest
9,Nellore,F1-score,0.0206,-0.1335,0.1904,False,Favours Random Forest


Saved: C:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\tables\regional_bootstrap_comparison.csv


In [64]:
# CELL 66: ADD NELLORE ACCURACY TO PAIRED BOOTSTRAP SUMMARY

y_test = y_nellore_final_test.to_numpy()
pond_ids = nellore_test["pond_id"].to_numpy()

lr_prob = final_test_probabilities[
    "Three-feature Logistic Regression"
]

rf_prob = final_test_probabilities[
    "Three-feature Random Forest"
]

lr_pred = (
    lr_prob >= selected_thresholds[
        "Three-feature Logistic Regression"
    ]
).astype(int)

rf_pred = (
    rf_prob >= selected_thresholds[
        "Three-feature Random Forest"
    ]
).astype(int)

unique_ponds = np.unique(pond_ids)

rng = np.random.default_rng(42)
accuracy_differences = []

for _ in range(2000):

    sampled_ponds = rng.choice(
        unique_ponds,
        size=len(unique_ponds),
        replace=True
    )

    positions = np.concatenate([
        np.flatnonzero(pond_ids == pond)
        for pond in sampled_ponds
    ])

    y_sample = y_test[positions]

    if len(np.unique(y_sample)) < 2:
        continue

    lr_accuracy = accuracy_score(
        y_sample,
        lr_pred[positions]
    )

    rf_accuracy = accuracy_score(
        y_sample,
        rf_pred[positions]
    )

    accuracy_differences.append(
        rf_accuracy - lr_accuracy
    )

nellore_accuracy_row = pd.DataFrame([{
    "Region": "Nellore",
    "Metric": "Accuracy",
    "Mean difference (RF - LR)": np.mean(accuracy_differences),
    "95% CI lower": np.percentile(accuracy_differences, 2.5),
    "95% CI upper": np.percentile(accuracy_differences, 97.5),
    "CI excludes zero": (
        np.percentile(accuracy_differences, 2.5) > 0
        or np.percentile(accuracy_differences, 97.5) < 0
    ),
    "Direction": (
        "Favours Random Forest"
        if np.mean(accuracy_differences) > 0
        else "Favours Logistic Regression"
    )
}])

# Add the missing accuracy row to the consolidated summary.
regional_bootstrap_summary = pd.concat(
    [
        regional_bootstrap_summary,
        nellore_accuracy_row
    ],
    ignore_index=True
)

# Keep regions and metrics in a consistent order.
metric_order = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "Balanced accuracy",
    "Average precision",
    "ROC-AUC"
]

regional_bootstrap_summary["Metric"] = pd.Categorical(
    regional_bootstrap_summary["Metric"],
    categories=metric_order,
    ordered=True
)

regional_bootstrap_summary = (
    regional_bootstrap_summary
    .sort_values(["Region", "Metric"])
    .reset_index(drop=True)
)

regional_bootstrap_summary["Metric"] = (
    regional_bootstrap_summary["Metric"].astype(str)
)

numeric_columns = [
    "Mean difference (RF - LR)",
    "95% CI lower",
    "95% CI upper"
]

regional_bootstrap_summary[numeric_columns] = (
    regional_bootstrap_summary[numeric_columns].round(4)
)

# Save the complete comparison.
output_file = Path(
    "../outputs/tables/regional_bootstrap_comparison.csv"
)

regional_bootstrap_summary.to_csv(
    output_file,
    index=False
)

display(regional_bootstrap_summary)

,Region,Metric,Mean difference (RF - LR),95% CI lower,95% CI upper,CI excludes zero,Direction
0,Eluru,Accuracy,0.2166,0.1880,0.2458,True,Favours Random Forest
1,Eluru,Precision,0.1337,0.0942,0.1734,True,Favours Random Forest
2,Eluru,Recall,-0.0917,-0.1648,-0.0166,True,Favours Logistic Regression
3,Eluru,F1-score,0.1127,0.0713,0.1540,True,Favours Random Forest
4,Eluru,Balanced accuracy,0.0874,0.0489,0.1255,True,Favours Random Forest
5,Eluru,Average precision,0.0741,0.0269,0.1219,True,Favours Random Forest
6,Eluru,ROC-AUC,0.0950,0.0520,0.1395,True,Favours Random Forest
7,Nellore,Accuracy,-0.0125,-0.0297,0.0039,False,Favours Logistic Regression
8,Nellore,Precision,-0.1207,-0.2857,0.0479,False,Favours Logistic Regression
9,Nellore,Recall,0.0988,-0.0962,0.2750,False,Favours Random Forest


## Regional Model Comparison — Uncertainty Assessment

### 1. Evaluation approach

A paired pond-level bootstrap comparison was conducted separately for Eluru and Nellore.

Both Logistic Regression and Random Forest used the same three predictors:

- pH
- Dissolved oxygen (DO)
- Water temperature

The models were evaluated on identical test observations within each region. Their operating thresholds were selected using regional validation data.

The bootstrap procedure resampled ponds rather than individual observations, preserving the grouping of repeated measurements within each pond. Each regional comparison used 2,000 valid bootstrap samples.

The reported differences are calculated as:

**Random Forest performance minus Logistic Regression performance.**

### 2. Eluru findings

The Eluru test set contained 2,232 observations from 118 ponds.

All seven evaluated metrics had 95% bootstrap confidence intervals that excluded zero.

Random Forest showed higher accuracy, precision, F1-score, balanced accuracy, average precision and ROC-AUC at the selected operating thresholds.

Logistic Regression showed higher recall, indicating that it detected a larger proportion of actual out-of-range observations.

These findings support a performance advantage for Random Forest across several measures in the Eluru evaluation, alongside a recall advantage for Logistic Regression.

### 3. Nellore findings

The Nellore test set contained 573 observations from 49 ponds.

Among the seven evaluated metrics, only the ROC-AUC difference had a 95% bootstrap confidence interval that excluded zero.

The ROC-AUC difference favoured Random Forest:

- Mean difference: 0.1423
- 95% confidence interval: 0.0621 to 0.2394

The confidence intervals for accuracy, precision, recall, F1-score, balanced accuracy and average precision included zero.

Therefore, the Nellore results provide evidence favouring Random Forest in ranking performance, but do not establish a clear advantage across the other evaluated metrics.

### 4. Combined interpretation

The two regional evaluations show different levels of evidence.

- Eluru provides more consistent evidence favouring Random Forest across the evaluated metrics, although Logistic Regression retains a recall advantage.
- Nellore provides clearer evidence favouring Random Forest in ROC-AUC, while differences in the other metrics remain uncertain.

The results do not establish that one algorithm is universally superior across all aquaculture regions.

### 5. Statistical limitations

The bootstrap intervals are conditional on the fitted models, the selected operating thresholds and the available test observations.

They do not account for all uncertainty associated with model training, feature selection or threshold selection.

The regional test sets also cover different historical periods and sample sizes. Consequently, their results should not be interpreted as a direct comparison of model performance between regions.

### 6. Conclusion

Random Forest is a promising candidate for further AQUA model development, particularly because of its consistently stronger ROC-AUC in both regional evaluations.

However, model selection must also consider the practical trade-off between detecting out-of-range conditions and limiting false alerts.

Further validation on independent and more recent pond observations is necessary before recommending either model for operational use.

In [65]:
# CELL 68: FINAL MODELLING RESULTS CONSISTENCY AUDIT

audit_rows = []

for _, row in regional_model_comparison.iterrows():

    tp = int(row["True positives"])
    fp = int(row["False positives"])
    fn = int(row["False negatives"])
    tn = int(row["True negatives"])

    total = int(row["Test observations"])
    out_of_range = int(row["Out-of-range observations"])

    # Recalculate classification metrics from confusion-matrix counts.
    accuracy = (tp + tn) / total
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    specificity = tn / (tn + fp) if tn + fp else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    balanced_accuracy = (recall + specificity) / 2
    alert_rate = (tp + fp) / total * 100

    # Check bootstrap coverage and confidence-interval flags.
    regional_bootstrap = regional_bootstrap_summary[
        regional_bootstrap_summary["Region"] == row["Region"]
    ]

    expected_metrics = {
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced accuracy",
        "Average precision",
        "ROC-AUC"
    }

    actual_metrics = set(regional_bootstrap["Metric"])

    bootstrap_coverage = actual_metrics == expected_metrics

    bootstrap_flags_valid = (
        (
            (
                regional_bootstrap["95% CI lower"] > 0
            )
            |
            (
                regional_bootstrap["95% CI upper"] < 0
            )
        )
        == regional_bootstrap["CI excludes zero"]
    ).all()

    checks = {
        "Confusion counts": tp + fp + fn + tn == total,
        "Out-of-range support": tp + fn == out_of_range,
        "Accuracy": np.isclose(accuracy, row["Accuracy"], atol=0.00015),
        "Precision": np.isclose(precision, row["Precision"], atol=0.00015),
        "Recall": np.isclose(recall, row["Recall"], atol=0.00015),
        "F1-score": np.isclose(f1, row["F1-score"], atol=0.00015),
        "Balanced accuracy": np.isclose(
            balanced_accuracy,
            row["Balanced accuracy"],
            atol=0.00015
        ),
        "Alert rate": np.isclose(
            alert_rate,
            row["Alert rate (%)"],
            atol=0.015
        ),
        "Bootstrap metric coverage": bootstrap_coverage,
        "Bootstrap CI flags": bootstrap_flags_valid
    }

    audit_rows.append({
        "Region": row["Region"],
        "Model": row["Model"],
        "Checks passed": sum(checks.values()),
        "Total checks": len(checks),
        "Result": "PASS" if all(checks.values()) else "REVIEW",
        "Failed checks": ", ".join(
            name for name, passed in checks.items() if not passed
        )
    })

final_modelling_audit = pd.DataFrame(audit_rows)

display(final_modelling_audit)

# Save the audit record.
audit_file = Path(
    "../outputs/tables/final_modelling_consistency_audit.csv"
)

audit_file.parent.mkdir(parents=True, exist_ok=True)

final_modelling_audit.to_csv(
    audit_file,
    index=False
)

print(f"Audit saved: {audit_file.resolve()}")

,Region,Model,Checks passed,Total checks,Result,Failed checks
0,Eluru,Logistic Regression,10,10,PASS,
1,Eluru,Random Forest,9,10,REVIEW,Accuracy
2,Nellore,Logistic Regression,10,10,PASS,
3,Nellore,Random Forest,10,10,PASS,


Audit saved: C:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\tables\final_modelling_consistency_audit.csv


In [66]:
# CELL 68: CORRECT ELURU RANDOM FOREST ACCURACY

mask = (
    (regional_model_comparison["Region"] == "Eluru")
    &
    (regional_model_comparison["Model"] == "Random Forest")
)

tp = regional_model_comparison.loc[mask, "True positives"].iloc[0]
tn = regional_model_comparison.loc[mask, "True negatives"].iloc[0]
total = regional_model_comparison.loc[mask, "Test observations"].iloc[0]

correct_accuracy = (tp + tn) / total

regional_model_comparison.loc[mask, "Accuracy"] = correct_accuracy

# Save the corrected regional comparison.
regional_model_comparison.to_csv(
    "../outputs/tables/regional_model_comparison.csv",
    index=False
)

print(f"Corrected Eluru Random Forest accuracy: {correct_accuracy:.4f}")
print("Regional comparison CSV updated.")

display(
    regional_model_comparison[
        ["Region", "Model", "Accuracy"]
    ]
)

Corrected Eluru Random Forest accuracy: 0.7760
Regional comparison CSV updated.


,Region,Model,Accuracy
0,Eluru,Logistic Regression,0.559100
1,Eluru,Random Forest,0.775986
2,Nellore,Logistic Regression,0.935400
3,Nellore,Random Forest,0.923200


In [67]:
# CELL 70: FINAL MODELLING RESULTS CONSISTENCY AUDIT

audit_rows = []

for _, row in regional_model_comparison.iterrows():

    tp = int(row["True positives"])
    fp = int(row["False positives"])
    fn = int(row["False negatives"])
    tn = int(row["True negatives"])

    total = int(row["Test observations"])
    out_of_range = int(row["Out-of-range observations"])

    accuracy = (tp + tn) / total
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    specificity = tn / (tn + fp) if tn + fp else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    balanced_accuracy = (recall + specificity) / 2
    alert_rate = (tp + fp) / total * 100

    regional_bootstrap = regional_bootstrap_summary[
        regional_bootstrap_summary["Region"] == row["Region"]
    ]

    expected_metrics = {
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Balanced accuracy",
        "Average precision",
        "ROC-AUC"
    }

    bootstrap_coverage = (
        set(regional_bootstrap["Metric"]) == expected_metrics
    )

    bootstrap_flags_valid = (
        (
            (regional_bootstrap["95% CI lower"] > 0)
            | (regional_bootstrap["95% CI upper"] < 0)
        )
        == regional_bootstrap["CI excludes zero"]
    ).all()

    checks = {
        "Confusion counts": tp + fp + fn + tn == total,
        "Out-of-range support": tp + fn == out_of_range,
        "Accuracy": np.isclose(accuracy, row["Accuracy"], atol=0.00015),
        "Precision": np.isclose(precision, row["Precision"], atol=0.00015),
        "Recall": np.isclose(recall, row["Recall"], atol=0.00015),
        "F1-score": np.isclose(f1, row["F1-score"], atol=0.00015),
        "Balanced accuracy": np.isclose(
            balanced_accuracy,
            row["Balanced accuracy"],
            atol=0.00015
        ),
        "Alert rate": np.isclose(
            alert_rate,
            row["Alert rate (%)"],
            atol=0.015
        ),
        "Bootstrap metric coverage": bootstrap_coverage,
        "Bootstrap CI flags": bootstrap_flags_valid
    }

    audit_rows.append({
        "Region": row["Region"],
        "Model": row["Model"],
        "Checks passed": sum(checks.values()),
        "Total checks": len(checks),
        "Result": "PASS" if all(checks.values()) else "REVIEW",
        "Failed checks": ", ".join(
            name for name, passed in checks.items() if not passed
        )
    })

final_modelling_audit = pd.DataFrame(audit_rows)

display(final_modelling_audit)

audit_file = Path(
    "../outputs/tables/final_modelling_consistency_audit.csv"
)

audit_file.parent.mkdir(parents=True, exist_ok=True)

final_modelling_audit.to_csv(audit_file, index=False)

print(f"Audit saved: {audit_file.resolve()}")

,Region,Model,Checks passed,Total checks,Result,Failed checks
0,Eluru,Logistic Regression,10,10,PASS,
1,Eluru,Random Forest,10,10,PASS,
2,Nellore,Logistic Regression,10,10,PASS,
3,Nellore,Random Forest,10,10,PASS,


Audit saved: C:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\tables\final_modelling_consistency_audit.csv


# Final Summary — Predictive Modelling

## 1. Objective

This notebook investigated whether the water-quality status at the next recorded observation for a pond could be predicted using measurements from an earlier observation of the same pond.

The prediction target was:

- `0` — Water quality in range.
- `1` — Water quality out of range.

The initial predictive models used three water-quality measurements:

- pH
- Dissolved oxygen (DO)
- Temperature

The modelling process used chronological data partitions to reduce future-data leakage and evaluated performance separately for Eluru and Nellore.

## 2. Final Regional Test Results

| Region | Model | Test observations | Test ponds | Accuracy | Precision | Recall | F1-score | Balanced accuracy | Average precision | ROC-AUC |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Eluru | Logistic Regression | 2,232 | 118 | 0.5591 | 0.1812 | 0.6122 | 0.2796 | 0.5814 | 0.3797 | 0.6501 |
| Eluru | Random Forest | 2,232 | 118 | 0.7760 | 0.3171 | 0.5224 | 0.3947 | 0.6698 | 0.4560 | 0.7456 |
| Nellore | Logistic Regression | 573 | 49 | 0.9354 | 0.6296 | 0.3864 | 0.4789 | 0.6837 | 0.4413 | 0.6313 |
| Nellore | Random Forest | 573 | 49 | 0.9232 | 0.5000 | 0.4773 | 0.4884 | 0.7188 | 0.4818 | 0.7731 |

Thresholds were selected using the corresponding regional validation datasets.

## 3. Model Comparison Findings

### Eluru

Random Forest demonstrated higher accuracy, precision, F1-score, balanced accuracy, average precision and ROC-AUC at the selected thresholds.

Logistic Regression achieved higher recall, detecting a larger proportion of the actual out-of-range observations, but generated more false alerts.

### Nellore

Random Forest achieved higher recall, F1-score, balanced accuracy, average precision and ROC-AUC.

Logistic Regression achieved higher accuracy and precision and generated fewer false alerts.

These results demonstrate that model performance depends on the evaluation metric and operating threshold.

## 4. Statistical Uncertainty Assessment

Paired pond-level bootstrap comparisons were performed using 2,000 resamples per region.

The Eluru comparison showed confidence intervals excluding zero for all seven evaluated metric differences, with Random Forest favoured in six metrics and Logistic Regression favoured in recall.

For Nellore, only the ROC-AUC difference had a confidence interval entirely above zero, favouring Random Forest. The differences in the other six metrics remained uncertain.

These intervals are conditional on the fitted models, selected thresholds and available test observations.

## 5. Important Research Limitations

1. The task predicts the status at the next recorded observation, not at a fixed future time horizon.
2. Observation intervals are irregular and vary across ponds.
3. Measurements are repeated within ponds and therefore cannot be treated as fully independent observations.
4. Eluru and Nellore have different chronological coverage and test sample sizes.
5. Water-quality classification practices and ammonia measurement protocols changed during the dataset period.
6. The initial models use only pH, DO and temperature.
7. Retrospective model performance does not establish real-world deployment readiness.

## 6. Final Conclusion

The experiments indicate that earlier pH, dissolved oxygen and temperature measurements contain predictive information about subsequent recorded water-quality status.

Random Forest is a promising candidate for further development, particularly because of its stronger ranking performance in both regional evaluations.

However, no universally superior model has been established across all metrics and regions. The trade-off between detecting actual water-quality problems and generating false alerts must be considered in future development.

Further interpretation, error analysis, robustness assessment and independent validation are necessary before operational deployment.

## 7. Transition to the Next Stage

The next stage will investigate the behaviour of the existing predictive models in greater detail.

Notebook 04 will focus on model interpretation and error analysis, including:

- False-positive and false-negative observations.
- Prediction errors across ponds.
- Morning and evening measurement behaviour.
- Effects of irregular observation intervals.
- Feature contribution and model interpretation.

The objective is to understand not only how well the models perform, but also where, why and under what conditions their predictions become unreliable.

In [68]:
# CELL 71: EXPORT MODELS AND TEST PREDICTIONS FOR NOTEBOOK 04

from pathlib import Path
import joblib
import json

# --------------------------------------------------
# 1. Create output directory
# --------------------------------------------------

artifact_directory = Path("../outputs/model_artifacts")
artifact_directory.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Define a helper to prepare regional predictions
# --------------------------------------------------

def prepare_error_analysis_data(
    region,
    test_data,
    actual_classes,
    probabilities,
    lr_threshold,
    rf_threshold
):

    predictors = [
        "pH",
        "DO (mg/L)",
        "Temp (in °C)"
    ]

    actual_classes = np.asarray(actual_classes, dtype=int)

    lr_prob = np.asarray(
        probabilities["Three-feature Logistic Regression"]
    )

    rf_prob = np.asarray(
        probabilities["Three-feature Random Forest"]
    )

    assert len(test_data) == len(actual_classes)
    assert len(test_data) == len(lr_prob) == len(rf_prob)

    # Verify actual target encoding.
    expected_classes = (
        test_data["Next water quality status"]
        .map({"Yes": 0, "No": 1})
        .to_numpy()
    )

    assert np.array_equal(actual_classes, expected_classes)

    result = pd.DataFrame({
        "Region": region,
        "pond_id": test_data["pond_id"].to_numpy(),
        "Actual status": test_data[
            "Next water quality status"
        ].to_numpy(),
        "Actual class": actual_classes
    })

    # Preserve observation timestamp when available.
    if "Observation timestamp" in test_data.columns:
        result["Observation timestamp"] = (
            test_data["Observation timestamp"].to_numpy()
        )

    for column in predictors:
        result[column] = test_data[column].to_numpy()

    # Logistic Regression predictions.
    result["LR probability"] = lr_prob
    result["LR threshold"] = lr_threshold

    result["LR predicted class"] = (
        lr_prob >= lr_threshold
    ).astype(int)

    # Random Forest predictions.
    result["RF probability"] = rf_prob
    result["RF threshold"] = rf_threshold

    result["RF predicted class"] = (
        rf_prob >= rf_threshold
    ).astype(int)

    # Identify correct and incorrect predictions.
    result["LR correct"] = (
        result["LR predicted class"] == actual_classes
    )

    result["RF correct"] = (
        result["RF predicted class"] == actual_classes
    )

    return result


# --------------------------------------------------
# 3. Prepare Eluru test predictions
# --------------------------------------------------

eluru_export = prepare_error_analysis_data(
    region="Eluru",
    test_data=test_shared,
    actual_classes=y_eluru_test,
    probabilities=eluru_test_probabilities,
    lr_threshold=0.2,
    rf_threshold=0.4
)

# --------------------------------------------------
# 4. Prepare Nellore test predictions
# --------------------------------------------------

nellore_export = prepare_error_analysis_data(
    region="Nellore",
    test_data=nellore_test,
    actual_classes=y_nellore_final_test,
    probabilities=final_test_probabilities,
    lr_threshold=0.4,
    rf_threshold=0.5
)

# --------------------------------------------------
# 5. Combine regional prediction records
# --------------------------------------------------

error_analysis_data = pd.concat(
    [eluru_export, nellore_export],
    ignore_index=True
)

# Save prediction records.
prediction_file = (
    artifact_directory
    / "regional_test_predictions_for_error_analysis.csv"
)

error_analysis_data.to_csv(
    prediction_file,
    index=False
)

# --------------------------------------------------
# 6. Save fitted model objects
# --------------------------------------------------

model_bundle = {
    "Eluru": {
        "Logistic Regression": eluru_lr_model,
        "Random Forest": eluru_rf_model
    },
    "Nellore": {
        "Logistic Regression": nellore_final_lr_three,
        "Random Forest": nellore_final_rf_three
    }
}

model_file = (
    artifact_directory
    / "regional_three_feature_models.joblib"
)

joblib.dump(model_bundle, model_file)

# --------------------------------------------------
# 7. Save model information
# --------------------------------------------------

manifest = {
    "project": "AQUA",
    "target": "Next recorded water-quality status",
    "target_encoding": {
        "0": "Water quality in range (Yes)",
        "1": "Water quality out of range (No)"
    },
    "predictors": [
        "pH",
        "DO (mg/L)",
        "Temp (in °C)"
    ],
    "thresholds": {
        "Eluru": {
            "Logistic Regression": 0.2,
            "Random Forest": 0.4
        },
        "Nellore": {
            "Logistic Regression": 0.4,
            "Random Forest": 0.5
        }
    },
    "prediction_records": len(error_analysis_data),
    "regions": ["Eluru", "Nellore"]
}

manifest_file = artifact_directory / "model_manifest.json"

with open(manifest_file, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=4)

# --------------------------------------------------
# 8. Display export confirmation
# --------------------------------------------------

export_summary = pd.DataFrame([
    {
        "Artifact": "Test prediction records",
        "Rows": len(error_analysis_data),
        "File": prediction_file.name
    },
    {
        "Artifact": "Fitted regional models",
        "Rows": 4,
        "File": model_file.name
    },
    {
        "Artifact": "Model manifest",
        "Rows": 1,
        "File": manifest_file.name
    }
])

display(export_summary)

print("Model and prediction export completed.")
print(f"Export directory: {artifact_directory.resolve()}")

,Artifact,Rows,File
0,Test prediction records,2805,regional_test_predictions_for_error_analysis.csv
1,Fitted regional models,4,regional_three_feature_models.joblib
2,Model manifest,1,model_manifest.json


Model and prediction export completed.
Export directory: C:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\model_artifacts


In [69]:
# CELL 72: INSPECT TIMESTAMP FIELDS IN REGIONAL TEST DATA

timestamp_inventory = []

for dataset_name, frame in [
    ("Eluru test data", test_shared),
    ("Nellore test data", nellore_test)
]:

    timestamp_columns = [
        column for column in frame.columns
        if any(
            term in column.lower()
            for term in ["timestamp", "date", "time"]
        )
    ]

    for column in timestamp_columns:

        timestamp_inventory.append({
            "Dataset": dataset_name,
            "Column": column,
            "Non-missing values": frame[column].notna().sum(),
            "Example value": frame[column].dropna().iloc[0]
                if frame[column].notna().any() else None
        })

display(pd.DataFrame(timestamp_inventory))

,Dataset,Column,Non-missing values,Example value
0,Eluru test data,Observation timestamp,2232,2025-11-18 08:10:00
1,Eluru test data,Next observation timestamp,2232,2025-11-18 15:31:00
2,Nellore test data,Date of data collection,573,02/05/2024
3,Nellore test data,Time of data collection,573,07:05
4,Nellore test data,Corrective actions implementation date,39,02/07/2024
5,Nellore test data,Self-initiated corrective actions implemented ...,0,None
6,Nellore test data,Self-initiated corrective actions implemented ...,0,None
7,Nellore test data,Parsed date,573,2024-02-05 00:00:00
8,Nellore test data,Next observation date,529,2024-03-04 00:00:00
9,Nellore test data,Observation timestamp,573,2024-02-05 07:05:00


In [70]:
# CELL 74: INSPECT PREDICTION EXPORT AND SOURCE TIMESTAMP KEYS, corrected cell of 73

from pathlib import Path
import pandas as pd

# Automatically locate the AQUA project root
current_dir = Path.cwd()

project_root = None

for folder in [current_dir, *current_dir.parents]:

    candidate_file = (
        folder
        / "outputs"
        / "model_artifacts"
        / "regional_test_predictions_for_error_analysis.csv"
    )

    if candidate_file.exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the AQUA prediction export. "
        "Please check the notebook's working directory."
    )

print("PROJECT ROOT:", project_root)

# Load the existing prediction export
prediction_file = (
    project_root
    / "outputs"
    / "model_artifacts"
    / "regional_test_predictions_for_error_analysis.csv"
)

predictions_export = pd.read_csv(prediction_file)

print("\nPREDICTION EXPORT")
print("Shape:", predictions_export.shape)

print("\nColumns:")
for column in predictions_export.columns:
    print(" -", column)

# Inspect source test datasets if they are already loaded
for dataset_name, variable_name in [
    ("Eluru test data", "test_shared"),
    ("Nellore test data", "nellore_test")
]:

    frame = globals().get(variable_name)

    print(f"\n{dataset_name.upper()}")

    if isinstance(frame, pd.DataFrame):

        print("Shape:", frame.shape)

        relevant_columns = [
            column for column in frame.columns
            if any(term in column.lower() for term in [
                "pond", "region", "timestamp",
                "date", "time", "range"
            ])
        ]

        print("Relevant columns:")
        for column in relevant_columns:
            print(" -", column)

    else:
        print(f"{variable_name} is not currently loaded.")

PROJECT ROOT: c:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026

PREDICTION EXPORT
Shape: (2805, 16)

Columns:
 - Region
 - pond_id
 - Actual status
 - Actual class
 - Observation timestamp
 - pH
 - DO (mg/L)
 - Temp (in °C)
 - LR probability
 - LR threshold
 - LR predicted class
 - RF probability
 - RF threshold
 - RF predicted class
 - LR correct
 - RF correct

ELURU TEST DATA
Shape: (2232, 9)
Relevant columns:
 - pond_id
 - Observation timestamp
 - Next observation timestamp

NELLORE TEST DATA
Shape: (573, 64)
Relevant columns:
 - pond_id
 - region
 - Date of data collection
 - Time of data collection
 - Pond status
 - Is WQ in range?
 - Parameters out of range
 - Corrective actions implementation date
 - Self-initiated corrective actions implemented on (exact date)
 - Self-initiated corrective actions implemented (date range)
 - Parsed date
 - Next observation date
 - Observation timestamp
 - Next observation timestamp
 - Timestamp duplicated within pond
 - Next timestamp duplicated
 - 

In [71]:
# CELL 75: ATTACH NEXT OBSERVATION TIMESTAMPS TO PREDICTIONS

import pandas as pd

# 1. Build the Eluru timestamp mapping
eluru_timestamp_map = test_shared[
    [
        "pond_id",
        "Observation timestamp",
        "Next observation timestamp"
    ]
].copy()

eluru_timestamp_map["Region"] = "Eluru"


# 2. Build the Nellore timestamp mapping
nellore_timestamp_map = nellore_test[
    [
        "pond_id",
        "Observation timestamp",
        "Next observation timestamp"
    ]
].copy()

nellore_timestamp_map["Region"] = "Nellore"


# 3. Combine both regional mappings
timestamp_map = pd.concat(
    [
        eluru_timestamp_map,
        nellore_timestamp_map
    ],
    ignore_index=True
)

# 4. Standardise matching keys
for frame in [timestamp_map, predictions_export]:

    frame["pond_id"] = (
        frame["pond_id"].astype("string").str.strip()
    )

    frame["Observation timestamp"] = pd.to_datetime(
        frame["Observation timestamp"],
        errors="coerce"
    )

    frame["Region"] = frame["Region"].astype("string").str.strip()


timestamp_map["Next observation timestamp"] = pd.to_datetime(
    timestamp_map["Next observation timestamp"],
    errors="coerce"
)


# 5. Validate the timestamp mapping before merging
mapping_keys = [
    "Region",
    "pond_id",
    "Observation timestamp"
]

duplicate_mapping_keys = timestamp_map[
    timestamp_map.duplicated(
        subset=mapping_keys,
        keep=False
    )
]

if not duplicate_mapping_keys.empty:
    display(duplicate_mapping_keys.head(10))
    raise ValueError(
        "Duplicate timestamp mapping keys found. "
        "Do not continue until these are investigated."
    )

if timestamp_map["Next observation timestamp"].isna().any():
    raise ValueError(
        "Some source records have missing next observation timestamps."
    )


# 6. Match future timestamps to the prediction export
predictions_with_target_timestamp = predictions_export.merge(
    timestamp_map[
        mapping_keys + ["Next observation timestamp"]
    ],
    on=mapping_keys,
    how="left",
    validate="many_to_one",
    indicator=True
)


# 7. Verify every prediction was matched
unmatched = predictions_with_target_timestamp[
    predictions_with_target_timestamp["_merge"] != "both"
]

if not unmatched.empty:
    print("UNMATCHED PREDICTION RECORDS:", len(unmatched))
    display(unmatched[mapping_keys].head(10))

    raise ValueError(
        "Some prediction records could not be matched. "
        "The new artifact has not been saved."
    )

predictions_with_target_timestamp = (
    predictions_with_target_timestamp.drop(columns="_merge")
)


# 8. Calculate the actual interval between observations
predictions_with_target_timestamp["Observation interval (hours)"] = (
    (
        predictions_with_target_timestamp["Next observation timestamp"]
        - predictions_with_target_timestamp["Observation timestamp"]
    ).dt.total_seconds() / 3600
)

if (
    predictions_with_target_timestamp["Observation interval (hours)"]
    .le(0).any()
):
    raise ValueError(
        "Non-positive observation intervals detected. "
        "Investigate these records before proceeding."
    )


# 9. Save as a separate artifact
output_file = (
    project_root
    / "outputs"
    / "model_artifacts"
    / "regional_test_predictions_with_target_timestamps.csv"
)

predictions_with_target_timestamp.to_csv(
    output_file,
    index=False
)


# 10. Display validation summary
timestamp_validation = (
    predictions_with_target_timestamp
    .groupby("Region")
    .agg(
        Prediction_records=("pond_id", "size"),
        Unique_ponds=("pond_id", "nunique"),
        Median_interval_hours=(
            "Observation interval (hours)", "median"
        ),
        Minimum_interval_hours=(
            "Observation interval (hours)", "min"
        ),
        Maximum_interval_hours=(
            "Observation interval (hours)", "max"
        )
    )
    .round(2)
    .reset_index()
)

print("TIMESTAMP MATCHING VALIDATION")
print("Original prediction records:", len(predictions_export))
print("Successfully matched records:", len(predictions_with_target_timestamp))
print("Unmatched records:", len(unmatched))
print("\nRegional observation intervals:")

display(timestamp_validation)

print("\nNEW ARTIFACT SAVED:")
print(output_file)

TIMESTAMP MATCHING VALIDATION
Original prediction records: 2805
Successfully matched records: 2805
Unmatched records: 0

Regional observation intervals:


,Region,Prediction_records,Unique_ponds,Median_interval_hours,Minimum_interval_hours,Maximum_interval_hours
0,Eluru,2232,118,13.92,0.15,5751.50
1,Nellore,573,49,10.45,0.15,6734.75



NEW ARTIFACT SAVED:
c:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\model_artifacts\regional_test_predictions_with_target_timestamps.csv


In [72]:
# CELL 76: MODEL PERFORMANCE BY NEXT OBSERVATION TIME

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Work with the timestamp-enriched prediction dataset
target_time_data = predictions_with_target_timestamp.copy()

# Extract the hour of the future observation
target_time_data["Target observation hour"] = (
    target_time_data["Next observation timestamp"].dt.hour
)

# Group observations by the time of the future measurement
target_time_data["Target measurement period"] = np.select(
    [
        target_time_data["Target observation hour"].between(6, 8),
        target_time_data["Target observation hour"].between(15, 17)
    ],
    [
        "Morning (06:00–08:59)",
        "Later day (15:00–17:59)"
    ],
    default="Other hours"
)

period_order = [
    "Morning (06:00–08:59)",
    "Later day (15:00–17:59)",
    "Other hours"
]

target_time_data["Target measurement period"] = pd.Categorical(
    target_time_data["Target measurement period"],
    categories=period_order,
    ordered=True
)

# Define the two model prediction columns
model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

summary_rows = []

# Analyse each region and future measurement period
for (region, period), group in target_time_data.groupby(
    ["Region", "Target measurement period"],
    observed=True,
    sort=True
):

    actual = group["Actual class"].astype(int)

    # Preserve the exact distribution of future observation hours
    hour_distribution = (
        group["Target observation hour"]
        .value_counts()
        .sort_index()
    )

    hour_profile = ", ".join(
        f"{int(hour):02d}:00 = {count}"
        for hour, count in hour_distribution.items()
    )

    intervals = group["Observation interval (hours)"]

    for model_name, prediction_column in model_columns.items():

        predicted = group[prediction_column].astype(int)

        tn, fp, fn, tp = confusion_matrix(
            actual,
            predicted,
            labels=[0, 1]
        ).ravel()

        summary_rows.append({

            "Region": region,
            "Model": model_name,
            "Target measurement period": str(period),

            "Observations": len(group),
            "Unique ponds": group["pond_id"].nunique(),

            "Target hour distribution": hour_profile,

            "Actual out-of-range": int(actual.sum()),
            "Detected problems": int(tp),
            "Missed problems": int(fn),

            "False alerts": int(fp),
            "Correct in-range predictions": int(tn),

            "Recall (%)": (
                tp / (tp + fn) * 100
                if tp + fn else np.nan
            ),

            "False positive rate (%)": (
                fp / (fp + tn) * 100
                if fp + tn else np.nan
            ),

            "Alert rate (%)": predicted.mean() * 100,

            "Median interval (hours)": intervals.median(),
            "Q25 interval (hours)": intervals.quantile(0.25),
            "Q75 interval (hours)": intervals.quantile(0.75)

        })

# Create the consolidated results table
target_time_performance = pd.DataFrame(summary_rows)

percentage_columns = [
    "Recall (%)",
    "False positive rate (%)",
    "Alert rate (%)"
]

interval_columns = [
    "Median interval (hours)",
    "Q25 interval (hours)",
    "Q75 interval (hours)"
]

target_time_performance[percentage_columns] = (
    target_time_performance[percentage_columns].round(2)
)

target_time_performance[interval_columns] = (
    target_time_performance[interval_columns].round(2)
)

print("MODEL PERFORMANCE BY FUTURE OBSERVATION TIME")
print("Grouping variable: Next observation timestamp")
print("Total prediction records:", len(target_time_data))

display(
    target_time_performance.sort_values(
        ["Region", "Target measurement period", "Model"]
    ).reset_index(drop=True)
)

MODEL PERFORMANCE BY FUTURE OBSERVATION TIME
Grouping variable: Next observation timestamp
Total prediction records: 2805


,Region,Model,Target measurement period,Observations,Unique ponds,Target hour distribution,Actual out-of-range,Detected problems,Missed problems,False alerts,Correct in-range predictions,Recall (%),False positive rate (%),Alert rate (%),Median interval (hours),Q25 interval (hours),Q75 interval (hours)
0,Eluru,Logistic Regression,Later day (15:00–17:59),1114,118,"15:00 = 286, 16:00 = 805, 17:00 = 23",147,134,13,619,348,91.16,64.01,67.59,8.68,8.14,9.05
1,Eluru,Random Forest,Later day (15:00–17:59),1114,118,"15:00 = 286, 16:00 = 805, 17:00 = 23",147,101,46,105,862,68.71,10.86,18.49,8.68,8.14,9.05
2,Eluru,Logistic Regression,Morning (06:00–08:59),1117,113,"06:00 = 79, 07:00 = 886, 08:00 = 152",165,57,108,243,709,34.55,25.53,26.86,399.23,279.00,688.40
3,Eluru,Random Forest,Morning (06:00–08:59),1117,113,"06:00 = 79, 07:00 = 886, 08:00 = 152",165,62,103,246,706,37.58,25.84,27.57,399.23,279.00,688.40
4,Eluru,Logistic Regression,Other hours,1,1,04:00 = 1,0,0,0,1,0,NaN,100.00,100.00,525.38,525.38,525.38
5,Eluru,Random Forest,Other hours,1,1,04:00 = 1,0,0,0,0,1,NaN,0.00,0.00,525.38,525.38,525.38
6,Nellore,Logistic Regression,Later day (15:00–17:59),309,49,"15:00 = 10, 16:00 = 176, 17:00 = 123",22,14,8,9,278,63.64,3.14,7.44,10.00,9.73,10.22
7,Nellore,Random Forest,Later day (15:00–17:59),309,49,"15:00 = 10, 16:00 = 176, 17:00 = 123",22,21,1,17,270,95.45,5.92,12.30,10.00,9.73,10.22
8,Nellore,Logistic Regression,Morning (06:00–08:59),264,47,"06:00 = 138, 07:00 = 126",22,3,19,1,241,13.64,0.41,1.52,709.73,613.58,781.72
9,Nellore,Random Forest,Morning (06:00–08:59),264,47,"06:00 = 138, 07:00 = 126",22,0,22,4,238,0.00,1.65,1.52,709.73,613.58,781.72


In [73]:
# CELL 77: DISTRIBUTION OF OBSERVATION INTERVALS BY FUTURE MEASUREMENT PERIOD

interval_analysis = predictions_with_target_timestamp.copy()

# Classify the future observation time
interval_analysis["Target observation hour"] = (
    interval_analysis["Next observation timestamp"].dt.hour
)

interval_analysis["Target measurement period"] = np.select(
    [
        interval_analysis["Target observation hour"].between(6, 8),
        interval_analysis["Target observation hour"].between(15, 17)
    ],
    [
        "Morning (06:00–08:59)",
        "Later day (15:00–17:59)"
    ],
    default="Other hours"
)

# Classify the time gap between observations
interval_analysis["Interval category"] = pd.cut(
    interval_analysis["Observation interval (hours)"],
    bins=[0, 24, 168, np.inf],
    labels=[
        "Short (<=24 hours)",
        "Medium (>24 hours–7 days)",
        "Long (>7 days)"
    ],
    include_lowest=True
)

# Summarise interval distributions and target prevalence
interval_summary = (
    interval_analysis
    .groupby(
        ["Region", "Target measurement period", "Interval category"],
        observed=True
    )
    .agg(
        Observations=("pond_id", "size"),
        Unique_ponds=("pond_id", "nunique"),
        Actual_out_of_range=("Actual class", "sum"),
        Median_interval_hours=(
            "Observation interval (hours)", "median"
        ),
        Q25_interval_hours=(
            "Observation interval (hours)",
            lambda x: x.quantile(0.25)
        ),
        Q75_interval_hours=(
            "Observation interval (hours)",
            lambda x: x.quantile(0.75)
        )
    )
    .reset_index()
)

interval_summary["Out_of_range_rate (%)"] = (
    interval_summary["Actual_out_of_range"]
    / interval_summary["Observations"] * 100
)

numeric_columns = [
    "Median_interval_hours",
    "Q25_interval_hours",
    "Q75_interval_hours",
    "Out_of_range_rate (%)"
]

interval_summary[numeric_columns] = (
    interval_summary[numeric_columns].round(2)
)

print("OBSERVATION INTERVAL DISTRIBUTION")
print("Total prediction records:", len(interval_analysis))

display(
    interval_summary.sort_values(
        ["Region", "Target measurement period", "Interval category"]
    ).reset_index(drop=True)
)

OBSERVATION INTERVAL DISTRIBUTION
Total prediction records: 2805


,Region,Target measurement period,Interval category,Observations,Unique_ponds,Actual_out_of_range,Median_interval_hours,Q25_interval_hours,Q75_interval_hours,Out_of_range_rate (%)
0,Eluru,Later day (15:00–17:59),Short (<=24 hours),1114,118,147,8.68,8.14,9.05,13.20
1,Eluru,Morning (06:00–08:59),Short (<=24 hours),58,32,21,15.11,14.92,15.43,36.21
2,Eluru,Morning (06:00–08:59),Medium (>24 hours–7 days),130,69,43,40.17,39.16,72.50,33.08
3,Eluru,Morning (06:00–08:59),Long (>7 days),929,112,101,494.32,328.05,759.28,10.87
4,Eluru,Other hours,Long (>7 days),1,1,0,525.38,525.38,525.38,0.00
5,Nellore,Later day (15:00–17:59),Short (<=24 hours),309,49,22,10.00,9.73,10.22,7.12
6,Nellore,Morning (06:00–08:59),Medium (>24 hours–7 days),29,21,3,39.17,37.78,62.27,10.34
7,Nellore,Morning (06:00–08:59),Long (>7 days),235,46,19,710.30,661.91,805.64,8.09


In [74]:
# CELL 78: MODEL PERFORMANCE STRATIFIED BY OBSERVATION INTERVAL

interval_performance_rows = []

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

for (region, period, interval_category), group in interval_analysis.groupby(
    ["Region", "Target measurement period", "Interval category"],
    observed=True,
    sort=True
):

    actual = group["Actual class"].astype(int)

    for model_name, prediction_column in model_columns.items():

        predicted = group[prediction_column].astype(int)

        tn, fp, fn, tp = confusion_matrix(
            actual,
            predicted,
            labels=[0, 1]
        ).ravel()

        interval_performance_rows.append({

            "Region": region,
            "Target measurement period": str(period),
            "Interval category": str(interval_category),
            "Model": model_name,

            "Observations": len(group),
            "Unique ponds": group["pond_id"].nunique(),

            "Actual out-of-range": int(actual.sum()),
            "Detected problems": int(tp),
            "Missed problems": int(fn),

            "False alerts": int(fp),
            "Correct in-range predictions": int(tn),

            "Recall (%)": (
                tp / (tp + fn) * 100
                if (tp + fn) else np.nan
            ),

            "False positive rate (%)": (
                fp / (fp + tn) * 100
                if (fp + tn) else np.nan
            ),

            "Alert rate (%)": predicted.mean() * 100,

            "Median interval (hours)": (
                group["Observation interval (hours)"].median()
            )

        })

interval_performance = pd.DataFrame(interval_performance_rows)

percentage_columns = [
    "Recall (%)",
    "False positive rate (%)",
    "Alert rate (%)"
]

interval_performance[percentage_columns] = (
    interval_performance[percentage_columns].round(2)
)

interval_performance["Median interval (hours)"] = (
    interval_performance["Median interval (hours)"].round(2)
)

print("MODEL PERFORMANCE WITHIN OBSERVATION-INTERVAL CATEGORIES")

display(
    interval_performance.sort_values(
        ["Region", "Interval category", "Target measurement period", "Model"]
    ).reset_index(drop=True)
)

MODEL PERFORMANCE WITHIN OBSERVATION-INTERVAL CATEGORIES


,Region,Target measurement period,Interval category,Model,Observations,Unique ponds,Actual out-of-range,Detected problems,Missed problems,False alerts,Correct in-range predictions,Recall (%),False positive rate (%),Alert rate (%),Median interval (hours)
0,Eluru,Morning (06:00–08:59),Long (>7 days),Logistic Regression,929,112,101,23,78,182,646,22.77,21.98,22.07,494.32
1,Eluru,Morning (06:00–08:59),Long (>7 days),Random Forest,929,112,101,25,76,172,656,24.75,20.77,21.21,494.32
2,Eluru,Other hours,Long (>7 days),Logistic Regression,1,1,0,0,0,1,0,NaN,100.00,100.00,525.38
3,Eluru,Other hours,Long (>7 days),Random Forest,1,1,0,0,0,0,1,NaN,0.00,0.00,525.38
4,Eluru,Morning (06:00–08:59),Medium (>24 hours–7 days),Logistic Regression,130,69,43,20,23,47,40,46.51,54.02,51.54,40.17
5,Eluru,Morning (06:00–08:59),Medium (>24 hours–7 days),Random Forest,130,69,43,23,20,59,28,53.49,67.82,63.08,40.17
6,Eluru,Later day (15:00–17:59),Short (<=24 hours),Logistic Regression,1114,118,147,134,13,619,348,91.16,64.01,67.59,8.68
7,Eluru,Later day (15:00–17:59),Short (<=24 hours),Random Forest,1114,118,147,101,46,105,862,68.71,10.86,18.49,8.68
8,Eluru,Morning (06:00–08:59),Short (<=24 hours),Logistic Regression,58,32,21,14,7,14,23,66.67,37.84,48.28,15.11
9,Eluru,Morning (06:00–08:59),Short (<=24 hours),Random Forest,58,32,21,14,7,15,22,66.67,40.54,50.00,15.11


In [75]:
# CELL 79: CHECK OVERLAP IN ELAPSED-TIME DISTRIBUTIONS

overlap_data = interval_analysis[
    interval_analysis["Target measurement period"].isin([
        "Morning (06:00–08:59)",
        "Later day (15:00–17:59)"
    ])
].copy()

periods = [
    "Morning (06:00–08:59)",
    "Later day (15:00–17:59)"
]

distribution_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = overlap_data[
        overlap_data["Region"] == region
    ]

    period_groups = {
        period: region_data[
            region_data["Target measurement period"] == period
        ]
        for period in periods
    }

    # Calculate the common interval support
    morning = period_groups[periods[0]]
    later_day = period_groups[periods[1]]

    if morning.empty or later_day.empty:

        common_lower = np.nan
        common_upper = np.nan

    else:

        common_lower = max(
            morning["Observation interval (hours)"].min(),
            later_day["Observation interval (hours)"].min()
        )

        common_upper = min(
            morning["Observation interval (hours)"].max(),
            later_day["Observation interval (hours)"].max()
        )

        if common_lower > common_upper:
            common_lower = np.nan
            common_upper = np.nan

    # Summarise both periods
    for period in periods:

        group = period_groups[period]

        if group.empty:
            continue

        intervals = group["Observation interval (hours)"]

        if pd.isna(common_lower):
            common_group = group.iloc[0:0]
        else:
            common_group = group[
                intervals.between(common_lower, common_upper)
            ]

        distribution_rows.append({

            "Region": region,
            "Target measurement period": period,

            "Observations": len(group),
            "Unique ponds": group["pond_id"].nunique(),

            "Minimum interval (hours)": intervals.min(),
            "5th percentile (hours)": intervals.quantile(0.05),
            "Q25 interval (hours)": intervals.quantile(0.25),
            "Median interval (hours)": intervals.median(),
            "Q75 interval (hours)": intervals.quantile(0.75),
            "95th percentile (hours)": intervals.quantile(0.95),
            "Maximum interval (hours)": intervals.max(),

            "Common interval lower bound (hours)": common_lower,
            "Common interval upper bound (hours)": common_upper,

            "Observations within common interval": len(common_group),
            "Actual out-of-range within common interval": int(
                common_group["Actual class"].sum()
            )

        })

interval_overlap_summary = pd.DataFrame(distribution_rows)

numeric_columns = [
    column for column in interval_overlap_summary.columns
    if "hours)" in column
]

interval_overlap_summary[numeric_columns] = (
    interval_overlap_summary[numeric_columns].round(2)
)

print("ELAPSED-TIME DISTRIBUTION AND COMMON SUPPORT")

display(interval_overlap_summary)

ELAPSED-TIME DISTRIBUTION AND COMMON SUPPORT


,Region,Target measurement period,Observations,Unique ponds,Minimum interval (hours),5th percentile (hours),Q25 interval (hours),Median interval (hours),Q75 interval (hours),95th percentile (hours),Maximum interval (hours),Common interval lower bound (hours),Common interval upper bound (hours),Observations within common interval,Actual out-of-range within common interval
0,Eluru,Morning (06:00–08:59),1117,113,2.17,16.37,279.00,399.23,688.40,1095.38,5751.50,2.17,10.18,1,0
1,Eluru,Later day (15:00–17:59),1114,118,0.15,7.74,8.14,8.68,9.05,9.43,10.18,2.17,10.18,1113,147
2,Nellore,Morning (06:00–08:59),264,47,37.05,39.12,613.58,709.73,781.72,1858.20,6734.75,NaN,NaN,0,0
3,Nellore,Later day (15:00–17:59),309,49,0.15,8.82,9.73,10.00,10.22,10.49,11.20,NaN,NaN,0,0


In [76]:
# CELL 80: CONSOLIDATED MODEL PERFORMANCE BY OBSERVATION INTERVAL

pooled_interval_data = interval_analysis.copy()

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

pooled_interval_rows = []

interval_order = [
    "Short (<=24 hours)",
    "Medium (>24 hours–7 days)",
    "Long (>7 days)"
]

for region in ["Eluru", "Nellore"]:

    region_data = pooled_interval_data[
        pooled_interval_data["Region"] == region
    ]

    for interval_category in interval_order:

        group = region_data[
            region_data["Interval category"].astype(str)
            == interval_category
        ]

        if group.empty:
            continue

        actual = group["Actual class"].astype(int)

        # Describe the mixture of future measurement times
        morning_count = (
            group["Target measurement period"]
            == "Morning (06:00–08:59)"
        ).sum()

        later_day_count = (
            group["Target measurement period"]
            == "Later day (15:00–17:59)"
        ).sum()

        other_hours_count = len(group) - morning_count - later_day_count

        intervals = group["Observation interval (hours)"]

        for model_name, prediction_column in model_columns.items():

            predicted = group[prediction_column].astype(int)

            tn, fp, fn, tp = confusion_matrix(
                actual,
                predicted,
                labels=[0, 1]
            ).ravel()

            precision = (
                tp / (tp + fp)
                if (tp + fp) else np.nan
            )

            recall = (
                tp / (tp + fn)
                if (tp + fn) else np.nan
            )

            specificity = (
                tn / (tn + fp)
                if (tn + fp) else np.nan
            )

            f1 = (
                2 * precision * recall / (precision + recall)
                if (
                    pd.notna(precision)
                    and pd.notna(recall)
                    and (precision + recall) > 0
                )
                else np.nan
            )

            balanced_accuracy = (
                (recall + specificity) / 2
                if pd.notna(recall) and pd.notna(specificity)
                else np.nan
            )

            pooled_interval_rows.append({

                "Region": region,
                "Interval category": interval_category,
                "Model": model_name,

                "Observations": len(group),
                "Unique ponds": group["pond_id"].nunique(),

                "Morning observations": int(morning_count),
                "Later-day observations": int(later_day_count),
                "Other-hour observations": int(other_hours_count),

                "Actual out-of-range": int(actual.sum()),
                "Detected problems": int(tp),
                "Missed problems": int(fn),

                "False alerts": int(fp),
                "Correct in-range predictions": int(tn),

                "Precision (%)": precision * 100,
                "Recall (%)": recall * 100,
                "Specificity (%)": specificity * 100,
                "F1 score": f1,
                "Balanced accuracy (%)": (
                    balanced_accuracy * 100
                    if pd.notna(balanced_accuracy)
                    else np.nan
                ),

                "False positive rate (%)": (
                    (1 - specificity) * 100
                    if pd.notna(specificity)
                    else np.nan
                ),

                "Alert rate (%)": predicted.mean() * 100,

                "Median interval (hours)": intervals.median(),
                "Q25 interval (hours)": intervals.quantile(0.25),
                "Q75 interval (hours)": intervals.quantile(0.75)

            })


pooled_interval_performance = pd.DataFrame(
    pooled_interval_rows
)

percentage_columns = [
    "Precision (%)",
    "Recall (%)",
    "Specificity (%)",
    "Balanced accuracy (%)",
    "False positive rate (%)",
    "Alert rate (%)"
]

pooled_interval_performance[percentage_columns] = (
    pooled_interval_performance[percentage_columns].round(2)
)

pooled_interval_performance["F1 score"] = (
    pooled_interval_performance["F1 score"].round(4)
)

interval_columns = [
    "Median interval (hours)",
    "Q25 interval (hours)",
    "Q75 interval (hours)"
]

pooled_interval_performance[interval_columns] = (
    pooled_interval_performance[interval_columns].round(2)
)

print("CONSOLIDATED MODEL PERFORMANCE BY OBSERVATION INTERVAL")
print("Total prediction records:", len(pooled_interval_data))

display(
    pooled_interval_performance.sort_values(
        ["Region", "Interval category", "Model"]
    ).reset_index(drop=True)
)

CONSOLIDATED MODEL PERFORMANCE BY OBSERVATION INTERVAL
Total prediction records: 2805


,Region,Interval category,Model,Observations,Unique ponds,Morning observations,Later-day observations,Other-hour observations,Actual out-of-range,Detected problems,...,Precision (%),Recall (%),Specificity (%),F1 score,Balanced accuracy (%),False positive rate (%),Alert rate (%),Median interval (hours),Q25 interval (hours),Q75 interval (hours)
0,Eluru,Long (>7 days),Logistic Regression,930,112,929,0,1,101,23,...,11.17,22.77,77.93,0.1498,50.35,22.07,22.15,494.43,328.06,759.25
1,Eluru,Long (>7 days),Random Forest,930,112,929,0,1,101,25,...,12.69,24.75,79.25,0.1678,52.00,20.75,21.18,494.43,328.06,759.25
2,Eluru,Medium (>24 hours–7 days),Logistic Regression,130,69,130,0,0,43,20,...,29.85,46.51,45.98,0.3636,46.24,54.02,51.54,40.17,39.16,72.50
3,Eluru,Medium (>24 hours–7 days),Random Forest,130,69,130,0,0,43,23,...,28.05,53.49,32.18,0.3680,42.84,67.82,63.08,40.17,39.16,72.50
4,Eluru,Short (<=24 hours),Logistic Regression,1172,118,58,1114,0,168,148,...,18.95,88.10,36.95,0.3119,62.52,63.05,66.64,8.75,8.17,9.08
5,Eluru,Short (<=24 hours),Random Forest,1172,118,58,1114,0,168,115,...,48.94,68.45,88.05,0.5707,78.25,11.95,20.05,8.75,8.17,9.08
6,Nellore,Long (>7 days),Logistic Regression,235,46,235,0,0,19,1,...,50.00,5.26,99.54,0.0952,52.40,0.46,0.85,710.30,661.91,805.64
7,Nellore,Long (>7 days),Random Forest,235,46,235,0,0,19,0,...,NaN,0.00,100.00,NaN,50.00,0.00,0.00,710.30,661.91,805.64
8,Nellore,Medium (>24 hours–7 days),Logistic Regression,29,21,29,0,0,3,2,...,100.00,66.67,100.00,0.8000,83.33,0.00,6.90,39.17,37.78,62.27
9,Nellore,Medium (>24 hours–7 days),Random Forest,29,21,29,0,0,3,0,...,0.00,0.00,84.62,NaN,42.31,15.38,13.79,39.17,37.78,62.27


In [ ]:
# CELL 82: OPTIMISED POND-CLUSTERED BOOTSTRAP BY INTERVAL

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(42)
N_BOOTSTRAP = 2000

model_prefixes = {
    "Logistic Regression": "LR",
    "Random Forest": "RF"
}

metrics_to_compare = [
    "Recall",
    "F1",
    "Balanced accuracy"
]


def calculate_metric_arrays(confusion_totals, prefix):

    tp = confusion_totals[:, 0]
    fp = confusion_totals[:, 1]
    tn = confusion_totals[:, 2]
    fn = confusion_totals[:, 3]

    recall = np.divide(
        tp, tp + fn,
        out=np.full(len(tp), np.nan),
        where=(tp + fn) > 0
    )

    specificity = np.divide(
        tn, tn + fp,
        out=np.full(len(tp), np.nan),
        where=(tn + fp) > 0
    )

    f1 = np.divide(
        2 * tp,
        2 * tp + fp + fn,
        out=np.full(len(tp), np.nan),
        where=(2 * tp + fp + fn) > 0
    )

    balanced_accuracy = (
        (recall + specificity) / 2
    )

    return {
        "Recall": recall,
        "F1": f1,
        "Balanced accuracy": balanced_accuracy
    }


bootstrap_rows = []

interval_order = [
    "Short (<=24 hours)",
    "Medium (>24 hours–7 days)",
    "Long (>7 days)"
]

for region in ["Eluru", "Nellore"]:

    for interval_category in interval_order:

        group = interval_analysis[
            (interval_analysis["Region"] == region)
            &
            (
                interval_analysis["Interval category"].astype(str)
                == interval_category
            )
        ].copy()

        if group.empty:
            continue

        actual = group["Actual class"].to_numpy(dtype=int)

        # Calculate each pond's confusion counts only once
        pond_level = pd.DataFrame({
            "pond_id": group["pond_id"].to_numpy()
        })

        for model_name, prefix in model_prefixes.items():

            predicted = group[
                f"{prefix} predicted class"
            ].to_numpy(dtype=int)

            pond_level[f"{prefix}_TP"] = (
                (actual == 1) & (predicted == 1)
            ).astype(int)

            pond_level[f"{prefix}_FP"] = (
                (actual == 0) & (predicted == 1)
            ).astype(int)

            pond_level[f"{prefix}_TN"] = (
                (actual == 0) & (predicted == 0)
            ).astype(int)

            pond_level[f"{prefix}_FN"] = (
                (actual == 1) & (predicted == 0)
            ).astype(int)

        count_columns = [
            "LR_TP", "LR_FP", "LR_TN", "LR_FN",
            "RF_TP", "RF_FP", "RF_TN", "RF_FN"
        ]

        pond_counts = (
            pond_level
            .groupby("pond_id", sort=False)[count_columns]
            .sum()
        )

        pond_confusion = pond_counts.to_numpy(dtype=float)
        n_ponds = len(pond_confusion)

        # Draw bootstrap samples of pond indices
        sampled_indices = rng.integers(
            0,
            n_ponds,
            size=(N_BOOTSTRAP, n_ponds)
        )

        # Convert each draw into pond-sampling frequencies
        sample_counts = np.stack([
            np.bincount(row, minlength=n_ponds)
            for row in sampled_indices
        ])

        # Aggregate confusion counts using matrix multiplication
        bootstrap_totals = sample_counts @ pond_confusion

        original_totals = pond_confusion.sum(axis=0).reshape(1, -1)

        # Calculate metrics for both models
        lr_bootstrap_metrics = calculate_metric_arrays(
            bootstrap_totals[:, :4], "LR"
        )

        rf_bootstrap_metrics = calculate_metric_arrays(
            bootstrap_totals[:, 4:], "RF"
        )

        lr_original_metrics = calculate_metric_arrays(
            original_totals[:, :4], "LR"
        )

        rf_original_metrics = calculate_metric_arrays(
            original_totals[:, 4:], "RF"
        )

        for metric in metrics_to_compare:

            differences = (
                rf_bootstrap_metrics[metric]
                - lr_bootstrap_metrics[metric]
            )

            valid_differences = differences[
                np.isfinite(differences)
            ]

            if len(valid_differences) > 0:

                lower_ci, upper_ci = np.quantile(
                    valid_differences,
                    [0.025, 0.975]
                )

            else:

                lower_ci = np.nan
                upper_ci = np.nan

            original_difference = (
                rf_original_metrics[metric][0]
                - lr_original_metrics[metric][0]
            )

            bootstrap_rows.append({

                "Region": region,
                "Interval category": interval_category,
                "Metric": metric,

                "Unique ponds": n_ponds,

                "Random Forest minus Logistic Regression": (
                    original_difference * 100
                ),

                "95% CI lower (percentage points)": (
                    lower_ci * 100
                ),

                "95% CI upper (percentage points)": (
                    upper_ci * 100
                ),

                "Valid bootstrap replicates": len(valid_differences)

            })


interval_bootstrap_comparison = pd.DataFrame(bootstrap_rows)

numeric_columns = [
    "Random Forest minus Logistic Regression",
    "95% CI lower (percentage points)",
    "95% CI upper (percentage points)"
]

interval_bootstrap_comparison[numeric_columns] = (
    interval_bootstrap_comparison[numeric_columns].round(2)
)

print("OPTIMISED POND-CLUSTERED BOOTSTRAP")
print("Bootstrap repetitions:", N_BOOTSTRAP)

display(interval_bootstrap_comparison)

OPTIMISED POND-CLUSTERED BOOTSTRAP
Bootstrap repetitions: 2000


,Region,Interval category,Metric,Unique ponds,Random Forest minus Logistic Regression,95% CI lower (percentage points),95% CI upper (percentage points),Valid bootstrap replicates
0,Eluru,Short (<=24 hours),Recall,118,-19.64,-29.89,-11.18,2000
1,Eluru,Short (<=24 hours),F1,118,25.88,18.92,32.05,2000
2,Eluru,Short (<=24 hours),Balanced accuracy,118,15.73,9.96,20.68,2000
3,Eluru,Medium (>24 hours–7 days),Recall,69,6.98,-15.00,30.00,2000
4,Eluru,Medium (>24 hours–7 days),F1,69,0.44,-12.96,15.08,2000
5,Eluru,Medium (>24 hours–7 days),Balanced accuracy,69,-3.41,-16.34,10.03,2000
6,Eluru,Long (>7 days),Recall,112,1.98,-8.85,13.21,2000
7,Eluru,Long (>7 days),F1,112,1.79,-5.02,8.68,2000
8,Eluru,Long (>7 days),Balanced accuracy,112,1.65,-4.08,7.57,2000
9,Nellore,Short (<=24 hours),Recall,49,31.82,13.33,53.85,2000


In [ ]:
# CELL 83: EXPORT FINAL OBSERVATION-INTERVAL ANALYSIS

from pathlib import Path

tables_dir = project_root / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

# Save descriptive model performance
performance_file = (
    tables_dir / "observation_interval_model_performance.csv"
)

pooled_interval_performance.to_csv(
    performance_file,
    index=False
)

# Save pond-clustered bootstrap comparisons
bootstrap_file = (
    tables_dir / "observation_interval_bootstrap_comparison.csv"
)

interval_bootstrap_comparison.to_csv(
    bootstrap_file,
    index=False
)

# Verify exported artifacts
export_validation = pd.DataFrame([
    {
        "Artifact": performance_file.name,
        "Rows exported": len(
            pd.read_csv(performance_file)
        ),
        "File exists": performance_file.exists()
    },
    {
        "Artifact": bootstrap_file.name,
        "Rows exported": len(
            pd.read_csv(bootstrap_file)
        ),
        "File exists": bootstrap_file.exists()
    }
])

print("FINAL INTERVAL-ANALYSIS EXPORT VALIDATION")

display(export_validation)

print("\nSAVED FILES:")
print(performance_file)
print(bootstrap_file)

FINAL INTERVAL-ANALYSIS EXPORT VALIDATION


,Artifact,Rows exported,File exists
0,observation_interval_model_performance.csv,12,True
1,observation_interval_bootstrap_comparison.csv,18,True



SAVED FILES:
c:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\tables\observation_interval_model_performance.csv
c:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\tables\observation_interval_bootstrap_comparison.csv


In [ ]:
# CELL 84: CORRECT F1 SCORES FOR INTERVAL-BASED MODEL PERFORMANCE

corrected_performance = pooled_interval_performance.copy()

tp = corrected_performance["Detected problems"]
fp = corrected_performance["False alerts"]
fn = corrected_performance["Missed problems"]

f1_denominator = 2 * tp + fp + fn

corrected_performance["F1 score"] = np.divide(
    2 * tp,
    f1_denominator,
    out=np.full(len(corrected_performance), np.nan),
    where=f1_denominator > 0
)

corrected_performance["F1 score"] = (
    corrected_performance["F1 score"].round(4)
)

# Replace the existing in-memory table with the corrected version
pooled_interval_performance = corrected_performance

# Re-export the corrected performance artifact
performance_file = (
    project_root
    / "outputs"
    / "tables"
    / "observation_interval_model_performance.csv"
)

pooled_interval_performance.to_csv(
    performance_file,
    index=False
)

print("CORRECTED F1-SCORE VALIDATION")

display(
    pooled_interval_performance[
        pooled_interval_performance["Region"] == "Nellore"
    ][
        [
            "Interval category",
            "Model",
            "Actual out-of-range",
            "Detected problems",
            "Missed problems",
            "F1 score"
        ]
    ].reset_index(drop=True)
)

print("\nCorrected artifact saved:")
print(performance_file)

CORRECTED F1-SCORE VALIDATION


,Interval category,Model,Actual out-of-range,Detected problems,Missed problems,F1 score
0,Short (<=24 hours),Logistic Regression,22,14,8,0.6222
1,Short (<=24 hours),Random Forest,22,21,1,0.7000
2,Medium (>24 hours–7 days),Logistic Regression,3,2,1,0.8000
3,Medium (>24 hours–7 days),Random Forest,3,0,3,0.0000
4,Long (>7 days),Logistic Regression,19,1,18,0.0952
5,Long (>7 days),Random Forest,19,0,19,0.0000



Corrected artifact saved:
c:\Users\hymar\OneDrive\Desktop\AQUA-DS-2026\outputs\tables\observation_interval_model_performance.csv


# Final Interpretation and Conclusions

## 1. Purpose of the analysis

This notebook examined the predictive behaviour of the Logistic Regression and Random Forest models developed in Notebook 03.

The analysis focused on prediction errors, pond-level error patterns, measurement timing, observation intervals and differences between model performance across Eluru and Nellore.

The target represents the water-quality status at the **next recorded observation for the same pond**. It does not represent a prediction at a fixed future time horizon.

## 2. Prediction errors and model behaviour

Both models produced false positives and false negatives, but their error profiles differed.

Logistic Regression generally achieved higher recall in selected settings, meaning it identified a greater proportion of recorded out-of-range observations. However, this sometimes came with a substantially higher false-positive rate.

Random Forest generally provided a better balance between detecting out-of-range observations and correctly identifying in-range observations in several comparisons, particularly for short observation intervals.

Neither model consistently dominated across every region, interval category and evaluation metric.

## 3. Importance of regional differences

Model behaviour differed between Eluru and Nellore.

In Eluru, Random Forest demonstrated better F1 and balanced accuracy for short observation intervals, while Logistic Regression achieved higher recall.

In Nellore, Random Forest performed better on recall and balanced accuracy for short observation intervals. Its F1 advantage, however, was not statistically conclusive under the paired pond-level bootstrap comparison.

The results should therefore be interpreted regionally rather than assuming that one model will perform identically across both regions.

## 4. Measurement timing and observation intervals

An important finding was the strong association between future measurement time and elapsed time between observations.

Later-day observations predominantly had short intervals, whereas morning observations were mostly associated with medium or long intervals.

There was insufficient overlap in the elapsed-time distributions to support a fair morning-versus-later-day comparison:

- Eluru had only one morning observation within the common elapsed-time range.
- Nellore had no common elapsed-time range between morning and later-day observations.

Consequently, the differences in time-of-day classification performance cannot be attributed independently to measurement timing.

The interval-based analysis showed that predictive performance varied across short, medium and long observation gaps. However, interval duration and measurement-time composition remained associated, so these results do not establish a causal effect of longer observation gaps.

## 5. Interval-based model comparison

The pond-clustered bootstrap provided the following evidence:

### Eluru

For short intervals, Random Forest had significantly higher F1 and balanced accuracy, while Logistic Regression had significantly higher recall.

For medium and long intervals, the confidence intervals for the three compared metrics included zero. The analysis therefore did not establish a consistent performance advantage for either model in those categories.

### Nellore

For short intervals, Random Forest had significantly higher recall and balanced accuracy. The F1 difference was not statistically conclusive.

Medium-interval results were based on only 29 observations, 21 ponds and three actual out-of-range cases. These findings are exploratory.

For long intervals, both models showed poor detection of out-of-range cases. Random Forest failed to detect any of the 19 actual out-of-range observations in this subgroup.

## 6. Statistical and methodological considerations

Repeated observations from the same pond are not independent.

Pond-level bootstrap resampling was therefore used for the model comparisons to preserve the grouping of repeated observations.

The bootstrap comparisons used 2,000 resamples. The Nellore medium-interval comparison had 1,757 valid replicates because some resamples produced undefined metric values.

The results remain dependent on the recorded observations, the target classification and the temporal structure of the available datasets.

The recorded classification should not automatically be treated as a universally consistent biological definition of water quality across every observation period.

## 7. Limitations

The following limitations apply to the interpretation of this notebook:

1. The prediction target is the next recorded observation, not a fixed-duration forecast.
2. Observation intervals are highly irregular, ranging from minutes to several months.
3. Measurement timing and observation interval have limited overlap, preventing an independent comparison of their effects.
4. Some regional interval subgroups contain relatively few actual out-of-range observations.
5. Repeated observations from the same ponds limit the independence of individual records.
6. The recorded target classification may reflect changes in monitoring or classification practices.
7. Predictive associations do not establish causal relationships between water-quality variables and subsequent out-of-range status.
8. Model performance on these test partitions does not guarantee equivalent performance on new farms, ponds or future monitoring periods.

## 8. Overall conclusion

The analysis demonstrates that the predictive behaviour of AQUA's models depends on the region, observation interval and evaluation metric.

Random Forest provides stronger balanced performance in several short-interval settings, while Logistic Regression retains a recall advantage in some comparisons.

Both models show important limitations when predicting the status of observations separated by long intervals, particularly in Nellore.

AQUA should therefore be presented as a system for analysing and predicting the status of the next recorded water-quality observation, with performance that varies across the available monitoring conditions.

The findings support further investigation into temporal validation, observation frequency and region-specific model behaviour. They do not establish that either model is universally superior or that the models are ready for unsupervised farm-level decision-making.